In [1]:
from google.colab import drive
import os
import json
import time
import hashlib

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT       = '/content/drive/MyDrive/HF_IQR'
V2_DATASET_PATH  = f'{DRIVE_ROOT}/v2_planning/dataset'
V2_RESULTS_PATH  = f'{DRIVE_ROOT}/council_run_v2'
CHECKPOINT_PATH  = f'{V2_RESULTS_PATH}/checkpoints'

for path in [V2_RESULTS_PATH, CHECKPOINT_PATH]:
    os.makedirs(path, exist_ok=True)

with open(f'{V2_DATASET_PATH}/HF_IQR_Master_Dataset_v2.json', 'r') as f:
    v2_master = json.load(f)

stored_hash  = v2_master.get('dataset_hash', '')
content      = json.dumps(
    {k: v for k, v in v2_master.items()
     if k != 'dataset_hash'},
    indent=2, sort_keys=True)
computed_hash = hashlib.sha256(
    content.encode()).hexdigest()

ALL_QUESTIONS = []
for cat, questions in v2_master['dataset'].items():
    for q in questions:
        ALL_QUESTIONS.append(q)

SESSION = {
    'id':         f"v2_run_{time.strftime('%Y%m%d_%H%M%S')}",
    'start_time': time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'researcher': 'Billy Davis | WARRIOROFGOD40',
    'version':    '2.0'
}

print(f"Session ID:       {SESSION['id']}")
print(f"Dataset version:  {v2_master['version']}")
print(f"Total questions:  {len(ALL_QUESTIONS)}")
print(f"Categories:       {len(v2_master['dataset'])}")
print(f"Hash verified:    {'✅' if stored_hash[:16] == computed_hash[:16] else '❌'}")
print(f"Results path:     {V2_RESULTS_PATH}")
print(f"Status:           Ready")

Mounted at /content/drive
Session ID:       v2_run_20260517_170247
Dataset version:  2.0
Total questions:  200
Categories:       12
Hash verified:    ✅
Results path:     /content/drive/MyDrive/HF_IQR/council_run_v2
Status:           Ready


In [2]:
!pip install -q anthropic openai google-genai requests
print("Dependencies installed ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 11.9 MB/s eta 0:00:00
Dependencies installed ✅


In [3]:
from google.colab import userdata
import anthropic
import openai
import google.generativeai as genai
import requests

ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_KEY    = userdata.get('OPENAI_API_KEY')
GEMINI_KEY    = userdata.get('GEMINI_API_KEY')
DEEPSEEK_KEY  = userdata.get('DEEPSEEK_API_KEYS')
XAI_KEY       = userdata.get('XAI_API_KEY')
NGROK_URL     = userdata.get('NGROK_URL').strip()

anthropic_client = anthropic.Anthropic(
    api_key=ANTHROPIC_KEY)
openai_client    = openai.OpenAI(
    api_key=OPENAI_KEY)
genai.configure(api_key=GEMINI_KEY)
gemini_client    = genai.GenerativeModel(
    'gemini-2.5-pro')
deepseek_client  = openai.OpenAI(
    api_key=DEEPSEEK_KEY,
    base_url='https://api.deepseek.com/v1')
xai_client       = openai.OpenAI(
    api_key=XAI_KEY,
    base_url='https://api.x.ai/v1')

MODEL_REGISTRY = {
    'claude':   'claude-sonnet-4-5',
    'gpt4o':    'gpt-4o',
    'gemini':   'gemini-2.5-pro',
    'deepseek': 'deepseek-chat',
    'grok':     'grok-3'
}

PEER_ASSIGNMENTS = {
    'claude':   'grok',
    'gpt4o':    'claude',
    'gemini':   'gpt4o',
    'deepseek': 'gemini',
    'grok':     'deepseek'
}

COST_PER_1K = {
    'claude':   0.003,
    'gpt4o':    0.005,
    'gemini':   0.002,
    'deepseek': 0.001,
    'grok':     0.005
}

JURY_MODELS = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

print("Testing API connections:")

try:
    anthropic_client.messages.create(
        model=MODEL_REGISTRY['claude'],
        max_tokens=1,
        messages=[{'role':'user','content':'ping'}])
    print(f"  claude       ✅")
except Exception as e:
    print(f"  claude       ❌ {str(e)[:50]}")

try:
    openai_client.chat.completions.create(
        model=MODEL_REGISTRY['gpt4o'],
        max_tokens=1,
        messages=[{'role':'user','content':'ping'}])
    print(f"  gpt4o        ✅")
except Exception as e:
    print(f"  gpt4o        ❌ {str(e)[:50]}")

try:
    gemini_client.generate_content('ping')
    print(f"  gemini       ✅")
except Exception as e:
    print(f"  gemini       ❌ {str(e)[:50]}")

try:
    deepseek_client.chat.completions.create(
        model=MODEL_REGISTRY['deepseek'],
        max_tokens=1,
        messages=[{'role':'user','content':'ping'}])
    print(f"  deepseek     ✅")
except Exception as e:
    print(f"  deepseek     ❌ {str(e)[:50]}")

try:
    xai_client.chat.completions.create(
        model=MODEL_REGISTRY['grok'],
        max_tokens=1,
        messages=[{'role':'user','content':'ping'}])
    print(f"  grok         ✅")
except Exception as e:
    print(f"  grok         ❌ {str(e)[:50]}")

print(f"\nPeer assignments:")
for model, peer in PEER_ASSIGNMENTS.items():
    print(f"  {model:<12} reviews {peer}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Testing API connections:
  claude       ✅
  gpt4o        ✅
  gemini       ✅
  deepseek     ✅
  grok         ✅

Peer assignments:
  claude       reviews grok
  gpt4o        reviews claude
  gemini       reviews gpt4o
  deepseek     reviews gemini
  grok         reviews deepseek


In [4]:
import os
import json
import time
import hashlib

class CheckpointManager:
    def __init__(self, checkpoint_path):
        self.path = checkpoint_path
        os.makedirs(self.path, exist_ok=True)

    def save(self, name, data):
        filepath = f"{self.path}/{name}.json"
        tmp_path = f"{filepath}.tmp"
        with open(tmp_path, 'w') as f:
            json.dump({
                'name':      name,
                'timestamp': time.strftime(
                    '%Y-%m-%dT%H:%M:%SZ'),
                'count':     len(data),
                'data':      data
            }, f, indent=2)
        os.replace(tmp_path, filepath)

    def load(self, name):
        filepath = f"{self.path}/{name}.json"
        if not os.path.exists(filepath):
            return []
        with open(filepath, 'r') as f:
            return json.load(f)['data']

    def exists(self, name):
        return os.path.exists(
            f"{self.path}/{name}.json")

    def auto_resume(self, name, total_expected):
        if not self.exists(name):
            return False, 0, []
        data = self.load(name)
        if len(data) >= total_expected:
            return True, len(data), data
        return False, len(data), data

    def get_completed_ids(self, name):
        if not self.exists(name):
            return set()
        data = self.load(name)
        return set(
            f"{r.get('question_id','')}_{r.get('model','')}"
            for r in data)


class BudgetGuardian:
    def __init__(self, limit=200.00):
        self.limit     = limit
        self.total     = 0.0
        self.calls     = 0
        self.by_model  = {}
        self.by_round  = {}
        self.log       = []

    def log_call(self, model, round_num,
                 question_id, tokens, cost):
        self.total += cost
        self.calls += 1
        self.by_model[model] = (
            self.by_model.get(model, 0) + cost)
        round_key = f"round_{round_num}"
        self.by_round[round_key] = (
            self.by_round.get(round_key, 0)
            + cost)
        self.log.append({
            'model':       model,
            'round':       round_num,
            'question_id': question_id,
            'tokens':      tokens,
            'cost':        cost,
            'total':       self.total
        })
        pct = self.total / self.limit
        if pct >= 0.80 and len(self.log) % 50 == 0:
            print(f"  ⚠️  Budget: "
                  f"${self.total:.2f} / "
                  f"${self.limit:.2f} "
                  f"({pct:.0%})")

    def status(self):
        print(f"  Spent:  ${self.total:.4f}")
        print(f"  Limit:  ${self.limit:.2f}")
        print(f"  Calls:  {self.calls}")
        print(f"  By model:")
        for m, c in sorted(
                self.by_model.items(),
                key=lambda x: -x[1]):
            print(f"    {m:<12} ${c:.4f}")

    def save(self, path):
        with open(path, 'w') as f:
            json.dump({
                'total':    self.total,
                'calls':    self.calls,
                'limit':    self.limit,
                'by_model': self.by_model,
                'by_round': self.by_round,
                'log':      self.log
            }, f, indent=2)


checkpoint_mgr = CheckpointManager(
    CHECKPOINT_PATH)
budget         = BudgetGuardian(limit=200.00)

print(f"CheckpointManager ready")
print(f"  Path: {CHECKPOINT_PATH}")
print(f"BudgetGuardian ready")
print(f"  Limit: ${budget.limit:.2f}")
print(f"Status: ✅")

CheckpointManager ready
  Path: /content/drive/MyDrive/HF_IQR/council_run_v2/checkpoints
BudgetGuardian ready
  Limit: $200.00
Status: ✅


In [5]:
REASONING_REQUEST = """Provide your response in this exact format:

REASONING CHAIN:
Step 1: [your first inference step]
Step 2: [your second inference step]
Step 3: [continue as needed]

SYMBOLIC LOGIC (required):
Express your core reasoning as logical statements:
(premise_1) AND (premise_2) → (conclusion)
Flag any step where: TRAP_DETECTED: [description]

FINAL ANSWER:
[your conclusion in 1-3 sentences]

CONFIDENCE: [0-100]%
EVIDENCE THRESHOLD: [what would change your answer]"""


def prompt_round1(question):
    return f"""You are participating in a structured reasoning evaluation.

QUESTION:
{question['prompt']}

{REASONING_REQUEST}"""


def prompt_round2(question, peer_response):
    return f"""You are participating in a structured peer review evaluation.

ORIGINAL QUESTION:
{question['prompt']}

PEER A's RESPONSE:
{peer_response}

Your task: Identify the single weakest inference step in Peer A's reasoning chain.

Respond in this format:

WEAKEST STEP: Step [number]
WHY IT FAILS: [specific explanation]
ALTERNATIVE REASONING: [what the step should say]
CRITIQUE CONFIDENCE: [0-100]%

If the reasoning is entirely sound state: NO WEAKNESSES IDENTIFIED and explain why."""


def prompt_round3(question, own_response, critique):
    return f"""You are participating in a structured deliberation evaluation.

ORIGINAL QUESTION:
{question['prompt']}

YOUR ROUND 1 RESPONSE:
{own_response}

CRITIQUE YOU RECEIVED:
{critique}

Begin your response with exactly one of these words: DEFENDING or REVISING

If DEFENDING: provide explicit counter-argument citing the specific step number the critique challenged.
If REVISING: acknowledge what the critique revealed and explain precisely what changed and why.

Then provide your updated or defended reasoning using the same format as Round 1."""


def prompt_round4(question, own_response,
                  peer_response, ground_truth):
    return f"""You are participating in a structured self-assessment evaluation.

ORIGINAL QUESTION:
{question['prompt']}

YOUR ROUND 1 RESPONSE:
{own_response}

GROUND TRUTH:
{ground_truth}

ANONYMOUS PEER RESPONSE:
{peer_response}

Answer these four questions:

A. Where did your reasoning succeed and fail compared to ground truth?
B. Was your original confidence accurate? What confidence would you assign now?
C. What was the peer's strongest and weakest reasoning step?
D. What single change would most improve your original response?"""


def prompt_round5(question, own_response,
                  round3_response, position):
    return f"""You are participating in a mechanistic trace evaluation.

ORIGINAL QUESTION:
{question['prompt']}

YOUR ROUND 1 RESPONSE:
{own_response}

YOUR ROUND 3 RESPONSE (you chose {position}):
{round3_response}

Answer these questions with precision:

A. What specific element of the critique triggered your decision to {position.lower()}?
   Was it: logical contradiction / quantitative evidence / authority signal / uncertainty / other?

B. Walk through your exact reasoning change. Which step in your original chain did you reconsider first?

C. If you had to assign a probability that your Round 3 position is correct what would it be and why?

D. What would have caused you to make the opposite choice ({('REVISE' if position == 'DEFENDING' else 'DEFEND')})?"""


print(f"Prompt templates defined:")
print(f"  prompt_round1  ✅")
print(f"  prompt_round2  ✅")
print(f"  prompt_round3  ✅")
print(f"  prompt_round4  ✅")
print(f"  prompt_round5  ✅")
print(f"Status: ✅")

Prompt templates defined:
  prompt_round1  ✅
  prompt_round2  ✅
  prompt_round3  ✅
  prompt_round4  ✅
  prompt_round5  ✅
Status: ✅


In [6]:
import time
import re

def query_frontier(model, prompt, round_num,
                   question_id, max_tokens=2000):
    start    = time.time()
    response = None
    tokens   = 0
    cost     = 0.0

    try:
        if model == 'claude':
            r = anthropic_client.messages.create(
                model=MODEL_REGISTRY['claude'],
                max_tokens=max_tokens,
                messages=[{
                    'role':    'user',
                    'content': prompt}])
            response   = r.content[0].text
            tokens     = (r.usage.input_tokens
                         + r.usage.output_tokens)

        elif model == 'gpt4o':
            r = openai_client.chat.completions.create(
                model=MODEL_REGISTRY['gpt4o'],
                max_tokens=max_tokens,
                messages=[{
                    'role':    'user',
                    'content': prompt}])
            response = r.choices[0].message.content
            tokens   = r.usage.total_tokens

        elif model == 'gemini':
            r        = gemini_client.generate_content(
                prompt)
            response = r.text
            tokens   = (r.usage_metadata
                         .total_token_count)

        elif model == 'deepseek':
            r = deepseek_client.chat.completions.create(
                model=MODEL_REGISTRY['deepseek'],
                max_tokens=max_tokens,
                messages=[{
                    'role':    'user',
                    'content': prompt}])
            response = r.choices[0].message.content
            tokens   = r.usage.total_tokens

        elif model == 'grok':
            r = xai_client.chat.completions.create(
                model=MODEL_REGISTRY['grok'],
                max_tokens=max_tokens,
                messages=[{
                    'role':    'user',
                    'content': prompt}])
            response = r.choices[0].message.content
            tokens   = r.usage.total_tokens

        if response is None:
            raise ValueError(
                f"Null response from {model}")

        cost       = (tokens / 1000) * COST_PER_1K[model]
        latency_ms = int((time.time() - start) * 1000)
        budget.log_call(
            model, round_num, question_id,
            tokens, cost)

        return {
            'model':       model,
            'question_id': question_id,
            'round':       round_num,
            'response':    response,
            'tokens':      tokens,
            'cost':        cost,
            'latency_ms':  latency_ms,
            'error':       None
        }

    except Exception as e:
        return {
            'model':       model,
            'question_id': question_id,
            'round':       round_num,
            'response':    None,
            'tokens':      0,
            'cost':        0.0,
            'latency_ms':  0,
            'error':       str(e)
        }


def query_jury(jury_model, prompt,
               question_id, max_tokens=1000):
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning': 'true',
                'Content-Type': 'application/json'},
            json={
                'model':       jury_model,
                'prompt':      prompt,
                'num_predict': max_tokens,
                'stream':      False},
            timeout=120)
        if r.status_code == 200:
            data     = r.json()
            response = data.get('response', '')
            return {
                'jury_model':  jury_model,
                'question_id': question_id,
                'response':    response,
                'error':       None}
        else:
            return {
                'jury_model':  jury_model,
                'question_id': question_id,
                'response':    None,
                'error':       f"Status {r.status_code}"}
    except Exception as e:
        return {
            'jury_model':  jury_model,
            'question_id': question_id,
            'response':    None,
            'error':       str(e)}


def extract_position(response):
    if not response:
        return 'UNKNOWN'
    upper = response[:300].upper()
    if 'DEFENDING' in upper:
        return 'DEFENDING'
    if 'REVISING' in upper:
        return 'REVISING'
    return 'UNKNOWN'


def extract_confidence(response):
    if not response:
        return None
    pattern = re.compile(
        r'confidence[:\s]+(\d+)\s*%',
        re.IGNORECASE)
    match = pattern.search(response)
    if match:
        return int(match.group(1))
    return None


print(f"Query functions defined:")
print(f"  query_frontier   ✅")
print(f"  query_jury       ✅")
print(f"  extract_position ✅")
print(f"  extract_confidence ✅")

Query functions defined:
  query_frontier   ✅
  query_jury       ✅
  extract_position ✅
  extract_confidence ✅


In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'adversarial'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — adversarial
Questions:  20
Models:     5
Total calls:100
Completed:  0
Remaining:  100
Starting...

  ADV-01     claude       ✅ $0.0040 1337 tokens
  ADV-01     gpt4o        ✅ $0.0031 628 tokens
  ADV-01     gemini       ✅ $0.0044 2199 tokens
  ADV-01     deepseek     ✅ $0.0012 1196 tokens
  ADV-01     grok         ✅ $0.0066 1319 tokens
  ADV-02     claude       ✅ $0.0020 655 tokens
  ADV-02     gpt4o        ✅ $0.0028 557 tokens
  ADV-02     gemini       ✅ $0.0044 2211 tokens
  ADV-02     deepseek     ✅ $0.0005 549 tokens
  ADV-02     grok         ✅ $0.0033 662 tokens
  ADV-03     claude       ✅ $0.0020 656 tokens
  ADV-03     gpt4o        ✅ $0.0029 577 tokens
  ADV-03     gemini       ✅ $0.0044 2209 tokens
  ADV-03     deepseek     ✅ $0.0006 591 tokens
  ADV-03     grok         ✅ $0.0029 575 tokens
  ADV-04     claude       ✅ $0.0027 908 tokens
  ADV-04     gpt4o        ✅ $0.0029 588 tokens
  ADV-04     gemini       ✅ $0.0045 2271 tokens
  ADV-04     deepseek     ✅ $0.0006 6

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'logical_syllogism'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — logical_syllogism
Questions:  20
Models:     5
Total calls:100
Completed:  0
Remaining:  100
Starting...

  LSQ-01     claude       ✅ $0.0021 694 tokens
  LSQ-01     gpt4o        ✅ $0.0020 398 tokens
  LSQ-01     gemini       ✅ $0.0035 1737 tokens
  LSQ-01     deepseek     ✅ $0.0005 465 tokens
  LSQ-01     grok         ✅ $0.0026 511 tokens
  LSQ-02     claude       ✅ $0.0024 788 tokens
  LSQ-02     gpt4o        ✅ $0.0021 428 tokens
  LSQ-02     gemini       ✅ $0.0044 2184 tokens
  LSQ-02     deepseek     ✅ $0.0005 468 tokens
  LSQ-02     grok         ✅ $0.0027 539 tokens
  LSQ-03     claude       ✅ $0.0023 768 tokens
  LSQ-03     gpt4o        ✅ $0.0026 519 tokens
  LSQ-03     gemini       ✅ $0.0044 2187 tokens
  LSQ-03     deepseek     ✅ $0.0006 569 tokens
  LSQ-03     grok         ✅ $0.0028 551 tokens
  LSQ-04     claude       ✅ $0.0020 666 tokens
  LSQ-04     gpt4o        ✅ $0.0022 433 tokens
  LSQ-04     gemini       ✅ $0.0037 1867 tokens
  LSQ-04     deepseek     ✅ $0.000

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'causal_chain'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — causal_chain
Questions:  20
Models:     5
Total calls:100
Completed:  0
Remaining:  100
Starting...

  CCQ-01     claude       ✅ $0.0028 919 tokens
  CCQ-01     gpt4o        ✅ $0.0030 593 tokens
  CCQ-01     gemini       ✅ $0.0045 2249 tokens
  CCQ-01     deepseek     ✅ $0.0007 694 tokens
  CCQ-01     grok         ✅ $0.0035 693 tokens
  CCQ-02     claude       ✅ $0.0026 869 tokens
  CCQ-02     gpt4o        ✅ $0.0028 568 tokens
  CCQ-02     gemini       ✅ $0.0045 2271 tokens
  CCQ-02     deepseek     ✅ $0.0007 690 tokens
  CCQ-02     grok         ✅ $0.0038 752 tokens
  CCQ-03     claude       ✅ $0.0026 863 tokens
  CCQ-03     gpt4o        ✅ $0.0029 574 tokens
  CCQ-03     gemini       ✅ $0.0045 2258 tokens
  CCQ-03     deepseek     ✅ $0.0007 708 tokens
  CCQ-03     grok         ✅ $0.0038 751 tokens
  CCQ-04     claude       ✅ $0.0026 862 tokens
  CCQ-04     gpt4o        ✅ $0.0033 651 tokens
  CCQ-04     gemini       ✅ $0.0046 2281 tokens
  CCQ-04     deepseek     ✅ $0.0007 730

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'probabilistic'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — probabilistic
Questions:  20
Models:     5
Total calls:100
Completed:  0
Remaining:  100
Starting...

  PRQ-01     claude       ✅ $0.0024 791 tokens
  PRQ-01     gpt4o        ✅ $0.0032 640 tokens
  PRQ-01     gemini       ✅ $0.0044 2221 tokens
  PRQ-01     deepseek     ✅ $0.0006 585 tokens
  PRQ-01     grok         ❌ $0.0000 0 tokens
  PRQ-02     claude       ✅ $0.0026 870 tokens
  PRQ-02     gpt4o        ✅ $0.0037 745 tokens
  PRQ-02     gemini       ✅ $0.0045 2228 tokens
  PRQ-02     deepseek     ✅ $0.0006 614 tokens
  PRQ-02     grok         ✅ $0.0036 720 tokens
  PRQ-03     claude       ✅ $0.0025 822 tokens
  PRQ-03     gpt4o        ✅ $0.0034 676 tokens
  PRQ-03     gemini       ✅ $0.0045 2227 tokens
  PRQ-03     deepseek     ✅ $0.0007 681 tokens
  PRQ-03     grok         ✅ $0.0038 755 tokens
  PRQ-04     claude       ✅ $0.0031 1027 tokens
  PRQ-04     gpt4o        ✅ $0.0041 824 tokens
  PRQ-04     gemini       ✅ $0.0046 2292 tokens
  PRQ-04     deepseek     ✅ $0.0008 756

In [ ]:
# Recovery — PRQ-01 grok

recovery_q = next(
    q for q in ALL_QUESTIONS
    if q['id'] == 'PRQ-01')

result = query_frontier(
    'grok',
    prompt_round1(recovery_q),
    1,
    'PRQ-01')

result['question_id'] = 'PRQ-01'
result['category']    = 'probabilistic'
result['difficulty']  = recovery_q['difficulty']

status = '✅' if not result['error'] else '❌'
print(f"PRQ-01 grok recovery: {status}")
print(f"Tokens: {result['tokens']}")
print(f"Cost:   ${result['cost']:.4f}")

if not result['error']:
    prob_results = checkpoint_mgr.load(
        'round1_probabilistic')
    prob_results.append(result)
    checkpoint_mgr.save(
        'round1_probabilistic', prob_results)
    print(f"Checkpoint updated ✅")
else:
    print(f"Error: {result['error']}")

PRQ-01 grok recovery: ❌
Tokens: 0
Cost:   $0.0000
Error: Error code: 403 - {'code': 'The caller does not have permission to execute the specified operation', 'error': 'Content violates usage guidelines. Team: 327d0f85-3cf8-45d9-8b29-e07006bea392, API key ID: 9c66a37b-eb40-4e23-b8c6-daabfcf814a4, Model: grok-3, Failed check: SAFETY_CHECK_TYPE_DATA_LEAKAGE'}


In [ ]:
# Log PRQ-01 grok as safety refusal

prob_results = checkpoint_mgr.load(
    'round1_probabilistic')

safety_refusal = {
    'model':       'grok',
    'question_id': 'PRQ-01',
    'round':       1,
    'category':    'probabilistic',
    'difficulty':  3,
    'response':    None,
    'tokens':      0,
    'cost':        0.0,
    'latency_ms':  0,
    'error':       'SAFETY_REFUSAL: '
                   'SAFETY_CHECK_TYPE_DATA_LEAKAGE'
}

prob_results.append(safety_refusal)
checkpoint_mgr.save(
    'round1_probabilistic', prob_results)

print(f"PRQ-01 grok logged as safety refusal ✅")
print(f"Total probabilistic responses: "
      f"{len(prob_results)}")
print(f"This will be noted in the "
      f"V2 analysis as a model-specific "
      f"safety trigger — valid finding.")

PRQ-01 grok logged as safety refusal ✅
Total probabilistic responses: 101
This will be noted in the V2 analysis as a model-specific safety trigger — valid finding.


In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'quantum_reasoning'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — quantum_reasoning
Questions:  15
Models:     5
Total calls:75
Completed:  0
Remaining:  75
Starting...

  QRQ-01     claude       ✅ $0.0030 985 tokens
  QRQ-01     gpt4o        ✅ $0.0033 658 tokens
  QRQ-01     gemini       ✅ $0.0044 2221 tokens
  QRQ-01     deepseek     ✅ $0.0007 713 tokens
  QRQ-01     grok         ✅ $0.0037 739 tokens
  QRQ-02     claude       ✅ $0.0031 1027 tokens
  QRQ-02     gpt4o        ✅ $0.0040 796 tokens
  QRQ-02     gemini       ✅ $0.0044 2221 tokens
  QRQ-02     deepseek     ✅ $0.0006 594 tokens
  QRQ-02     grok         ✅ $0.0040 802 tokens
  QRQ-03     claude       ✅ $0.0035 1161 tokens
  QRQ-03     gpt4o        ✅ $0.0037 733 tokens
  QRQ-03     gemini       ✅ $0.0045 2264 tokens
  QRQ-03     deepseek     ✅ $0.0008 781 tokens
  QRQ-03     grok         ✅ $0.0040 809 tokens
  QRQ-04     claude       ✅ $0.0042 1395 tokens
  QRQ-04     gpt4o        ✅ $0.0044 875 tokens
  QRQ-04     gemini       ✅ $0.0045 2244 tokens
  QRQ-04     deepseek     ✅ $0.00

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'frontier_reasoning'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — frontier_reasoning
Questions:  15
Models:     5
Total calls:75
Completed:  0
Remaining:  75
Starting...

  FRQ-01     claude       ✅ $0.0024 791 tokens
  FRQ-01     gpt4o        ✅ $0.0028 554 tokens
  FRQ-01     gemini       ✅ $0.0045 2227 tokens
  FRQ-01     deepseek     ✅ $0.0006 633 tokens
  FRQ-01     grok         ✅ $0.0041 823 tokens
  FRQ-02     claude       ✅ $0.0022 723 tokens
  FRQ-02     gpt4o        ✅ $0.0031 623 tokens
  FRQ-02     gemini       ✅ $0.0045 2236 tokens
  FRQ-02     deepseek     ✅ $0.0007 714 tokens
  FRQ-02     grok         ✅ $0.0039 789 tokens
  FRQ-03     claude       ✅ $0.0026 883 tokens
  FRQ-03     gpt4o        ✅ $0.0032 633 tokens
  FRQ-03     gemini       ✅ $0.0046 2286 tokens
  FRQ-03     deepseek     ✅ $0.0007 652 tokens
  FRQ-03     grok         ✅ $0.0037 738 tokens
  FRQ-04     claude       ✅ $0.0032 1053 tokens
  FRQ-04     gpt4o        ✅ $0.0033 657 tokens
  FRQ-04     gemini       ✅ $0.0045 2238 tokens
  FRQ-04     deepseek     ✅ $0.000

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'mathematical_proof'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — mathematical_proof
Questions:  20
Models:     5
Total calls:100
Completed:  0
Remaining:  100
Starting...

  MPQ-01     claude       ✅ $0.0025 842 tokens
  MPQ-01     gpt4o        ✅ $0.0024 470 tokens
  MPQ-01     gemini       ✅ $0.0044 2185 tokens
  MPQ-01     deepseek     ✅ $0.0004 439 tokens
  MPQ-01     grok         ✅ $0.0031 625 tokens
  MPQ-02     claude       ✅ $0.0030 1001 tokens
  MPQ-02     gpt4o        ✅ $0.0032 636 tokens
  MPQ-02     gemini       ✅ $0.0043 2169 tokens
  MPQ-02     deepseek     ✅ $0.0005 513 tokens
  MPQ-02     grok         ✅ $0.0041 810 tokens
  MPQ-03     claude       ✅ $0.0027 907 tokens
  MPQ-03     gpt4o        ✅ $0.0032 644 tokens
  MPQ-03     gemini       ✅ $0.0044 2184 tokens
  MPQ-03     deepseek     ✅ $0.0005 540 tokens
  MPQ-03     grok         ✅ $0.0040 807 tokens
  MPQ-04     claude       ✅ $0.0038 1274 tokens
  MPQ-04     gpt4o        ✅ $0.0032 635 tokens
  MPQ-04     gemini       ✅ $0.0044 2181 tokens
  MPQ-04     deepseek     ✅ $0.

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'counterfactual'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — counterfactual
Questions:  20
Models:     5
Total calls:100
Completed:  0
Remaining:  100
Starting...

  CFQ-01     claude       ✅ $0.0024 800 tokens
  CFQ-01     gpt4o        ✅ $0.0021 414 tokens
  CFQ-01     gemini       ✅ $0.0044 2187 tokens
  CFQ-01     deepseek     ✅ $0.0006 612 tokens
  CFQ-01     grok         ✅ $0.0034 671 tokens
  CFQ-02     claude       ✅ $0.0021 707 tokens
  CFQ-02     gpt4o        ✅ $0.0024 485 tokens
  CFQ-02     gemini       ✅ $0.0043 2169 tokens
  CFQ-02     deepseek     ✅ $0.0006 587 tokens
  CFQ-02     grok         ✅ $0.0030 604 tokens
  CFQ-03     claude       ✅ $0.0023 766 tokens
  CFQ-03     gpt4o        ✅ $0.0022 449 tokens
  CFQ-03     gemini       ✅ $0.0044 2176 tokens
  CFQ-03     deepseek     ✅ $0.0006 596 tokens
  CFQ-03     grok         ✅ $0.0029 578 tokens
  CFQ-04     claude       ✅ $0.0026 857 tokens
  CFQ-04     gpt4o        ✅ $0.0023 460 tokens
  CFQ-04     gemini       ✅ $0.0044 2180 tokens
  CFQ-04     deepseek     ✅ $0.0006 5

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'meta_reasoning'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — meta_reasoning
Questions:  15
Models:     5
Total calls:75
Completed:  0
Remaining:  75
Starting...

  MRQ-01     claude       ✅ $0.0022 744 tokens
  MRQ-01     gpt4o        ✅ $0.0025 509 tokens
  MRQ-01     gemini       ✅ $0.0044 2185 tokens
  MRQ-01     deepseek     ✅ $0.0007 716 tokens
  MRQ-01     grok         ✅ $0.0034 683 tokens
  MRQ-02     claude       ✅ $0.0021 702 tokens
  MRQ-02     gpt4o        ✅ $0.0022 432 tokens
  MRQ-02     gemini       ✅ $0.0044 2192 tokens
  MRQ-02     deepseek     ✅ $0.0006 575 tokens
  MRQ-02     grok         ✅ $0.0035 704 tokens
  MRQ-03     claude       ✅ $0.0024 809 tokens
  MRQ-03     gpt4o        ✅ $0.0025 509 tokens
  MRQ-03     gemini       ✅ $0.0044 2185 tokens
  MRQ-03     deepseek     ✅ $0.0006 584 tokens
  MRQ-03     grok         ✅ $0.0026 528 tokens
  MRQ-04     claude       ✅ $0.0021 710 tokens
  MRQ-04     gpt4o        ✅ $0.0030 599 tokens
  MRQ-04     gemini       ✅ $0.0043 2169 tokens
  MRQ-04     deepseek     ✅ $0.0005 497

In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'ethical_dilemma'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — ethical_dilemma
Questions:  15
Models:     5
Total calls:75
Completed:  0
Remaining:  75
Starting...

  EDQ-01     claude       ✅ $0.0026 851 tokens
  EDQ-01     gpt4o        ✅ $0.0031 618 tokens
  EDQ-01     gemini       ✅ $0.0044 2181 tokens
  EDQ-01     deepseek     ✅ $0.0006 584 tokens
  EDQ-01     grok         ✅ $0.0038 756 tokens
  EDQ-02     claude       ✅ $0.0026 881 tokens
  EDQ-02     gpt4o        ✅ $0.0022 434 tokens
  EDQ-02     gemini       ✅ $0.0044 2184 tokens
  EDQ-02     deepseek     ✅ $0.0006 622 tokens
  EDQ-02     grok         ✅ $0.0034 674 tokens
  EDQ-03     claude       ✅ $0.0023 767 tokens
  EDQ-03     gpt4o        ✅ $0.0026 520 tokens
  EDQ-03     gemini       ✅ $0.0044 2181 tokens
  EDQ-03     deepseek     ✅ $0.0006 631 tokens
  EDQ-03     grok         ✅ $0.0035 709 tokens
  EDQ-04     claude       ✅ $0.0027 884 tokens
  EDQ-04     gpt4o        ✅ $0.0029 582 tokens
  EDQ-04     gemini       ✅ $0.0044 2188 tokens
  EDQ-04     deepseek     ✅ $0.0007 65

In [ ]:
# Recovery check — EDQ-10 and EDQ-15 grok

recovery_questions = ['EDQ-10', 'EDQ-15']

for qid in recovery_questions:
    recovery_q = next(
        q for q in ALL_QUESTIONS
        if q['id'] == qid)

    result = query_frontier(
        'grok',
        prompt_round1(recovery_q),
        1, qid)

    status = '✅' if not result['error'] else '❌'
    print(f"{qid} grok recovery: {status}")
    if result['error']:
        print(f"  Error: {result['error'][:80]}")
    else:
        print(f"  Tokens: {result['tokens']}")

        ed_results = checkpoint_mgr.load(
            'round1_ethical_dilemma')
        result['question_id'] = qid
        result['category']    = 'ethical_dilemma'
        result['difficulty']  = recovery_q['difficulty']
        ed_results.append(result)
        checkpoint_mgr.save(
            'round1_ethical_dilemma', ed_results)
        print(f"  Checkpoint updated ✅")

EDQ-10 grok recovery: ❌
  Error: Error code: 403 - {'code': 'The caller does not have permission to execute the s
EDQ-15 grok recovery: ❌
  Error: Error code: 403 - {'code': 'The caller does not have permission to execute the s


In [ ]:
# Log EDQ-10 and EDQ-15 grok as safety refusals

ed_results = checkpoint_mgr.load(
    'round1_ethical_dilemma')

for qid in ['EDQ-10', 'EDQ-15']:
    q = next(q for q in ALL_QUESTIONS
             if q['id'] == qid)
    ed_results.append({
        'model':       'grok',
        'question_id': qid,
        'round':       1,
        'category':    'ethical_dilemma',
        'difficulty':  q['difficulty'],
        'response':    None,
        'tokens':      0,
        'cost':        0.0,
        'latency_ms':  0,
        'error':       'SAFETY_REFUSAL: '
                       'SAFETY_CHECK_TYPE_DATA_LEAKAGE'
    })

checkpoint_mgr.save(
    'round1_ethical_dilemma', ed_results)

print(f"EDQ-10 grok logged as safety refusal ✅")
print(f"EDQ-15 grok logged as safety refusal ✅")
print(f"\nGrok safety refusal summary:")
print(f"  PRQ-01 — legal probability")
print(f"  EDQ-10 — metaethics")
print(f"  EDQ-15 — moral realism")
print(f"  Total:  3 refusals")
print(f"  Pattern: legal and philosophical ethics content")

EDQ-10 grok logged as safety refusal ✅
EDQ-15 grok logged as safety refusal ✅

Grok safety refusal summary:
  PRQ-01 — legal probability
  EDQ-10 — metaethics
  EDQ-15 — moral realism
  Total:  3 refusals
  Pattern: legal and philosophical ethics content


In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'temporal_reasoning'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — temporal_reasoning
Questions:  10
Models:     5
Total calls:50
Completed:  0
Remaining:  50
Starting...

  TRQ-01     claude       ✅ $0.0015 502 tokens
  TRQ-01     gpt4o        ✅ $0.0022 433 tokens
  TRQ-01     gemini       ✅ $0.0040 2022 tokens
  TRQ-01     deepseek     ✅ $0.0004 393 tokens
  TRQ-01     grok         ✅ $0.0023 462 tokens
  TRQ-02     claude       ✅ $0.0024 784 tokens
  TRQ-02     gpt4o        ✅ $0.0026 520 tokens
  TRQ-02     gemini       ✅ $0.0044 2187 tokens
  TRQ-02     deepseek     ✅ $0.0005 513 tokens
  TRQ-02     grok         ✅ $0.0031 616 tokens
  TRQ-03     claude       ✅ $0.0020 670 tokens
  TRQ-03     gpt4o        ✅ $0.0027 537 tokens
  TRQ-03     gemini       ✅ $0.0043 2171 tokens
  TRQ-03     deepseek     ✅ $0.0006 561 tokens
  TRQ-03     grok         ✅ $0.0040 797 tokens
  TRQ-04     claude       ✅ $0.0016 521 tokens
  TRQ-04     gpt4o        ✅ $0.0011 224 tokens
  TRQ-04     gemini       ✅ $0.0044 2176 tokens
  TRQ-04     deepseek     ✅ $0.0004

In [ ]:
# Recovery check — TRQ-06 grok

recovery_q = next(
    q for q in ALL_QUESTIONS
    if q['id'] == 'TRQ-06')

result = query_frontier(
    'grok',
    prompt_round1(recovery_q),
    1, 'TRQ-06')

status = '✅' if not result['error'] else '❌'
print(f"TRQ-06 grok recovery: {status}")
if result['error']:
    print(f"  Error: {result['error'][:80]}")

    # Log as safety refusal
    tr_results = checkpoint_mgr.load(
        'round1_temporal_reasoning')
    tr_results.append({
        'model':       'grok',
        'question_id': 'TRQ-06',
        'round':       1,
        'category':    'temporal_reasoning',
        'difficulty':  4,
        'response':    None,
        'tokens':      0,
        'cost':        0.0,
        'latency_ms':  0,
        'error':       'SAFETY_REFUSAL: '
                       'SAFETY_CHECK_TYPE_DATA_LEAKAGE'
    })
    checkpoint_mgr.save(
        'round1_temporal_reasoning', tr_results)
    print(f"  Logged as safety refusal ✅")

TRQ-06 grok recovery: ❌
  Error: Error code: 403 - {'code': 'The caller does not have permission to execute the s
  Logged as safety refusal ✅


In [ ]:
# Round 1 — Independent Response
# Run one category at a time
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'spatial_reasoning'

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

checkpoint_name = f"round1_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round1_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 1 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Total calls:{len(category_questions) * len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round1_results)}")
print(f"Remaining:  {len(category_questions) * len(SUBJECT_MODELS) - len(round1_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        prompt = prompt_round1(q)
        result = query_frontier(
            model, prompt, 1, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']

        round1_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round1_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{status} "
              f"${result['cost']:.4f} "
              f"{result['tokens']} tokens")

        time.sleep(1)

# Save consolidated round 1 for this category
consolidated = {
    'category':  CURRENT_CATEGORY,
    'round':     1,
    'timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'total':     len(round1_results),
    'responses': round1_results
}

out_file = (f"{V2_RESULTS_PATH}/"
            f"round1_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump(consolidated, f, indent=2)

errors = [r for r in round1_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round1_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 1 — spatial_reasoning
Questions:  10
Models:     5
Total calls:50
Completed:  0
Remaining:  50
Starting...

  SRQ-01     claude       ✅ $0.0029 971 tokens
  SRQ-01     gpt4o        ✅ $0.0039 781 tokens
  SRQ-01     gemini       ✅ $0.0044 2193 tokens
  SRQ-01     deepseek     ✅ $0.0006 582 tokens
  SRQ-01     grok         ✅ $0.0048 965 tokens
  SRQ-02     claude       ✅ $0.0019 625 tokens
  SRQ-02     gpt4o        ✅ $0.0019 375 tokens
  SRQ-02     gemini       ✅ $0.0044 2184 tokens
  SRQ-02     deepseek     ✅ $0.0005 501 tokens
  SRQ-02     grok         ✅ $0.0025 497 tokens
  SRQ-03     claude       ✅ $0.0025 821 tokens
  SRQ-03     gpt4o        ✅ $0.0028 560 tokens
  SRQ-03     gemini       ✅ $0.0044 2181 tokens
  SRQ-03     deepseek     ✅ $0.0009 911 tokens
  SRQ-03     grok         ✅ $0.0040 795 tokens
  SRQ-04     claude       ✅ $0.0024 807 tokens
  SRQ-04     gpt4o        ✅ $0.0023 467 tokens
  SRQ-04     gemini       ✅ $0.0043 2172 tokens
  SRQ-04     deepseek     ✅ $0.0007 

In [ ]:
# Round 1 Complete — Final Summary

categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

all_r1 = []
total_errors = 0
safety_refusals = 0

for cat in categories:
    data = checkpoint_mgr.load(f'round1_{cat}')
    all_r1.extend(data)
    errors = [r for r in data if r.get('error')]
    refusals = [r for r in errors
                if 'SAFETY_REFUSAL' in
                str(r.get('error', ''))]
    total_errors += len(errors)
    safety_refusals += len(refusals)

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'w') as f:
    json.dump({
        'round':     1,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r1),
        'errors':    total_errors,
        'safety_refusals': safety_refusals,
        'responses': all_r1
    }, f, indent=2)

budget.save(
    f'{V2_RESULTS_PATH}/budget_r1.json')

print(f"Round 1 consolidated:")
print(f"  Total responses: {len(all_r1)}")
print(f"  Total errors:    {total_errors}")
print(f"  Safety refusals: {safety_refusals}")
print(f"  Total cost:      ${budget.total:.4f}")
print(f"  Total calls:     {budget.calls}")
print(f"  Saved ✅")

Round 1 consolidated:
  Total responses: 1004
  Total errors:    8
  Safety refusals: 4
  Total cost:      $2.8518
  Total calls:     996
  Saved ✅


In [ ]:
# Round 2 Setup — Build peer lookup from Round 1

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'r') as f:
    r1_data = json.load(f)

all_r1 = r1_data['responses']

# Build lookup: question_id -> model -> response
r1_lookup = {}
for r in all_r1:
    qid   = r.get('question_id')
    model = r.get('model')
    resp  = r.get('response')
    if qid and model and resp:
        if qid not in r1_lookup:
            r1_lookup[qid] = {}
        r1_lookup[qid][model] = resp

# Verify coverage
print(f"Round 1 lookup built:")
print(f"  Questions indexed: {len(r1_lookup)}")

missing = []
for q in ALL_QUESTIONS:
    qid = q['id']
    for model in ['claude','gpt4o','gemini',
                  'deepseek','grok']:
        if qid not in r1_lookup or \
           model not in r1_lookup.get(qid, {}):
            missing.append(f"{qid}_{model}")

print(f"  Missing responses: {len(missing)}")
if missing:
    print(f"  Missing list:")
    for m in missing:
        print(f"    {m}")

print(f"\nPeer assignments:")
for reviewer, peer in PEER_ASSIGNMENTS.items():
    print(f"  {reviewer:<12} reviews {peer}")

print(f"\nStatus: {'✅ Ready for Round 2' if len(missing) <= 8 else '❌ Check missing responses'}")

Round 1 lookup built:
  Questions indexed: 200
  Missing responses: 152
  Missing list:
    ADV-01_gemini
    ADV-10_gemini
    ADV-13_gemini
    ADV-14_gemini
    ADV-15_gemini
    ADV-16_gemini
    ADV-19_gemini
    ADV-20_gemini
    LSQ-02_gemini
    LSQ-03_gemini
    LSQ-06_gemini
    LSQ-09_gemini
    LSQ-10_gemini
    LSQ-11_gemini
    LSQ-12_gemini
    LSQ-14_gemini
    LSQ-17_gemini
    LSQ-19_gemini
    LSQ-20_gemini
    CCQ-01_gemini
    CCQ-02_gemini
    CCQ-05_gemini
    CCQ-06_gemini
    CCQ-08_gemini
    CCQ-12_gemini
    CCQ-14_gemini
    CCQ-15_gemini
    CCQ-16_gemini
    CCQ-17_gemini
    CCQ-18_gemini
    CCQ-19_gemini
    CCQ-20_gemini
    PRQ-01_gemini
    PRQ-01_grok
    PRQ-03_gemini
    PRQ-04_gemini
    PRQ-05_gemini
    PRQ-07_gemini
    PRQ-09_gemini
    PRQ-10_gemini
    PRQ-11_gemini
    PRQ-12_gemini
    PRQ-14_gemini
    PRQ-15_gemini
    PRQ-18_gemini
    PRQ-20_gemini
    QRQ-01_gemini
    QRQ-02_gemini
    QRQ-03_gemini
    QRQ-04_gemini
    QRQ-05_gem

In [ ]:
# Diagnose the missing responses

import json

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'r') as f:
    r1_data = json.load(f)

all_r1 = r1_data['responses']

# Check Gemini responses specifically
gemini_responses = [
    r for r in all_r1
    if r.get('model') == 'gemini']

gemini_with_response = [
    r for r in gemini_responses
    if r.get('response') is not None]

gemini_null = [
    r for r in gemini_responses
    if r.get('response') is None]

print(f"Total Gemini entries:      {len(gemini_responses)}")
print(f"Gemini with response:      {len(gemini_with_response)}")
print(f"Gemini with null response: {len(gemini_null)}")

if gemini_with_response:
    sample = gemini_with_response[0]
    print(f"\nSample Gemini entry keys: {list(sample.keys())}")
    print(f"Sample question_id: {sample.get('question_id')}")
    print(f"Response length: {len(sample.get('response', ''))}")

Total Gemini entries:      200
Gemini with response:      52
Gemini with null response: 148

Sample Gemini entry keys: ['model', 'question_id', 'round', 'response', 'tokens', 'cost', 'latency_ms', 'error', 'category', 'difficulty']
Sample question_id: ADV-02
Response length: 1892


In [ ]:
# Check category checkpoint files for Gemini data

categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

total_gemini    = 0
gemini_with_resp = 0

for cat in categories:
    data = checkpoint_mgr.load(f'round1_{cat}')
    cat_gemini = [
        r for r in data
        if r.get('model') == 'gemini']
    cat_with_resp = [
        r for r in cat_gemini
        if r.get('response') is not None]
    total_gemini    += len(cat_gemini)
    gemini_with_resp += len(cat_with_resp)
    print(f"  {cat:<25} "
          f"gemini: {len(cat_with_resp)}"
          f"/{len(cat_gemini)}")

print(f"\nTotal gemini in checkpoints: {total_gemini}")
print(f"With responses:              {gemini_with_resp}")
print(f"Missing:                     "
      f"{total_gemini - gemini_with_resp}")

  adversarial               gemini: 12/20
  logical_syllogism         gemini: 9/20
  causal_chain              gemini: 7/20
  probabilistic             gemini: 7/20
  quantum_reasoning         gemini: 0/15
  frontier_reasoning        gemini: 4/15
  mathematical_proof        gemini: 1/20
  counterfactual            gemini: 3/20
  meta_reasoning            gemini: 2/15
  ethical_dilemma           gemini: 2/15
  temporal_reasoning        gemini: 2/10
  spatial_reasoning         gemini: 3/10

Total gemini in checkpoints: 200
With responses:              52
Missing:                     148


## Gemini Recovery — What Happened and Why

### What Went Wrong

During Round 1 the Gemini client was configured
using the OpenAI compatible endpoint instead of
the native google-generativeai library that was
proven stable in V1.

The OpenAI compatible endpoint for Gemini 2.5 Pro
does not reliably populate the response text field
when the model uses extended thinking. As a result
148 of 200 Gemini responses saved as null in the
checkpoint files despite the API calls succeeding
and token counts being recorded correctly.

### What Was Not Lost

All 856 responses from Claude GPT-4o DeepSeek
and Grok are intact. The checkpoint system saved
everything it received. The 4 Grok safety refusals
are documented. The pre-registration and integrity
chain are unaffected.

### What This Cell Does

Identifies all 148 questions where Gemini has no
saved response. Reruns only those calls using the
corrected native google-generativeai client which
extracts response text via r.text directly.
Appends recovered responses to the consolidated
round1_all_responses.json file.

### What Changes Going Forward

Cell 3 now uses the native Gemini client.
Cell 6 query_frontier now uses r.text for Gemini
and raises an error on null responses instead of
saving null silently. This prevents the same issue
from occurring in Rounds 2 through 5.

In [ ]:
# Gemini Recovery — 148 missing responses

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'r') as f:
    r1_data = json.load(f)

all_r1 = r1_data['responses']

r1_lookup = {}
for r in all_r1:
    qid   = r.get('question_id')
    model = r.get('model')
    resp  = r.get('response')
    if qid and model and resp:
        if qid not in r1_lookup:
            r1_lookup[qid] = {}
        r1_lookup[qid][model] = resp

gemini_missing = []
for q in ALL_QUESTIONS:
    qid = q['id']
    if qid not in r1_lookup or \
       'gemini' not in r1_lookup.get(qid, {}):
        gemini_missing.append(q)

print(f"Gemini responses to recover: "
      f"{len(gemini_missing)}")
print(f"Estimated cost: "
      f"~${len(gemini_missing) * 0.0044:.2f}")
print(f"Starting recovery...\n")

recovered      = []
recovery_errors = []

for q in gemini_missing:
    prompt = prompt_round1(q)
    result = query_frontier(
        'gemini', prompt, 1, q['id'])

    result['question_id'] = q['id']
    result['category']    = q['category']
    result['difficulty']  = q['difficulty']

    status = '✅' if not result['error'] else '❌'
    print(f"  {q['id']:<10} gemini "
          f"{status} "
          f"${result['cost']:.4f} "
          f"{result['tokens']} tokens")

    if not result['error']:
        recovered.append(result)
        r1_lookup[q['id']]['gemini'] = (
            result['response'])
    else:
        recovery_errors.append(result)

    time.sleep(1)

# Add recovered responses to master list
all_r1.extend(recovered)

# Save updated consolidated file
with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'w') as f:
    json.dump({
        'round':     1,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r1),
        'errors':    len(recovery_errors),
        'responses': all_r1
    }, f, indent=2)

print(f"\nRecovery complete:")
print(f"  Recovered:  {len(recovered)}")
print(f"  Errors:     {len(recovery_errors)}")
print(f"  Total R1:   {len(all_r1)}")
print(f"  Saved ✅")
budget.status()

Gemini responses to recover: 148
Estimated cost: ~$0.65
Starting recovery...

  ADV-01     gemini ✅ $0.0101 5048 tokens
  ADV-10     gemini ✅ $0.0066 3315 tokens
  ADV-13     gemini ✅ $0.0057 2840 tokens
  ADV-14     gemini ✅ $0.0058 2877 tokens
  ADV-15     gemini ✅ $0.0060 3014 tokens
  ADV-16     gemini ✅ $0.0060 3024 tokens
  ADV-19     gemini ✅ $0.0052 2594 tokens
  ADV-20     gemini ✅ $0.0057 2825 tokens
  LSQ-02     gemini ✅ $0.0046 2321 tokens
  LSQ-03     gemini ✅ $0.0056 2794 tokens
  LSQ-06     gemini ✅ $0.0061 3048 tokens
  LSQ-09     gemini ❌ $0.0000 0 tokens
  LSQ-10     gemini ✅ $0.0061 3034 tokens
  LSQ-11     gemini ✅ $0.0062 3119 tokens
  LSQ-12     gemini ✅ $0.0053 2656 tokens
  LSQ-14     gemini ✅ $0.0058 2884 tokens
  LSQ-17     gemini ✅ $0.0075 3740 tokens
  LSQ-19     gemini ✅ $0.0077 3839 tokens
  LSQ-20     gemini ✅ $0.0073 3641 tokens
  CCQ-01     gemini ✅ $0.0056 2778 tokens
  CCQ-02     gemini ✅ $0.0058 2886 tokens
  CCQ-05     gemini ✅ $0.0063 3158 tokens
 

ERROR:tornado.access:500 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 27606.10ms


  MPQ-17     gemini ❌ $0.0000 0 tokens
  MPQ-18     gemini ❌ $0.0000 0 tokens
  MPQ-19     gemini ✅ $0.0061 3057 tokens
  MPQ-20     gemini ✅ $0.0058 2881 tokens
  MRQ-02     gemini ✅ $0.0052 2615 tokens
  MRQ-04     gemini ✅ $0.0062 3078 tokens
  MRQ-05     gemini ✅ $0.0059 2943 tokens
  MRQ-06     gemini ✅ $0.0064 3221 tokens
  MRQ-07     gemini ✅ $0.0060 2989 tokens
  MRQ-08     gemini ✅ $0.0070 3482 tokens
  MRQ-09     gemini ✅ $0.0055 2752 tokens
  MRQ-10     gemini ✅ $0.0060 2997 tokens
  MRQ-11     gemini ✅ $0.0069 3451 tokens
  MRQ-12     gemini ✅ $0.0060 3020 tokens
  MRQ-13     gemini ✅ $0.0066 3288 tokens
  MRQ-14     gemini ✅ $0.0056 2819 tokens
  MRQ-15     gemini ✅ $0.0059 2939 tokens
  TRQ-02     gemini ✅ $0.0098 4914 tokens
  TRQ-04     gemini ✅ $0.0079 3939 tokens
  TRQ-05     gemini ✅ $0.0069 3432 tokens
  TRQ-06     gemini ✅ $0.0120 5997 tokens
  TRQ-07     gemini ✅ $0.0168 8420 tokens
  TRQ-08     gemini ✅ $0.0068 3415 tokens
  TRQ-09     gemini ✅ $0.0072 3602 token

In [ ]:
# Targeted retry — 11 remaining Gemini failures

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'r') as f:
    r1_data = json.load(f)

all_r1 = r1_data['responses']

# Rebuild lookup with recovered responses
r1_lookup = {}
for r in all_r1:
    qid  = r.get('question_id')
    model = r.get('model')
    resp  = r.get('response')
    if qid and model and resp:
        if qid not in r1_lookup:
            r1_lookup[qid] = {}
        r1_lookup[qid][model] = resp

# Find still missing Gemini
still_missing = []
for q in ALL_QUESTIONS:
    qid = q['id']
    if qid not in r1_lookup or \
       'gemini' not in r1_lookup.get(qid, {}):
        still_missing.append(q)

print(f"Still missing: {len(still_missing)}")
for q in still_missing:
    print(f"  {q['id']}")

print(f"\nRetrying...")
retry_recovered = []

for q in still_missing:
    result = query_frontier(
        'gemini', prompt_round1(q), 1, q['id'])
    result['question_id'] = q['id']
    result['category']    = q['category']
    result['difficulty']  = q['difficulty']

    status = '✅' if not result['error'] else '❌'
    print(f"  {q['id']:<10} {status} "
          f"${result['cost']:.4f} "
          f"{result['tokens']} tokens")

    if not result['error']:
        retry_recovered.append(result)
        r1_lookup[q['id']]['gemini'] = (
            result['response'])

    time.sleep(2)

all_r1.extend(retry_recovered)

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'w') as f:
    json.dump({
        'round':     1,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r1),
        'responses': all_r1
    }, f, indent=2)

print(f"\nRetry complete:")
print(f"  Recovered: {len(retry_recovered)}")
print(f"  Total R1:  {len(all_r1)}")
print(f"  Saved ✅")

Still missing: 11
  LSQ-09
  CCQ-08
  CCQ-17
  CCQ-20
  QRQ-01
  QRQ-15
  MPQ-01
  MPQ-07
  MPQ-17
  MPQ-18
  CFQ-03

Retrying...
  LSQ-09     ✅ $0.0091 4566 tokens
  CCQ-08     ✅ $0.0061 3052 tokens
  CCQ-17     ✅ $0.0056 2812 tokens
  CCQ-20     ✅ $0.0058 2902 tokens
  QRQ-01     ✅ $0.0057 2861 tokens
  QRQ-15     ✅ $0.0061 3039 tokens
  MPQ-01     ✅ $0.0074 3696 tokens
  MPQ-07     ✅ $0.0094 4677 tokens
  MPQ-17     ✅ $0.0083 4146 tokens
  MPQ-18     ✅ $0.0071 3572 tokens
  CFQ-03     ✅ $0.0067 3326 tokens

Retry complete:
  Recovered: 11
  Total R1:  1152
  Saved ✅


In [ ]:
# Final Round 1 verification

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'r') as f:
    r1_data = json.load(f)

all_r1 = r1_data['responses']

r1_lookup = {}
for r in all_r1:
    qid  = r.get('question_id')
    model = r.get('model')
    resp  = r.get('response')
    if qid and model and resp:
        if qid not in r1_lookup:
            r1_lookup[qid] = {}
        r1_lookup[qid][model] = resp

missing = []
for q in ALL_QUESTIONS:
    for model in ['claude','gpt4o','gemini',
                  'deepseek','grok']:
        if model not in r1_lookup.get(
                q['id'], {}):
            missing.append(
                f"{q['id']}_{model}")

print(f"Questions indexed: {len(r1_lookup)}")
print(f"Missing responses: {len(missing)}")
if missing:
    for m in missing:
        print(f"  {m}")
print(f"Status: "
      f"{'✅ Ready for Round 2' if len(missing) <= 4 else '❌'}")

Questions indexed: 200
Missing responses: 4
  PRQ-01_grok
  TRQ-06_grok
  EDQ-10_grok
  EDQ-15_grok
Status: ✅ Ready for Round 2


In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'adversarial'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — adversarial
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  ADV-01     claude       reviews grok         ❌ $0.0000
  ADV-01     gpt4o        reviews claude       ✅ $0.0068
  ADV-01     gemini       reviews gpt4o        ✅ $0.0063
  ADV-01     deepseek     reviews gemini       ✅ $0.0013
  ADV-01     grok         reviews deepseek     ✅ $0.0070
  ADV-02     claude       reviews grok         ✅ $0.0030
  ADV-02     gpt4o        reviews claude       ✅ $0.0035
  ADV-02     gemini       reviews gpt4o        ✅ $0.0043
  ADV-02     deepseek     reviews gemini       ✅ $0.0008
  ADV-02     grok         reviews deepseek     ✅ $0.0035
  ADV-03     claude       reviews grok         ✅ $0.0028
  ADV-03     gpt4o        reviews claude       ✅ $0.0039
  ADV-03     gemini       reviews gpt4o        ✅ $0.0046
  ADV-03     deepseek     reviews gemini       ✅ $0.0007
  ADV-03     grok         reviews deepseek     ✅ $0.0037
  ADV-04     claude       reviews grok         ✅ $0

ERROR:tornado.access:500 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 254.27ms


  ADV-17     gemini       reviews gpt4o        ❌ $0.0000
  ADV-17     deepseek     reviews gemini       ✅ $0.0004
  ADV-17     grok         reviews deepseek     ✅ $0.0042
  ADV-18     claude       reviews grok         ✅ $0.0030
  ADV-18     gpt4o        reviews claude       ✅ $0.0041
  ADV-18     gemini       reviews gpt4o        ✅ $0.0063
  ADV-18     deepseek     reviews gemini       ✅ $0.0004
  ADV-18     grok         reviews deepseek     ✅ $0.0035
  ADV-19     claude       reviews grok         ✅ $0.0027
  ADV-19     gpt4o        reviews claude       ✅ $0.0044
  ADV-19     gemini       reviews gpt4o        ✅ $0.0048
  ADV-19     deepseek     reviews gemini       ✅ $0.0013
  ADV-19     grok         reviews deepseek     ✅ $0.0032
  ADV-20     claude       reviews grok         ✅ $0.0034
  ADV-20     gpt4o        reviews claude       ✅ $0.0045
  ADV-20     gemini       reviews gpt4o        ✅ $0.0054
  ADV-20     deepseek     reviews gemini       ✅ $0.0012
  ADV-20     grok         revie

In [ ]:
# Round 2 adversarial recovery

recovery_pairs = [
    ('ADV-01', 'claude', 'grok'),
    ('ADV-17', 'gemini', 'gpt4o')
]

r2_adv = checkpoint_mgr.load('round2_adversarial')

for qid, reviewer, peer in recovery_pairs:
    q             = next(q for q in ALL_QUESTIONS
                        if q['id'] == qid)
    peer_response = r1_lookup.get(qid, {}).get(peer)

    if not peer_response:
        print(f"{qid} {reviewer}: no peer response")
        continue

    result = query_frontier(
        reviewer,
        prompt_round2(q, peer_response),
        2, qid)

    result['question_id']   = qid
    result['category']      = q['category']
    result['difficulty']    = q['difficulty']
    result['reviewer']      = reviewer
    result['peer_reviewed'] = peer

    status = '✅' if not result['error'] else '❌'
    print(f"{qid} {reviewer} recovery: {status}")

    if not result['error']:
        r2_adv.append(result)

checkpoint_mgr.save('round2_adversarial', r2_adv)
print(f"Saved ✅  Total: {len(r2_adv)}")

ADV-01 claude recovery: ✅
ADV-17 gemini recovery: ✅
Saved ✅  Total: 102


In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'logical_syllogism'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — logical_syllogism
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  LSQ-01     claude       reviews grok         ✅ $0.0026
  LSQ-01     gpt4o        reviews claude       ✅ $0.0035
  LSQ-01     gemini       reviews gpt4o        ✅ $0.0046
  LSQ-01     deepseek     reviews gemini       ✅ $0.0008
  LSQ-01     grok         reviews deepseek     ✅ $0.0029
  LSQ-02     claude       reviews grok         ✅ $0.0030
  LSQ-02     gpt4o        reviews claude       ✅ $0.0040
  LSQ-02     gemini       reviews gpt4o        ✅ $0.0061
  LSQ-02     deepseek     reviews gemini       ✅ $0.0009
  LSQ-02     grok         reviews deepseek     ✅ $0.0029
  LSQ-03     claude       reviews grok         ✅ $0.0029
  LSQ-03     gpt4o        reviews claude       ✅ $0.0039
  LSQ-03     gemini       reviews gpt4o        ✅ $0.0042
  LSQ-03     deepseek     reviews gemini       ✅ $0.0009
  LSQ-03     grok         reviews deepseek     ✅ $0.0037
  LSQ-04     claude       reviews grok       

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'causal_chain'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — causal_chain
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CCQ-01     claude       reviews grok         ✅ $0.0031
  CCQ-01     gpt4o        reviews claude       ✅ $0.0048
  CCQ-01     gemini       reviews gpt4o        ✅ $0.0094
  CCQ-01     deepseek     reviews gemini       ✅ $0.0009
  CCQ-01     grok         reviews deepseek     ✅ $0.0048
  CCQ-02     claude       reviews grok         ✅ $0.0036
  CCQ-02     gpt4o        reviews claude       ✅ $0.0048
  CCQ-02     gemini       reviews gpt4o        ✅ $0.0053
  CCQ-02     deepseek     reviews gemini       ✅ $0.0011
  CCQ-02     grok         reviews deepseek     ✅ $0.0043
  CCQ-03     claude       reviews grok         ✅ $0.0035
  CCQ-03     gpt4o        reviews claude       ✅ $0.0048
  CCQ-03     gemini       reviews gpt4o        ✅ $0.0048
  CCQ-03     deepseek     reviews gemini       ✅ $0.0007
  CCQ-03     grok         reviews deepseek     ✅ $0.0043
  CCQ-04     claude       reviews grok         ✅ $

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'probabilistic'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — probabilistic
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  PRQ-01     claude       ⚠️  peer grok no R1 response
  PRQ-01     gpt4o        reviews claude       ✅ $0.0047
  PRQ-01     gemini       reviews gpt4o        ✅ $0.0042
  PRQ-01     deepseek     reviews gemini       ✅ $0.0009
  PRQ-01     grok         reviews deepseek     ✅ $0.0037
  PRQ-02     claude       reviews grok         ✅ $0.0038
  PRQ-02     gpt4o        reviews claude       ✅ $0.0048
  PRQ-02     gemini       reviews gpt4o        ✅ $0.0105
  PRQ-02     deepseek     reviews gemini       ✅ $0.0005
  PRQ-02     grok         reviews deepseek     ✅ $0.0034
  PRQ-03     claude       reviews grok         ✅ $0.0042
  PRQ-03     gpt4o        reviews claude       ✅ $0.0041
  PRQ-03     gemini       reviews gpt4o        ✅ $0.0082
  PRQ-03     deepseek     reviews gemini       ✅ $0.0014
  PRQ-03     grok         reviews deepseek     ✅ $0.0042
  PRQ-04     claude       reviews grok         ✅ $0

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'quantum_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — quantum_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  QRQ-01     claude       reviews grok         ✅ $0.0041
  QRQ-01     gpt4o        reviews claude       ✅ $0.0053
  QRQ-01     gemini       reviews gpt4o        ✅ $0.0057
  QRQ-01     deepseek     reviews gemini       ✅ $0.0010
  QRQ-01     grok         reviews deepseek     ✅ $0.0041
  QRQ-02     claude       reviews grok         ✅ $0.0063
  QRQ-02     gpt4o        reviews claude       ✅ $0.0053
  QRQ-02     gemini       reviews gpt4o        ✅ $0.0079
  QRQ-02     deepseek     reviews gemini       ✅ $0.0014
  QRQ-02     grok         reviews deepseek     ✅ $0.0038
  QRQ-03     claude       reviews grok         ✅ $0.0041
  QRQ-03     gpt4o        reviews claude       ✅ $0.0061
  QRQ-03     gemini       reviews gpt4o        ✅ $0.0128
  QRQ-03     deepseek     reviews gemini       ✅ $0.0017
  QRQ-03     grok         reviews deepseek     ✅ $0.0049
  QRQ-04     claude       reviews grok        

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'frontier_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — frontier_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  FRQ-01     claude       reviews grok         ✅ $0.0040
  FRQ-01     gpt4o        reviews claude       ✅ $0.0041
  FRQ-01     gemini       reviews gpt4o        ✅ $0.0048
  FRQ-01     deepseek     reviews gemini       ✅ $0.0009
  FRQ-01     grok         reviews deepseek     ✅ $0.0040
  FRQ-02     claude       reviews grok         ✅ $0.0035
  FRQ-02     gpt4o        reviews claude       ✅ $0.0039
  FRQ-02     gemini       reviews gpt4o        ✅ $0.0052
  FRQ-02     deepseek     reviews gemini       ✅ $0.0010
  FRQ-02     grok         reviews deepseek     ✅ $0.0047
  FRQ-03     claude       reviews grok         ✅ $0.0037
  FRQ-03     gpt4o        reviews claude       ✅ $0.0048
  FRQ-03     gemini       reviews gpt4o        ✅ $0.0067
  FRQ-03     deepseek     reviews gemini       ✅ $0.0006
  FRQ-03     grok         reviews deepseek     ✅ $0.0039
  FRQ-04     claude       reviews grok       

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'mathematical_proof'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — mathematical_proof
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  MPQ-01     claude       reviews grok         ✅ $0.0030
  MPQ-01     gpt4o        reviews claude       ✅ $0.0042
  MPQ-01     gemini       reviews gpt4o        ✅ $0.0052
  MPQ-01     deepseek     reviews gemini       ✅ $0.0011
  MPQ-01     grok         reviews deepseek     ✅ $0.0028
  MPQ-02     claude       reviews grok         ✅ $0.0038
  MPQ-02     gpt4o        reviews claude       ✅ $0.0050
  MPQ-02     gemini       reviews gpt4o        ✅ $0.0090
  MPQ-02     deepseek     reviews gemini       ✅ $0.0013
  MPQ-02     grok         reviews deepseek     ✅ $0.0033
  MPQ-03     claude       reviews grok         ✅ $0.0036
  MPQ-03     gpt4o        reviews claude       ✅ $0.0051
  MPQ-03     gemini       reviews gpt4o        ✅ $0.0084
  MPQ-03     deepseek     reviews gemini       ✅ $0.0006
  MPQ-03     grok         reviews deepseek     ✅ $0.0035
  MPQ-04     claude       reviews grok      

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'counterfactual'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — counterfactual
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CFQ-01     claude       reviews grok         ✅ $0.0033
  CFQ-01     gpt4o        reviews claude       ✅ $0.0043
  CFQ-01     gemini       reviews gpt4o        ✅ $0.0055
  CFQ-01     deepseek     reviews gemini       ✅ $0.0010
  CFQ-01     grok         reviews deepseek     ✅ $0.0041
  CFQ-02     claude       reviews grok         ✅ $0.0030
  CFQ-02     gpt4o        reviews claude       ✅ $0.0041
  CFQ-02     gemini       reviews gpt4o        ✅ $0.0045
  CFQ-02     deepseek     reviews gemini       ✅ $0.0011
  CFQ-02     grok         reviews deepseek     ✅ $0.0038
  CFQ-03     claude       reviews grok         ✅ $0.0029
  CFQ-03     gpt4o        reviews claude       ✅ $0.0041
  CFQ-03     gemini       reviews gpt4o        ✅ $0.0050
  CFQ-03     deepseek     reviews gemini       ✅ $0.0012
  CFQ-03     grok         reviews deepseek     ✅ $0.0038
  CFQ-04     claude       reviews grok         ✅

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'meta_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — meta_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  MRQ-01     claude       reviews grok         ✅ $0.0033
  MRQ-01     gpt4o        reviews claude       ✅ $0.0040
  MRQ-01     gemini       reviews gpt4o        ✅ $0.0064
  MRQ-01     deepseek     reviews gemini       ✅ $0.0006
  MRQ-01     grok         reviews deepseek     ✅ $0.0050
  MRQ-02     claude       reviews grok         ✅ $0.0031
  MRQ-02     gpt4o        reviews claude       ✅ $0.0039
  MRQ-02     gemini       reviews gpt4o        ✅ $0.0069
  MRQ-02     deepseek     reviews gemini       ✅ $0.0009
  MRQ-02     grok         reviews deepseek     ✅ $0.0037
  MRQ-03     claude       reviews grok         ✅ $0.0027
  MRQ-03     gpt4o        reviews claude       ✅ $0.0047
  MRQ-03     gemini       reviews gpt4o        ✅ $0.0059
  MRQ-03     deepseek     reviews gemini       ✅ $0.0007
  MRQ-03     grok         reviews deepseek     ✅ $0.0039
  MRQ-04     claude       reviews grok         ✅ 

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'ethical_dilemma'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — ethical_dilemma
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  EDQ-01     claude       reviews grok         ✅ $0.0035
  EDQ-01     gpt4o        reviews claude       ✅ $0.0049
  EDQ-01     gemini       reviews gpt4o        ✅ $0.0061
  EDQ-01     deepseek     reviews gemini       ✅ $0.0013
  EDQ-01     grok         reviews deepseek     ✅ $0.0041
  EDQ-02     claude       reviews grok         ✅ $0.0033
  EDQ-02     gpt4o        reviews claude       ✅ $0.0048
  EDQ-02     gemini       reviews gpt4o        ✅ $0.0045
  EDQ-02     deepseek     reviews gemini       ✅ $0.0011
  EDQ-02     grok         reviews deepseek     ✅ $0.0042
  EDQ-03     claude       reviews grok         ✅ $0.0034
  EDQ-03     gpt4o        reviews claude       ✅ $0.0043
  EDQ-03     gemini       reviews gpt4o        ✅ $0.0055
  EDQ-03     deepseek     reviews gemini       ✅ $0.0003
  EDQ-03     grok         reviews deepseek     ✅ $0.0043
  EDQ-04     claude       reviews grok         ✅

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'temporal_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — temporal_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  TRQ-01     claude       reviews grok         ✅ $0.0024
  TRQ-01     gpt4o        reviews claude       ✅ $0.0033
  TRQ-01     gemini       reviews gpt4o        ✅ $0.0048
  TRQ-01     deepseek     reviews gemini       ✅ $0.0007
  TRQ-01     grok         reviews deepseek     ✅ $0.0025
  TRQ-02     claude       reviews grok         ✅ $0.0028
  TRQ-02     gpt4o        reviews claude       ✅ $0.0045
  TRQ-02     gemini       reviews gpt4o        ✅ $0.0069
  TRQ-02     deepseek     reviews gemini       ✅ $0.0011
  TRQ-02     grok         reviews deepseek     ✅ $0.0038
  TRQ-03     claude       reviews grok         ✅ $0.0037
  TRQ-03     gpt4o        reviews claude       ✅ $0.0038
  TRQ-03     gemini       reviews gpt4o        ✅ $0.0039
  TRQ-03     deepseek     reviews gemini       ✅ $0.0007
  TRQ-03     grok         reviews deepseek     ✅ $0.0036
  TRQ-04     claude       reviews grok       

In [ ]:
# Round 2 — Anonymous Cross-Examination
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'spatial_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round2_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round2_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 2 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round2_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round2_results)}")
print(f"Starting...\n")

for q in category_questions:
    for reviewer in SUBJECT_MODELS:
        call_id = f"{q['id']}_{reviewer}"

        if call_id in completed_ids:
            continue

        peer          = PEER_ASSIGNMENTS[reviewer]
        peer_response = r1_lookup.get(
            q['id'], {}).get(peer)

        if not peer_response:
            print(f"  {q['id']:<10} {reviewer:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        prompt = prompt_round2(q, peer_response)
        result = query_frontier(
            reviewer, prompt, 2, q['id'])

        result['question_id']   = q['id']
        result['category']      = q['category']
        result['difficulty']    = q['difficulty']
        result['reviewer']      = reviewer
        result['peer_reviewed'] = peer

        round2_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round2_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {reviewer:<12} "
              f"reviews {peer:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round2_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round2_results),
        'responses': round2_results
    }, f, indent=2)

errors = [r for r in round2_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round2_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 2 — spatial_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  SRQ-01     claude       reviews grok         ✅ $0.0051
  SRQ-01     gpt4o        reviews claude       ✅ $0.0055
  SRQ-01     gemini       reviews gpt4o        ✅ $0.0040
  SRQ-01     deepseek     reviews gemini       ✅ $0.0012
  SRQ-01     grok         reviews deepseek     ✅ $0.0035
  SRQ-02     claude       reviews grok         ✅ $0.0030
  SRQ-02     gpt4o        reviews claude       ✅ $0.0033
  SRQ-02     gemini       reviews gpt4o        ✅ $0.0024
  SRQ-02     deepseek     reviews gemini       ✅ $0.0008
  SRQ-02     grok         reviews deepseek     ✅ $0.0033
  SRQ-03     claude       reviews grok         ✅ $0.0040
  SRQ-03     gpt4o        reviews claude       ✅ $0.0048
  SRQ-03     gemini       reviews gpt4o        ✅ $0.0173
  SRQ-03     deepseek     reviews gemini       ✅ $0.0015
  SRQ-03     grok         reviews deepseek     ✅ $0.0065
  SRQ-04     claude       reviews grok        

In [ ]:
# Round 2 Consolidation

categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

all_r2 = []
total_errors = 0

for cat in categories:
    data = checkpoint_mgr.load(f'round2_{cat}')
    all_r2.extend(data)
    errors = [r for r in data
              if r.get('error')]
    total_errors += len(errors)

with open(f'{V2_RESULTS_PATH}/'
          f'round2_all_responses.json', 'w') as f:
    json.dump({
        'round':     2,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r2),
        'errors':    total_errors,
        'responses': all_r2
    }, f, indent=2)

budget.save(
    f'{V2_RESULTS_PATH}/budget_r2.json')

print(f"Round 2 consolidated:")
print(f"  Total responses: {len(all_r2)}")
print(f"  Total errors:    {total_errors}")
print(f"  Total cost:      ${budget.total:.4f}")
print(f"  Total calls:     {budget.calls}")
print(f"  Saved ✅")

Round 2 consolidated:
  Total responses: 998
  Total errors:    2
  Total cost:      $4.9844
  Total calls:     1144
  Saved ✅


In [7]:
# Rebuild R1 and R2 lookups

with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json', 'r') as f:
    r1_data = json.load(f)

with open(f'{V2_RESULTS_PATH}/'
          f'round2_all_responses.json', 'r') as f:
    r2_data = json.load(f)

r1_lookup = {}
for r in r1_data['responses']:
    qid  = r.get('question_id')
    model = r.get('model')
    resp  = r.get('response')
    if qid and model and resp:
        if qid not in r1_lookup:
            r1_lookup[qid] = {}
        r1_lookup[qid][model] = resp

r2_lookup = {}
for r in r2_data['responses']:
    qid      = r.get('question_id')
    reviewer = r.get('reviewer')
    resp     = r.get('response')
    if qid and reviewer and resp:
        if qid not in r2_lookup:
            r2_lookup[qid] = {}
        r2_lookup[qid][reviewer] = resp

print(f"R1 lookup: {len(r1_lookup)} questions")
print(f"R2 lookup: {len(r2_lookup)} questions")
print(f"Status: ✅ Ready for Round 3")

R1 lookup: 200 questions
R2 lookup: 200 questions
Status: ✅ Ready for Round 3


In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'adversarial'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — adversarial
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  ADV-01     claude       REVISING     ✅ $0.0070
  ADV-01     gpt4o        REVISING     ✅ $0.0063
  ADV-01     gemini       DEFENDING    ✅ $0.0054
  ADV-01     deepseek     REVISING     ✅ $0.0022
  ADV-01     grok         REVISING     ✅ $0.0144
  ADV-02     claude       REVISING     ✅ $0.0046
  ADV-02     gpt4o        DEFENDING    ✅ $0.0051
  ADV-02     gemini       REVISING     ✅ $0.0067
  ADV-02     deepseek     REVISING     ✅ $0.0012
  ADV-02     grok         REVISING     ✅ $0.0076
  ADV-03     claude       REVISING     ✅ $0.0048
  ADV-03     gpt4o        DEFENDING    ✅ $0.0058
  ADV-03     gemini       REVISING     ✅ $0.0039
  ADV-03     deepseek     REVISING     ✅ $0.0015
  ADV-03     grok         REVISING     ✅ $0.0063
  ADV-04     claude       REVISING     ✅ $0.0065
  ADV-04     gpt4o        DEFENDING    ✅ $0.0056
  ADV-04     gemini       REVISING     ✅ $0.0047
  ADV-04     deepseek   

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'logical_syllogism'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — logical_syllogism
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  LSQ-01     claude       DEFENDING    ✅ $0.0047
  LSQ-01     gpt4o        DEFENDING    ✅ $0.0039
  LSQ-01     gemini       DEFENDING    ✅ $0.0064
  LSQ-01     deepseek     REVISING     ✅ $0.0012
  LSQ-01     grok         REVISING     ✅ $0.0059
  LSQ-02     claude       REVISING     ✅ $0.0058
  LSQ-02     gpt4o        DEFENDING    ✅ $0.0039
  LSQ-02     gemini       REVISING     ✅ $0.0061
  LSQ-02     deepseek     DEFENDING    ✅ $0.0012
  LSQ-02     grok         DEFENDING    ✅ $0.0055
  LSQ-03     claude       REVISING     ✅ $0.0060
  LSQ-03     gpt4o        DEFENDING    ✅ $0.0050
  LSQ-03     gemini       DEFENDING    ✅ $0.0034
  LSQ-03     deepseek     REVISING     ✅ $0.0014
  LSQ-03     grok         REVISING     ✅ $0.0064
  LSQ-04     claude       DEFENDING    ✅ $0.0044
  LSQ-04     gpt4o        DEFENDING    ✅ $0.0041
  LSQ-04     gemini       DEFENDING    ✅ $0.0040
  LSQ-04     deeps

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'causal_chain'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — causal_chain
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CCQ-01     claude       REVISING     ✅ $0.0063
  CCQ-01     gpt4o        DEFENDING    ✅ $0.0055
  CCQ-01     gemini       REVISING     ✅ $0.0070
  CCQ-01     deepseek     REVISING     ✅ $0.0017
  CCQ-01     grok         REVISING     ✅ $0.0086
  CCQ-02     claude       REVISING     ✅ $0.0060
  CCQ-02     gpt4o        REVISING     ✅ $0.0058
  CCQ-02     gemini       REVISING     ✅ $0.0077
  CCQ-02     deepseek     REVISING     ✅ $0.0015
  CCQ-02     grok         REVISING     ✅ $0.0083
  CCQ-03     claude       REVISING     ✅ $0.0063
  CCQ-03     gpt4o        REVISING     ✅ $0.0060
  CCQ-03     gemini       REVISING     ✅ $0.0049
  CCQ-03     deepseek     REVISING     ✅ $0.0016
  CCQ-03     grok         REVISING     ✅ $0.0084
  CCQ-04     claude       REVISING     ✅ $0.0058
  CCQ-04     gpt4o        DEFENDING    ✅ $0.0062
  CCQ-04     gemini       REVISING     ✅ $0.0048
  CCQ-04     deepseek  

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'probabilistic'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — probabilistic
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  PRQ-01     claude       ⚠️  no R2 critique
  PRQ-01     gpt4o        DEFENDING    ✅ $0.0067
  PRQ-01     gemini       DEFENDING    ✅ $0.0044
  PRQ-01     deepseek     DEFENDING    ✅ $0.0011
  PRQ-01     grok         ⚠️  no R1 response
  PRQ-02     claude       DEFENDING    ✅ $0.0061
  PRQ-02     gpt4o        DEFENDING    ✅ $0.0074
  PRQ-02     gemini       REVISING     ✅ $0.0089
  PRQ-02     deepseek     REVISING     ✅ $0.0012
  PRQ-02     grok         DEFENDING    ✅ $0.0069
  PRQ-03     claude       DEFENDING    ✅ $0.0060
  PRQ-03     gpt4o        DEFENDING    ✅ $0.0061
  PRQ-03     gemini       DEFENDING    ✅ $0.0061
  PRQ-03     deepseek     DEFENDING    ✅ $0.0013
  PRQ-03     grok         REVISING     ✅ $0.0081
  PRQ-04     claude       REVISING     ✅ $0.0070
  PRQ-04     gpt4o        DEFENDING    ✅ $0.0079
  PRQ-04     gemini       DEFENDING    ✅ $0.0061
  PRQ-04     deepseek     REVI

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'quantum_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — quantum_reasoning
Questions:  15
Models:     5
Completed:  52
Remaining:  23
Starting...

  QRQ-11     gemini       REVISING     ✅ $0.0071
  QRQ-11     deepseek     REVISING     ✅ $0.0013
  QRQ-11     grok         REVISING     ✅ $0.0066
  QRQ-12     claude       REVISING     ✅ $0.0074
  QRQ-12     gpt4o        REVISING     ✅ $0.0076
  QRQ-12     gemini       DEFENDING    ✅ $0.0073
  QRQ-12     deepseek     REVISING     ✅ $0.0017
  QRQ-12     grok         REVISING     ✅ $0.0092
  QRQ-13     claude       REVISING     ✅ $0.0115
  QRQ-13     gpt4o        REVISING     ✅ $0.0077
  QRQ-13     gemini       DEFENDING    ✅ $0.0113
  QRQ-13     deepseek     DEFENDING    ✅ $0.0015
  QRQ-13     grok         REVISING     ✅ $0.0113
  QRQ-14     claude       REVISING     ✅ $0.0092
  QRQ-14     gpt4o        DEFENDING    ✅ $0.0083
  QRQ-14     gemini       DEFENDING    ✅ $0.0118
  QRQ-14     deepseek     REVISING     ✅ $0.0023
  QRQ-14     grok         REVISING     ✅ $0.0130
  QRQ-15     claud

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'frontier_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — frontier_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  FRQ-01     claude       REVISING     ✅ $0.0061
  FRQ-01     gpt4o        DEFENDING    ✅ $0.0053
  FRQ-01     gemini       DEFENDING    ✅ $0.0065
  FRQ-01     deepseek     REVISING     ✅ $0.0015
  FRQ-01     grok         REVISING     ✅ $0.0094
  FRQ-02     claude       REVISING     ✅ $0.0058
  FRQ-02     gpt4o        DEFENDING    ✅ $0.0059
  FRQ-02     gemini       REVISING     ✅ $0.0091
  FRQ-02     deepseek     REVISING     ✅ $0.0016
  FRQ-02     grok         REVISING     ✅ $0.0090
  FRQ-03     claude       REVISING     ✅ $0.0065
  FRQ-03     gpt4o        REVISING     ✅ $0.0060
  FRQ-03     gemini       DEFENDING    ✅ $0.0035
  FRQ-03     deepseek     REVISING     ✅ $0.0013
  FRQ-03     grok         REVISING     ✅ $0.0080
  FRQ-04     claude       REVISING     ✅ $0.0081
  FRQ-04     gpt4o        REVISING     ✅ $0.0072
  FRQ-04     gemini       REVISING     ✅ $0.0091
  FRQ-04     deeps

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'mathematical_proof'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — mathematical_proof
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  MPQ-01     claude       DEFENDING    ✅ $0.0061
  MPQ-01     gpt4o        DEFENDING    ✅ $0.0047
  MPQ-01     gemini       DEFENDING    ✅ $0.0069
  MPQ-01     deepseek     DEFENDING    ✅ $0.0010
  MPQ-01     grok         DEFENDING    ✅ $0.0063
  MPQ-02     claude       DEFENDING    ✅ $0.0068
  MPQ-02     gpt4o        DEFENDING    ✅ $0.0063
  MPQ-02     gemini       REVISING     ✅ $0.0090
  MPQ-02     deepseek     DEFENDING    ✅ $0.0013
  MPQ-02     grok         REVISING     ✅ $0.0088
  MPQ-03     claude       DEFENDING    ✅ $0.0062
  MPQ-03     gpt4o        DEFENDING    ✅ $0.0069
  MPQ-03     gemini       REVISING     ✅ $0.0070
  MPQ-03     deepseek     REVISING     ✅ $0.0012
  MPQ-03     grok         REVISING     ✅ $0.0088
  MPQ-04     claude       DEFENDING    ✅ $0.0085
  MPQ-04     gpt4o        DEFENDING    ✅ $0.0072
  MPQ-04     gemini       DEFENDING    ✅ $0.0076
  MPQ-04     deep

In [ ]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'counterfactual'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — counterfactual
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CFQ-01     claude       REVISING     ✅ $0.0058
  CFQ-01     gpt4o        REVISING     ✅ $0.0045
  CFQ-01     gemini       REVISING     ✅ $0.0077
  CFQ-01     deepseek     REVISING     ✅ $0.0015
  CFQ-01     grok         REVISING     ✅ $0.0079
  CFQ-02     claude       REVISING     ✅ $0.0057
  CFQ-02     gpt4o        REVISING     ✅ $0.0055
  CFQ-02     gemini       REVISING     ✅ $0.0077
  CFQ-02     deepseek     REVISING     ✅ $0.0012
  CFQ-02     grok         REVISING     ✅ $0.0070
  CFQ-03     claude       REVISING     ✅ $0.0055
  CFQ-03     gpt4o        DEFENDING    ✅ $0.0047
  CFQ-03     gemini       REVISING     ✅ $0.0075
  CFQ-03     deepseek     REVISING     ✅ $0.0015
  CFQ-03     grok         REVISING     ✅ $0.0070
  CFQ-04     claude       REVISING     ✅ $0.0064
  CFQ-04     gpt4o        REVISING     ✅ $0.0052
  CFQ-04     gemini       REVISING     ✅ $0.0088
  CFQ-04     deepseek

In [9]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'meta_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — meta_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  MRQ-01     claude       REVISING     ✅ $0.0057
  MRQ-01     gpt4o        REVISING     ✅ $0.0052
  MRQ-01     gemini       REVISING     ✅ $0.0083
  MRQ-01     deepseek     REVISING     ✅ $0.0013
  MRQ-01     grok         REVISING     ✅ $0.0088
  MRQ-02     claude       REVISING     ✅ $0.0049
  MRQ-02     gpt4o        DEFENDING    ✅ $0.0046
  MRQ-02     gemini       REVISING     ✅ $0.0055
  MRQ-02     deepseek     REVISING     ✅ $0.0014
  MRQ-02     grok         REVISING     ✅ $0.0080
  MRQ-03     claude       REVISING     ✅ $0.0064
  MRQ-03     gpt4o        REVISING     ✅ $0.0054
  MRQ-03     gemini       REVISING     ✅ $0.0049
  MRQ-03     deepseek     REVISING     ✅ $0.0013
  MRQ-03     grok         REVISING     ✅ $0.0066
  MRQ-04     claude       REVISING     ✅ $0.0051
  MRQ-04     gpt4o        REVISING     ✅ $0.0061
  MRQ-04     gemini       REVISING     ✅ $0.0073
  MRQ-04     deepseek 

In [11]:
# Recovery — MRQ-14 claude

q = next(q for q in ALL_QUESTIONS
         if q['id'] == 'MRQ-14')

own_response = r1_lookup.get('MRQ-14', {}).get('claude')
critique     = r2_lookup.get('MRQ-14', {}).get('claude')

if own_response and critique:
    result = query_frontier(
        'claude',
        prompt_round3(q, own_response, critique),
        3, 'MRQ-14')

    result['question_id'] = 'MRQ-14'
    result['category']    = 'meta_reasoning'
    result['difficulty']  = q['difficulty']
    result['position']    = extract_position(
        result.get('response', ''))

    status = '✅' if not result['error'] else '❌'
    print(f"MRQ-14 claude recovery: {status} "
          f"{result.get('position')}")

    if not result['error']:
        mr_results = checkpoint_mgr.load(
            'round3_meta_reasoning')
        mr_results.append(result)
        checkpoint_mgr.save(
            'round3_meta_reasoning', mr_results)
        print(f"Saved ✅")
else:
    print(f"Missing R1 or R2 data")

MRQ-14 claude recovery: ✅ REVISING
Saved ✅


In [10]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'ethical_dilemma'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — ethical_dilemma
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  EDQ-01     claude       REVISING     ✅ $0.0061
  EDQ-01     gpt4o        REVISING     ✅ $0.0071
  EDQ-01     gemini       REVISING     ✅ $0.0070
  EDQ-01     deepseek     REVISING     ✅ $0.0016
  EDQ-01     grok         REVISING     ✅ $0.0090
  EDQ-02     claude       REVISING     ✅ $0.0065
  EDQ-02     gpt4o        REVISING     ✅ $0.0049
  EDQ-02     gemini       REVISING     ✅ $0.0074
  EDQ-02     deepseek     REVISING     ✅ $0.0015
  EDQ-02     grok         REVISING     ✅ $0.0086
  EDQ-03     claude       REVISING     ✅ $0.0063
  EDQ-03     gpt4o        REVISING     ✅ $0.0053
  EDQ-03     gemini       REVISING     ✅ $0.0056
  EDQ-03     deepseek     REVISING     ✅ $0.0013
  EDQ-03     grok         REVISING     ✅ $0.0090
  EDQ-04     claude       REVISING     ✅ $0.0067
  EDQ-04     gpt4o        REVISING     ✅ $0.0064
  EDQ-04     gemini       REVISING     ✅ $0.0073
  EDQ-04     deepseek

In [12]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'temporal_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — temporal_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  TRQ-01     claude       DEFENDING    ✅ $0.0038
  TRQ-01     gpt4o        DEFENDING    ✅ $0.0054
  TRQ-01     gemini       REVISING     ✅ $0.0048
  TRQ-01     deepseek     REVISING     ✅ $0.0009
  TRQ-01     grok         DEFENDING    ✅ $0.0045
  TRQ-02     claude       REVISING     ✅ $0.0052
  TRQ-02     gpt4o        DEFENDING    ✅ $0.0054
  TRQ-02     gemini       REVISING     ✅ $0.0067
  TRQ-02     deepseek     REVISING     ✅ $0.0013
  TRQ-02     grok         REVISING     ✅ $0.0075
  TRQ-03     claude       DEFENDING    ✅ $0.0050
  TRQ-03     gpt4o        DEFENDING    ✅ $0.0054
  TRQ-03     gemini       DEFENDING    ✅ $0.0042
  TRQ-03     deepseek     REVISING     ✅ $0.0012
  TRQ-03     grok         REVISING     ✅ $0.0085
  TRQ-04     claude       DEFENDING    ✅ $0.0036
  TRQ-04     gpt4o        REVISING     ✅ $0.0032
  TRQ-04     gemini       DEFENDING    ✅ $0.0074
  TRQ-04     deeps

In [13]:
# Round 3 — Defense or Revision
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'spatial_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round3_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round3_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 3 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round3_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round3_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_response = r1_lookup.get(
            q['id'], {}).get(model)
        critique     = r2_lookup.get(
            q['id'], {}).get(model)

        if not own_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not critique:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R2 critique")
            continue

        prompt = prompt_round3(
            q, own_response, critique)
        result = query_frontier(
            model, prompt, 3, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = extract_position(
            result.get('response', ''))
        result['confidence']  = extract_confidence(
            result.get('response', ''))

        round3_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round3_results)

        status   = '✅' if not result['error'] \
                   else '❌'
        position = result.get('position', 'UNKNOWN')
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round3_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round3_results),
        'responses': round3_results
    }, f, indent=2)

errors    = [r for r in round3_results
             if r.get('error')]
defending = [r for r in round3_results
             if r.get('position') == 'DEFENDING']
revising  = [r for r in round3_results
             if r.get('position') == 'REVISING']

print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round3_results)}")
print(f"  Defending: {len(defending)}")
print(f"  Revising:  {len(revising)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 3 — spatial_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  SRQ-01     claude       DEFENDING    ✅ $0.0076
  SRQ-01     gpt4o        DEFENDING    ✅ $0.0072
  SRQ-01     gemini       DEFENDING    ✅ $0.0060
  SRQ-01     deepseek     REVISING     ✅ $0.0013
  SRQ-01     grok         DEFENDING    ✅ $0.0097
  SRQ-02     claude       DEFENDING    ✅ $0.0052
  SRQ-02     gpt4o        DEFENDING    ✅ $0.0035
  SRQ-02     gemini       DEFENDING    ✅ $0.0037
  SRQ-02     deepseek     DEFENDING    ✅ $0.0011
  SRQ-02     grok         DEFENDING    ✅ $0.0059
  SRQ-03     claude       REVISING     ✅ $0.0066
  SRQ-03     gpt4o        REVISING     ✅ $0.0057
  SRQ-03     gemini       DEFENDING    ✅ $0.0078
  SRQ-03     deepseek     REVISING     ✅ $0.0023
  SRQ-03     grok         REVISING     ✅ $0.0107
  SRQ-04     claude       REVISING     ✅ $0.0053
  SRQ-04     gpt4o        REVISING     ✅ $0.0050
  SRQ-04     gemini       DEFENDING    ✅ $0.0070
  SRQ-04     deepse

In [14]:
# Round 3 Consolidation

categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

all_r3       = []
total_errors = 0
total_def    = 0
total_rev    = 0
total_unk    = 0

for cat in categories:
    data = checkpoint_mgr.load(f'round3_{cat}')
    all_r3.extend(data)
    errors = [r for r in data
              if r.get('error')]
    total_errors += len(errors)
    total_def += sum(
        1 for r in data
        if r.get('position') == 'DEFENDING')
    total_rev += sum(
        1 for r in data
        if r.get('position') == 'REVISING')
    total_unk += sum(
        1 for r in data
        if r.get('position') == 'UNKNOWN')

with open(f'{V2_RESULTS_PATH}/'
          f'round3_all_responses.json', 'w') as f:
    json.dump({
        'round':     3,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r3),
        'errors':    total_errors,
        'defending': total_def,
        'revising':  total_rev,
        'unknown':   total_unk,
        'responses': all_r3
    }, f, indent=2)

budget.save(
    f'{V2_RESULTS_PATH}/budget_r3.json')

print(f"Round 3 consolidated:")
print(f"  Total responses: {len(all_r3)}")
print(f"  Defending:       {total_def}")
print(f"  Revising:        {total_rev}")
print(f"  Unknown:         {total_unk}")
print(f"  Errors:          {total_errors}")
print(f"  Total cost:      ${budget.total:.4f}")
print(f"  Saved ✅")

Round 3 consolidated:
  Total responses: 993
  Defending:       234
  Revising:        758
  Unknown:         1
  Errors:          1
  Total cost:      $1.3884
  Saved ✅


In [15]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'adversarial'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — adversarial
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  ADV-01     claude       R3:REVISING     ✅ $0.0097
  ADV-01     gpt4o        R3:REVISING     ✅ $0.0098
  ADV-01     gemini       R3:DEFENDING    ✅ $0.0062
  ADV-01     deepseek     R3:REVISING     ✅ $0.0021
  ADV-01     grok         R3:REVISING     ✅ $0.0159
  ADV-02     claude       R3:REVISING     ✅ $0.0050
  ADV-02     gpt4o        R3:DEFENDING    ✅ $0.0064
  ADV-02     gemini       R3:REVISING     ✅ $0.0056
  ADV-02     deepseek     R3:REVISING     ✅ $0.0016
  ADV-02     grok         R3:REVISING     ✅ $0.0083
  ADV-03     claude       R3:REVISING     ✅ $0.0048
  ADV-03     gpt4o        R3:DEFENDING    ✅ $0.0060
  ADV-03     gemini       R3:REVISING     ✅ $0.0055
  ADV-03     deepseek     R3:REVISING     ✅ $0.0013
  ADV-03     grok         R3:REVISING     ✅ $0.0080
  ADV-04     claude       R3:REVISING     ✅ $0.0068
  ADV-04     gpt4o        R3:DEFENDING    ✅ $0.0074
  ADV-04     gemini   

In [16]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'logical_syllogism'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — logical_syllogism
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  LSQ-01     claude       R3:DEFENDING    ✅ $0.0047
  LSQ-01     gpt4o        R3:DEFENDING    ✅ $0.0052
  LSQ-01     gemini       R3:DEFENDING    ✅ $0.0058
  LSQ-01     deepseek     R3:REVISING     ✅ $0.0014
  LSQ-01     grok         R3:REVISING     ✅ $0.0070
  LSQ-02     claude       R3:REVISING     ✅ $0.0055
  LSQ-02     gpt4o        R3:DEFENDING    ✅ $0.0058
  LSQ-02     gemini       R3:REVISING     ✅ $0.0079
  LSQ-02     deepseek     R3:DEFENDING    ✅ $0.0015
  LSQ-02     grok         R3:DEFENDING    ✅ $0.0071
  LSQ-03     claude       R3:REVISING     ✅ $0.0056
  LSQ-03     gpt4o        R3:DEFENDING    ✅ $0.0067
  LSQ-03     gemini       R3:DEFENDING    ✅ $0.0056
  LSQ-03     deepseek     R3:REVISING     ✅ $0.0015
  LSQ-03     grok         R3:REVISING     ✅ $0.0077
  LSQ-04     claude       R3:DEFENDING    ✅ $0.0048
  LSQ-04     gpt4o        R3:DEFENDING    ✅ $0.0055
  LSQ-04     gem

In [17]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'causal_chain'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — causal_chain
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CCQ-01     claude       R3:REVISING     ✅ $0.0063
  CCQ-01     gpt4o        R3:DEFENDING    ✅ $0.0077
  CCQ-01     gemini       R3:REVISING     ✅ $0.0066
  CCQ-01     deepseek     R3:REVISING     ✅ $0.0017
  CCQ-01     grok         R3:REVISING     ✅ $0.0088
  CCQ-02     claude       R3:REVISING     ✅ $0.0065
  CCQ-02     gpt4o        R3:REVISING     ✅ $0.0076
  CCQ-02     gemini       R3:REVISING     ✅ $0.0072
  CCQ-02     deepseek     R3:REVISING     ✅ $0.0019
  CCQ-02     grok         R3:REVISING     ✅ $0.0094
  CCQ-03     claude       R3:REVISING     ✅ $0.0061
  CCQ-03     gpt4o        R3:REVISING     ✅ $0.0068
  CCQ-03     gemini       R3:REVISING     ✅ $0.0058
  CCQ-03     deepseek     R3:REVISING     ✅ $0.0015
  CCQ-03     grok         R3:REVISING     ✅ $0.0098
  CCQ-04     claude       R3:REVISING     ✅ $0.0064
  CCQ-04     gpt4o        R3:DEFENDING    ✅ $0.0069
  CCQ-04     gemini  

In [18]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'quantum_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — quantum_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  QRQ-01     claude       R3:REVISING     ✅ $0.0071
  QRQ-01     gpt4o        R3:DEFENDING    ✅ $0.0080
  QRQ-01     gemini       R3:DEFENDING    ✅ $0.0078
  QRQ-01     deepseek     R3:REVISING     ✅ $0.0016
  QRQ-01     grok         R3:REVISING     ✅ $0.0095
  QRQ-02     claude       R3:DEFENDING    ✅ $0.0075
  QRQ-02     gpt4o        R3:DEFENDING    ✅ $0.0091
  QRQ-02     gemini       R3:DEFENDING    ✅ $0.0092
  QRQ-02     deepseek     R3:REVISING     ✅ $0.0017
  QRQ-02     grok         R3:DEFENDING    ✅ $0.0091
  QRQ-03     claude       R3:REVISING     ✅ $0.0079
  QRQ-03     gpt4o        R3:DEFENDING    ✅ $0.0093
  QRQ-03     gemini       R3:REVISING     ✅ $0.0077
  QRQ-03     deepseek     R3:REVISING     ✅ $0.0023
  QRQ-03     grok         R3:REVISING     ✅ $0.0094
  QRQ-04     claude       R3:DEFENDING    ✅ $0.0094
  QRQ-04     gpt4o        R3:DEFENDING    ✅ $0.0115
  QRQ-04     gemi

In [19]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'frontier_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — frontier_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  FRQ-01     claude       R3:REVISING     ✅ $0.0061
  FRQ-01     gpt4o        R3:DEFENDING    ✅ $0.0066
  FRQ-01     gemini       R3:DEFENDING    ✅ $0.0055
  FRQ-01     deepseek     R3:REVISING     ✅ $0.0017
  FRQ-01     grok         R3:REVISING     ✅ $0.0109
  FRQ-02     claude       R3:REVISING     ✅ $0.0059
  FRQ-02     gpt4o        R3:DEFENDING    ✅ $0.0069
  FRQ-02     gemini       R3:REVISING     ✅ $0.0073
  FRQ-02     deepseek     R3:REVISING     ✅ $0.0019
  FRQ-02     grok         R3:REVISING     ✅ $0.0101
  FRQ-03     claude       R3:REVISING     ✅ $0.0064
  FRQ-03     gpt4o        R3:REVISING     ✅ $0.0077
  FRQ-03     gemini       R3:DEFENDING    ✅ $0.0055
  FRQ-03     deepseek     R3:REVISING     ✅ $0.0014
  FRQ-03     grok         R3:REVISING     ✅ $0.0087
  FRQ-04     claude       R3:REVISING     ✅ $0.0078
  FRQ-04     gpt4o        R3:REVISING     ✅ $0.0085
  FRQ-04     gem

In [20]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'probabilistic'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — probabilistic
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  PRQ-01     claude       ⚠️  peer grok no R1 response
  PRQ-01     gpt4o        R3:DEFENDING    ✅ $0.0071
  PRQ-01     gemini       R3:DEFENDING    ✅ $0.0059
  PRQ-01     deepseek     R3:DEFENDING    ✅ $0.0013
  PRQ-01     grok         ⚠️  no R1 response
  PRQ-02     claude       R3:DEFENDING    ✅ $0.0062
  PRQ-02     gpt4o        R3:DEFENDING    ✅ $0.0080
  PRQ-02     gemini       R3:REVISING     ✅ $0.0067
  PRQ-02     deepseek     R3:REVISING     ✅ $0.0012
  PRQ-02     grok         R3:DEFENDING    ✅ $0.0088
  PRQ-03     claude       R3:DEFENDING    ✅ $0.0062
  PRQ-03     gpt4o        R3:DEFENDING    ✅ $0.0074
  PRQ-03     gemini       R3:DEFENDING    ✅ $0.0083
  PRQ-03     deepseek     R3:DEFENDING    ✅ $0.0021
  PRQ-03     grok         R3:REVISING     ✅ $0.0091
  PRQ-04     claude       R3:REVISING     ✅ $0.0078
  PRQ-04     gpt4o        R3:DEFENDING    ✅ $0.0092
  PRQ-04     gemini     

In [21]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'mathematical_proof'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — mathematical_proof
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  MPQ-01     claude       R3:DEFENDING    ✅ $0.0059
  MPQ-01     gpt4o        R3:DEFENDING    ✅ $0.0066
  MPQ-01     gemini       R3:DEFENDING    ✅ $0.0067
  MPQ-01     deepseek     R3:DEFENDING    ✅ $0.0014
  MPQ-01     grok         R3:DEFENDING    ✅ $0.0077
  MPQ-02     claude       R3:DEFENDING    ✅ $0.0070
  MPQ-02     gpt4o        R3:DEFENDING    ✅ $0.0082
  MPQ-02     gemini       R3:REVISING     ✅ $0.0098
  MPQ-02     deepseek     R3:DEFENDING    ✅ $0.0017
  MPQ-02     grok         R3:REVISING     ✅ $0.0092
  MPQ-03     claude       R3:DEFENDING    ✅ $0.0067
  MPQ-03     gpt4o        R3:DEFENDING    ✅ $0.0078
  MPQ-03     gemini       R3:REVISING     ✅ $0.0062
  MPQ-03     deepseek     R3:REVISING     ✅ $0.0011
  MPQ-03     grok         R3:REVISING     ✅ $0.0086
  MPQ-04     claude       R3:DEFENDING    ✅ $0.0086
  MPQ-04     gpt4o        R3:DEFENDING    ✅ $0.0094
  MPQ-04     ge

In [22]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'counterfactual'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — counterfactual
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CFQ-01     claude       R3:REVISING     ✅ $0.0059
  CFQ-01     gpt4o        R3:REVISING     ✅ $0.0065
  CFQ-01     gemini       R3:REVISING     ✅ $0.0060
  CFQ-01     deepseek     R3:REVISING     ✅ $0.0018
  CFQ-01     grok         R3:REVISING     ✅ $0.0094
  CFQ-02     claude       R3:REVISING     ✅ $0.0054
  CFQ-02     gpt4o        R3:REVISING     ✅ $0.0061
  CFQ-02     gemini       R3:REVISING     ✅ $0.0079
  CFQ-02     deepseek     R3:REVISING     ✅ $0.0019
  CFQ-02     grok         R3:REVISING     ✅ $0.0083
  CFQ-03     claude       R3:REVISING     ✅ $0.0058
  CFQ-03     gpt4o        R3:DEFENDING    ✅ $0.0065
  CFQ-03     gemini       R3:REVISING     ✅ $0.0078
  CFQ-03     deepseek     R3:REVISING     ✅ $0.0019
  CFQ-03     grok         R3:REVISING     ✅ $0.0082
  CFQ-04     claude       R3:REVISING     ✅ $0.0069
  CFQ-04     gpt4o        R3:REVISING     ✅ $0.0075
  CFQ-04     gemini

In [8]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'meta_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — meta_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  MRQ-01     claude       R3:REVISING     ✅ $0.0056
  MRQ-01     gpt4o        R3:REVISING     ✅ $0.0064
  MRQ-01     gemini       R3:REVISING     ✅ $0.0092
  MRQ-01     deepseek     R3:REVISING     ✅ $0.0015
  MRQ-01     grok         R3:REVISING     ✅ $0.0100
  MRQ-02     claude       R3:REVISING     ✅ $0.0054
  MRQ-02     gpt4o        R3:DEFENDING    ✅ $0.0063
  MRQ-02     gemini       R3:REVISING     ✅ $0.0060
  MRQ-02     deepseek     R3:REVISING     ✅ $0.0014
  MRQ-02     grok         R3:REVISING     ✅ $0.0091
  MRQ-03     claude       R3:REVISING     ✅ $0.0055
  MRQ-03     gpt4o        R3:REVISING     ✅ $0.0074
  MRQ-03     gemini       R3:REVISING     ✅ $0.0061
  MRQ-03     deepseek     R3:REVISING     ✅ $0.0014
  MRQ-03     grok         R3:REVISING     ✅ $0.0081
  MRQ-04     claude       R3:REVISING     ✅ $0.0055
  MRQ-04     gpt4o        R3:REVISING     ✅ $0.0069
  MRQ-04     gemini 

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6875.93ms


  MRQ-07     gemini       R3:REVISING     ✅ $0.0059
  MRQ-07     deepseek     R3:REVISING     ✅ $0.0017
  MRQ-07     grok         R3:REVISING     ✅ $0.0075
  MRQ-08     claude       R3:REVISING     ✅ $0.0061
  MRQ-08     gpt4o        R3:REVISING     ✅ $0.0077
  MRQ-08     gemini       R3:REVISING     ✅ $0.0077
  MRQ-08     deepseek     R3:REVISING     ✅ $0.0021
  MRQ-08     grok         R3:REVISING     ✅ $0.0087
  MRQ-09     claude       R3:REVISING     ✅ $0.0056
  MRQ-09     gpt4o        R3:REVISING     ✅ $0.0069
  MRQ-09     gemini       R3:DEFENDING    ✅ $0.0062
  MRQ-09     deepseek     R3:REVISING     ✅ $0.0020
  MRQ-09     grok         R3:REVISING     ✅ $0.0095
  MRQ-10     claude       R3:REVISING     ✅ $0.0071
  MRQ-10     gpt4o        R3:REVISING     ✅ $0.0085


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1997.43ms


  MRQ-10     gemini       R3:REVISING     ✅ $0.0068
  MRQ-10     deepseek     R3:REVISING     ✅ $0.0021
  MRQ-10     grok         R3:REVISING     ✅ $0.0091
  MRQ-11     claude       R3:REVISING     ✅ $0.0073
  MRQ-11     gpt4o        R3:REVISING     ✅ $0.0080


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6264.85ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3464.54ms


  MRQ-11     gemini       R3:REVISING     ✅ $0.0067
  MRQ-11     deepseek     R3:REVISING     ✅ $0.0022
  MRQ-11     grok         R3:REVISING     ✅ $0.0110
  MRQ-12     claude       R3:REVISING     ✅ $0.0069
  MRQ-12     gpt4o        R3:DEFENDING    ✅ $0.0081
  MRQ-12     gemini       R3:REVISING     ✅ $0.0073
  MRQ-12     deepseek     R3:REVISING     ✅ $0.0018
  MRQ-12     grok         R3:REVISING     ✅ $0.0096
  MRQ-13     claude       R3:REVISING     ✅ $0.0068
  MRQ-13     gpt4o        R3:REVISING     ✅ $0.0073
  MRQ-13     gemini       R3:REVISING     ✅ $0.0078
  MRQ-13     deepseek     R3:REVISING     ✅ $0.0024
  MRQ-13     grok         R3:REVISING     ✅ $0.0102
  MRQ-14     claude       R3:REVISING     ✅ $0.0063
  MRQ-14     gpt4o        R3:REVISING     ✅ $0.0073
  MRQ-14     gemini       R3:DEFENDING    ✅ $0.0062
  MRQ-14     deepseek     R3:REVISING     ✅ $0.0018
  MRQ-14     grok         R3:REVISING     ✅ $0.0099
  MRQ-15     claude       R3:REVISING     ✅ $0.0070
  MRQ-15    

In [9]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'ethical_dilemma'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — ethical_dilemma
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  EDQ-01     claude       R3:REVISING     ✅ $0.0068
  EDQ-01     gpt4o        R3:REVISING     ✅ $0.0079
  EDQ-01     gemini       R3:REVISING     ✅ $0.0082
  EDQ-01     deepseek     R3:REVISING     ✅ $0.0022
  EDQ-01     grok         R3:REVISING     ✅ $0.0091
  EDQ-02     claude       R3:REVISING     ✅ $0.0065
  EDQ-02     gpt4o        R3:REVISING     ✅ $0.0065
  EDQ-02     gemini       R3:REVISING     ✅ $0.0056
  EDQ-02     deepseek     R3:REVISING     ✅ $0.0020
  EDQ-02     grok         R3:REVISING     ✅ $0.0090
  EDQ-03     claude       R3:REVISING     ✅ $0.0060
  EDQ-03     gpt4o        R3:REVISING     ✅ $0.0068
  EDQ-03     gemini       R3:REVISING     ✅ $0.0054
  EDQ-03     deepseek     R3:REVISING     ✅ $0.0012
  EDQ-03     grok         R3:REVISING     ✅ $0.0092
  EDQ-04     claude       R3:REVISING     ✅ $0.0062
  EDQ-04     gpt4o        R3:REVISING     ✅ $0.0079


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5235.03ms


  EDQ-04     gemini       R3:REVISING     ✅ $0.0067
  EDQ-04     deepseek     R3:REVISING     ✅ $0.0014
  EDQ-04     grok         R3:REVISING     ✅ $0.0089
  EDQ-05     claude       R3:REVISING     ✅ $0.0066
  EDQ-05     gpt4o        R3:DEFENDING    ✅ $0.0068
  EDQ-05     gemini       R3:REVISING     ✅ $0.0075
  EDQ-05     deepseek     R3:REVISING     ✅ $0.0022
  EDQ-05     grok         R3:REVISING     ✅ $0.0109
  EDQ-06     claude       R3:REVISING     ✅ $0.0059
  EDQ-06     gpt4o        R3:REVISING     ✅ $0.0063
  EDQ-06     gemini       R3:DEFENDING    ✅ $0.0076
  EDQ-06     deepseek     R3:REVISING     ✅ $0.0019
  EDQ-06     grok         R3:REVISING     ✅ $0.0089
  EDQ-07     claude       R3:REVISING     ✅ $0.0068
  EDQ-07     gpt4o        R3:REVISING     ✅ $0.0073
  EDQ-07     gemini       R3:REVISING     ✅ $0.0065
  EDQ-07     deepseek     R3:REVISING     ✅ $0.0023
  EDQ-07     grok         R3:REVISING     ✅ $0.0107
  EDQ-08     claude       R3:REVISING     ✅ $0.0056
  EDQ-08    

In [11]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'temporal_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — temporal_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  TRQ-01     claude       R3:DEFENDING    ✅ $0.0036
  TRQ-01     gpt4o        R3:DEFENDING    ✅ $0.0048
  TRQ-01     gemini       R3:REVISING     ✅ $0.0052
  TRQ-01     deepseek     R3:REVISING     ✅ $0.0010
  TRQ-01     grok         R3:DEFENDING    ✅ $0.0053
  TRQ-02     claude       R3:REVISING     ✅ $0.0055
  TRQ-02     gpt4o        R3:DEFENDING    ✅ $0.0072
  TRQ-02     gemini       R3:REVISING     ✅ $0.0066
  TRQ-02     deepseek     R3:REVISING     ✅ $0.0013
  TRQ-02     grok         R3:REVISING     ✅ $0.0087
  TRQ-03     claude       R3:DEFENDING    ✅ $0.0059
  TRQ-03     gpt4o        R3:DEFENDING    ✅ $0.0061
  TRQ-03     gemini       R3:DEFENDING    ✅ $0.0055
  TRQ-03     deepseek     R3:REVISING     ✅ $0.0013
  TRQ-03     grok         R3:REVISING     ✅ $0.0094
  TRQ-04     claude       R3:DEFENDING    ✅ $0.0049
  TRQ-04     gpt4o        R3:REVISING     ✅ $0.0041
  TRQ-04     gem

In [12]:
# Round 4 — Mirror Self-Assessment
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'spatial_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round4_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round4_results  = checkpoint_mgr.load(
    checkpoint_name)

print(f"Round 4 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round4_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round4_results)}")
print(f"Starting...\n")

# Load R3 lookup
r3_lookup = {}
for cat in [CURRENT_CATEGORY]:
    data = checkpoint_mgr.load(
        f'round3_{cat}')
    for r in data:
        qid   = r.get('question_id')
        model = r.get('model')
        pos   = r.get('position', 'UNKNOWN')
        if qid and model:
            if qid not in r3_lookup:
                r3_lookup[qid] = {}
            r3_lookup[qid][model] = {
                'response': r.get('response'),
                'position': pos
            }

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        peer      = PEER_ASSIGNMENTS[model]
        peer_r1   = r1_lookup.get(
            q['id'], {}).get(peer)
        ground_truth = q.get('ground_truth', '')

        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if not peer_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  peer {peer} no R1 response")
            continue

        confidence = extract_confidence(own_r1)

        prompt = prompt_round4(
            q, own_r1, peer_r1, ground_truth)
        result = query_frontier(
            model, prompt, 4, q['id'])

        result['question_id']  = q['id']
        result['category']     = q['category']
        result['difficulty']   = q['difficulty']
        result['r1_confidence'] = confidence
        result['r3_position']  = r3_lookup.get(
            q['id'], {}).get(model, {}).get(
            'position', 'UNKNOWN')

        round4_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round4_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"R3:{result['r3_position']:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round4_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round4_results),
        'responses': round4_results
    }, f, indent=2)

errors = [r for r in round4_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round4_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 4 — spatial_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  SRQ-01     claude       R3:DEFENDING    ✅ $0.0071
  SRQ-01     gpt4o        R3:DEFENDING    ✅ $0.0085
  SRQ-01     gemini       R3:DEFENDING    ✅ $0.0073
  SRQ-01     deepseek     R3:REVISING     ✅ $0.0016
  SRQ-01     grok         R3:DEFENDING    ✅ $0.0098
  SRQ-02     claude       R3:DEFENDING    ✅ $0.0045
  SRQ-02     gpt4o        R3:DEFENDING    ✅ $0.0049
  SRQ-02     gemini       R3:DEFENDING    ✅ $0.0059
  SRQ-02     deepseek     R3:DEFENDING    ✅ $0.0010
  SRQ-02     grok         R3:DEFENDING    ✅ $0.0067
  SRQ-03     claude       R3:REVISING     ✅ $0.0066
  SRQ-03     gpt4o        R3:REVISING     ✅ $0.0073
  SRQ-03     gemini       R3:DEFENDING    ✅ $0.0171
  SRQ-03     deepseek     R3:REVISING     ✅ $0.0027
  SRQ-03     grok         R3:REVISING     ✅ $0.0122
  SRQ-04     claude       R3:REVISING     ✅ $0.0067
  SRQ-04     gpt4o        R3:REVISING     ✅ $0.0061
  SRQ-04     gemi

In [13]:
categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

print("Round 4 checkpoint status:")
for cat in categories:
    data = checkpoint_mgr.load(f'round4_{cat}')
    print(f"  {cat:<25} {len(data)} responses")

Round 4 checkpoint status:
  adversarial               100 responses
  logical_syllogism         100 responses
  causal_chain              100 responses
  probabilistic             98 responses
  quantum_reasoning         75 responses
  frontier_reasoning        75 responses
  mathematical_proof        100 responses
  counterfactual            100 responses
  meta_reasoning            75 responses
  ethical_dilemma           71 responses
  temporal_reasoning        48 responses
  spatial_reasoning         50 responses


In [14]:
# Round 4 Consolidation

categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

all_r4       = []
total_errors = 0

for cat in categories:
    data = checkpoint_mgr.load(f'round4_{cat}')
    all_r4.extend(data)
    errors = [r for r in data
              if r.get('error')]
    total_errors += len(errors)

with open(f'{V2_RESULTS_PATH}/'
          f'round4_all_responses.json', 'w') as f:
    json.dump({
        'round':     4,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r4),
        'errors':    total_errors,
        'responses': all_r4
    }, f, indent=2)

budget.save(
    f'{V2_RESULTS_PATH}/budget_r4.json')

print(f"Round 4 consolidated:")
print(f"  Total responses: {len(all_r4)}")
print(f"  Total errors:    {total_errors}")
print(f"  Total cost:      ${budget.total:.4f}")
print(f"  Total calls:     {budget.calls}")
print(f"  Saved ✅")

Round 4 consolidated:
  Total responses: 992
  Total errors:    0
  Total cost:      $1.5419
  Total calls:     244
  Saved ✅


In [15]:
# Pre-Round 5 verification

# Check prompt defined
try:
    test = prompt_round5(
        ALL_QUESTIONS[0],
        'test response',
        'test r3 response',
        'DEFENDING')
    print(f"prompt_round5: ✅ defined")
except:
    print(f"prompt_round5: ❌ run Cell 5 first")

# Check R3 data available
r3_check = checkpoint_mgr.load(
    'round3_adversarial')
print(f"R3 adversarial: {len(r3_check)} responses")

# Budget status
print(f"\nBudget status:")
budget.status()

prompt_round5: ✅ defined
R3 adversarial: 100 responses

Budget status:
  Spent:  $1.5419
  Limit:  $200.00
  Calls:  244
  By model:
    grok         $0.4370
    gemini       $0.3750
    gpt4o        $0.3480
    claude       $0.2889
    deepseek     $0.0929


In [16]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'adversarial'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — adversarial
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  ADV-01     claude       REVISING     ✅ $0.0098
  ADV-01     gpt4o        REVISING     ✅ $0.0069
  ADV-01     gemini       DEFENDING    ✅ $0.0087
  ADV-01     deepseek     REVISING     ✅ $0.0021
  ADV-01     grok         REVISING     ✅ $0.0186
  ADV-02     claude       REVISING     ✅ $0.0063
  ADV-02     gpt4o        DEFENDING    ✅ $0.0059
  ADV-02     gemini       REVISING     ✅ $0.0082
  ADV-02     deepseek     REVISING     ✅ $0.0018
  ADV-02     grok         REVISING     ✅ $0.0099
  ADV-03     claude       REVISING     ✅ $0.0069
  ADV-03     gpt4o        DEFENDING    ✅ $0.0061
  ADV-03     gemini       REVISING     ✅ $0.0061
  ADV-03     deepseek     REVISING     ✅ $0.0019
  ADV-03     grok         REVISING     ✅ $0.0094
  ADV-04     claude       REVISING     ✅ $0.0087
  ADV-04     gpt4o        DEFENDING    ✅ $0.0063
  ADV-04     gemini       REVISING     ✅ $0.0077
  ADV-04     deepseek   

In [8]:
# Round 5 adversarial — Claude recovery

failed_ids = [
    'ADV-10', 'ADV-11', 'ADV-12', 'ADV-13',
    'ADV-14', 'ADV-15', 'ADV-16', 'ADV-17',
    'ADV-18', 'ADV-19', 'ADV-20'
]

r3_data_adv = checkpoint_mgr.load(
    'round3_adversarial')
r3_map = {}
for r in r3_data_adv:
    qid = r.get('question_id')
    model = r.get('model')
    if qid and model:
        if qid not in r3_map:
            r3_map[qid] = {}
        r3_map[qid][model] = {
            'response': r.get('response'),
            'position': r.get('position')
        }

r5_adv = checkpoint_mgr.load(
    'round5_adversarial')
recovered = []

for qid in failed_ids:
    q        = next(q for q in ALL_QUESTIONS
                   if q['id'] == qid)
    own_r1   = r1_lookup.get(qid, {}).get('claude')
    r3_entry = r3_map.get(qid, {}).get('claude', {})
    r3_resp  = r3_entry.get('response')
    position = r3_entry.get('position', 'UNKNOWN')

    if not own_r1 or not r3_resp:
        print(f"{qid} claude: missing data")
        continue

    time.sleep(2)
    result = query_frontier(
        'claude',
        prompt_round5(q, own_r1, r3_resp, position),
        5, qid)

    result['question_id'] = qid
    result['category']    = 'adversarial'
    result['difficulty']  = q['difficulty']
    result['position']    = position

    status = '✅' if not result['error'] else '❌'
    print(f"{qid} claude recovery: {status} "
          f"${result['cost']:.4f}")

    if not result['error']:
        recovered.append(result)

r5_adv.extend(recovered)
checkpoint_mgr.save('round5_adversarial', r5_adv)
print(f"\nRecovered: {len(recovered)}")
print(f"Total R5 adversarial: {len(r5_adv)}")

ADV-10 claude recovery: ✅ $0.0079
ADV-11 claude recovery: ✅ $0.0076
ADV-12 claude recovery: ✅ $0.0069
ADV-13 claude recovery: ✅ $0.0082
ADV-14 claude recovery: ✅ $0.0071
ADV-15 claude recovery: ✅ $0.0089
ADV-16 claude recovery: ✅ $0.0088
ADV-17 claude recovery: ✅ $0.0090
ADV-18 claude recovery: ✅ $0.0085
ADV-19 claude recovery: ✅ $0.0094
ADV-20 claude recovery: ✅ $0.0093

Recovered: 11
Total R5 adversarial: 111


In [9]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'logical_syllogism'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — logical_syllogism
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  LSQ-01     claude       DEFENDING    ✅ $0.0067
  LSQ-01     gpt4o        DEFENDING    ✅ $0.0047
  LSQ-01     gemini       DEFENDING    ✅ $0.0082
  LSQ-01     deepseek     REVISING     ✅ $0.0016
  LSQ-01     grok         REVISING     ✅ $0.0089
  LSQ-02     claude       REVISING     ✅ $0.0072
  LSQ-02     gpt4o        DEFENDING    ✅ $0.0047
  LSQ-02     gemini       REVISING     ✅ $0.0080
  LSQ-02     deepseek     DEFENDING    ✅ $0.0014
  LSQ-02     grok         DEFENDING    ✅ $0.0081
  LSQ-03     claude       REVISING     ✅ $0.0074
  LSQ-03     gpt4o        DEFENDING    ✅ $0.0061
  LSQ-03     gemini       DEFENDING    ✅ $0.0069
  LSQ-03     deepseek     REVISING     ✅ $0.0016
  LSQ-03     grok         REVISING     ✅ $0.0085
  LSQ-04     claude       DEFENDING    ✅ $0.0059
  LSQ-04     gpt4o        DEFENDING    ✅ $0.0049
  LSQ-04     gemini       DEFENDING    ✅ $0.0066
  LSQ-04     deeps

In [10]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'causal_chain'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — causal_chain
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CCQ-01     claude       REVISING     ✅ $0.0101
  CCQ-01     gpt4o        DEFENDING    ✅ $0.0060
  CCQ-01     gemini       REVISING     ✅ $0.0073
  CCQ-01     deepseek     REVISING     ✅ $0.0020
  CCQ-01     grok         REVISING     ✅ $0.0105
  CCQ-02     claude       REVISING     ✅ $0.0090
  CCQ-02     gpt4o        REVISING     ✅ $0.0068
  CCQ-02     gemini       REVISING     ✅ $0.0090
  CCQ-02     deepseek     REVISING     ✅ $0.0019
  CCQ-02     grok         REVISING     ✅ $0.0113
  CCQ-03     claude       REVISING     ✅ $0.0086
  CCQ-03     gpt4o        REVISING     ✅ $0.0063
  CCQ-03     gemini       REVISING     ✅ $0.0060
  CCQ-03     deepseek     REVISING     ✅ $0.0020
  CCQ-03     grok         REVISING     ✅ $0.0116
  CCQ-04     claude       REVISING     ✅ $0.0080
  CCQ-04     gpt4o        DEFENDING    ✅ $0.0068
  CCQ-04     gemini       REVISING     ✅ $0.0067
  CCQ-04     deepseek  

In [11]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'probabilistic'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — probabilistic
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  PRQ-01     claude       ⚠️  unknown R3 position
  PRQ-01     gpt4o        DEFENDING    ✅ $0.0067
  PRQ-01     gemini       DEFENDING    ✅ $0.0070
  PRQ-01     deepseek     DEFENDING    ✅ $0.0013
  PRQ-01     grok         ⚠️  no R1 response
  PRQ-02     claude       DEFENDING    ✅ $0.0079
  PRQ-02     gpt4o        DEFENDING    ✅ $0.0076
  PRQ-02     gemini       REVISING     ✅ $0.0088
  PRQ-02     deepseek     REVISING     ✅ $0.0015
  PRQ-02     grok         DEFENDING    ✅ $0.0100
  PRQ-03     claude       DEFENDING    ✅ $0.0065
  PRQ-03     gpt4o        DEFENDING    ✅ $0.0070
  PRQ-03     gemini       DEFENDING    ✅ $0.0088
  PRQ-03     deepseek     DEFENDING    ✅ $0.0014
  PRQ-03     grok         REVISING     ✅ $0.0106
  PRQ-04     claude       REVISING     ✅ $0.0086
  PRQ-04     gpt4o        DEFENDING    ✅ $0.0085
  PRQ-04     gemini       DEFENDING    ✅ $0.0082
  PRQ-04     deepseek    

In [12]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'quantum_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — quantum_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  QRQ-01     claude       REVISING     ✅ $0.0098
  QRQ-01     gpt4o        DEFENDING    ✅ $0.0080
  QRQ-01     gemini       DEFENDING    ✅ $0.0075
  QRQ-01     deepseek     REVISING     ✅ $0.0020
  QRQ-01     grok         REVISING     ✅ $0.0117
  QRQ-02     claude       DEFENDING    ✅ $0.0088
  QRQ-02     gpt4o        DEFENDING    ✅ $0.0085
  QRQ-02     gemini       DEFENDING    ✅ $0.0093
  QRQ-02     deepseek     REVISING     ✅ $0.0016
  QRQ-02     grok         DEFENDING    ✅ $0.0104
  QRQ-03     claude       REVISING     ✅ $0.0107
  QRQ-03     gpt4o        DEFENDING    ✅ $0.0091
  QRQ-03     gemini       REVISING     ✅ $0.0108
  QRQ-03     deepseek     REVISING     ✅ $0.0020
  QRQ-03     grok         REVISING     ✅ $0.0121
  QRQ-04     claude       DEFENDING    ✅ $0.0118
  QRQ-04     gpt4o        DEFENDING    ✅ $0.0090
  QRQ-04     gemini       REVISING     ✅ $0.0092
  QRQ-04     deepse

In [13]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'frontier_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — frontier_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  FRQ-01     claude       REVISING     ✅ $0.0077
  FRQ-01     gpt4o        DEFENDING    ✅ $0.0060
  FRQ-01     gemini       DEFENDING    ✅ $0.0083
  FRQ-01     deepseek     REVISING     ✅ $0.0020
  FRQ-01     grok         REVISING     ✅ $0.0128
  FRQ-02     claude       REVISING     ✅ $0.0080
  FRQ-02     gpt4o        DEFENDING    ✅ $0.0068
  FRQ-02     gemini       REVISING     ✅ $0.0082
  FRQ-02     deepseek     REVISING     ✅ $0.0019
  FRQ-02     grok         REVISING     ✅ $0.0113
  FRQ-03     claude       REVISING     ✅ $0.0082
  FRQ-03     gpt4o        REVISING     ✅ $0.0066
  FRQ-03     gemini       DEFENDING    ✅ $0.0064
  FRQ-03     deepseek     REVISING     ✅ $0.0017
  FRQ-03     grok         REVISING     ✅ $0.0110
  FRQ-04     claude       REVISING     ✅ $0.0112
  FRQ-04     gpt4o        REVISING     ✅ $0.0076
  FRQ-04     gemini       REVISING     ✅ $0.0103
  FRQ-04     deeps

In [14]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'mathematical_proof'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — mathematical_proof
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  MPQ-01     claude       DEFENDING    ✅ $0.0078
  MPQ-01     gpt4o        DEFENDING    ✅ $0.0061
  MPQ-01     gemini       DEFENDING    ✅ $0.0081
  MPQ-01     deepseek     DEFENDING    ✅ $0.0012
  MPQ-01     grok         DEFENDING    ✅ $0.0087
  MPQ-02     claude       DEFENDING    ✅ $0.0081
  MPQ-02     gpt4o        DEFENDING    ✅ $0.0071
  MPQ-02     gemini       REVISING     ✅ $0.0093
  MPQ-02     deepseek     DEFENDING    ✅ $0.0014
  MPQ-02     grok         REVISING     ✅ $0.0122
  MPQ-03     claude       DEFENDING    ✅ $0.0077
  MPQ-03     gpt4o        DEFENDING    ✅ $0.0073
  MPQ-03     gemini       REVISING     ✅ $0.0074
  MPQ-03     deepseek     REVISING     ✅ $0.0015
  MPQ-03     grok         REVISING     ✅ $0.0110
  MPQ-04     claude       DEFENDING    ✅ $0.0105
  MPQ-04     gpt4o        DEFENDING    ✅ $0.0079
  MPQ-04     gemini       DEFENDING    ✅ $0.0102
  MPQ-04     deep

In [15]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'counterfactual'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — counterfactual
Questions:  20
Models:     5
Completed:  0
Remaining:  100
Starting...

  CFQ-01     claude       REVISING     ✅ $0.0083
  CFQ-01     gpt4o        REVISING     ✅ $0.0052
  CFQ-01     gemini       REVISING     ✅ $0.0088
  CFQ-01     deepseek     REVISING     ✅ $0.0019
  CFQ-01     grok         REVISING     ✅ $0.0106
  CFQ-02     claude       REVISING     ✅ $0.0083
  CFQ-02     gpt4o        REVISING     ✅ $0.0061
  CFQ-02     gemini       REVISING     ✅ $0.0094
  CFQ-02     deepseek     REVISING     ✅ $0.0016
  CFQ-02     grok         REVISING     ✅ $0.0093
  CFQ-03     claude       REVISING     ✅ $0.0075
  CFQ-03     gpt4o        DEFENDING    ✅ $0.0051
  CFQ-03     gemini       REVISING     ✅ $0.0101
  CFQ-03     deepseek     REVISING     ✅ $0.0019
  CFQ-03     grok         REVISING     ✅ $0.0097
  CFQ-04     claude       REVISING     ✅ $0.0092
  CFQ-04     gpt4o        REVISING     ✅ $0.0053
  CFQ-04     gemini       REVISING     ✅ $0.0112
  CFQ-04     deepseek

In [16]:
categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

print("Round 5 checkpoint status:")
for cat in categories:
    data = checkpoint_mgr.load(f'round5_{cat}')
    print(f"  {cat:<25} {len(data)} responses")

Round 5 checkpoint status:
  adversarial               111 responses
  logical_syllogism         100 responses
  causal_chain              100 responses
  probabilistic             98 responses
  quantum_reasoning         75 responses
  frontier_reasoning        75 responses
  mathematical_proof        100 responses
  counterfactual            100 responses
  meta_reasoning            0 responses
  ethical_dilemma           0 responses
  temporal_reasoning        0 responses
  spatial_reasoning         0 responses


In [17]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'meta_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — meta_reasoning
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  MRQ-01     claude       REVISING     ✅ $0.0074
  MRQ-01     gpt4o        REVISING     ✅ $0.0058
  MRQ-01     gemini       REVISING     ✅ $0.0089
  MRQ-01     deepseek     REVISING     ✅ $0.0017
  MRQ-01     grok         REVISING     ✅ $0.0109
  MRQ-02     claude       REVISING     ✅ $0.0069
  MRQ-02     gpt4o        DEFENDING    ✅ $0.0054
  MRQ-02     gemini       REVISING     ✅ $0.0080
  MRQ-02     deepseek     REVISING     ✅ $0.0018
  MRQ-02     grok         REVISING     ✅ $0.0101
  MRQ-03     claude       REVISING     ✅ $0.0086
  MRQ-03     gpt4o        REVISING     ✅ $0.0060
  MRQ-03     gemini       REVISING     ✅ $0.0077
  MRQ-03     deepseek     REVISING     ✅ $0.0018
  MRQ-03     grok         REVISING     ✅ $0.0096
  MRQ-04     claude       REVISING     ✅ $0.0076
  MRQ-04     gpt4o        REVISING     ✅ $0.0067
  MRQ-04     gemini       REVISING     ✅ $0.0087
  MRQ-04     deepseek 

In [18]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'ethical_dilemma'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — ethical_dilemma
Questions:  15
Models:     5
Completed:  0
Remaining:  75
Starting...

  EDQ-01     claude       REVISING     ✅ $0.0080
  EDQ-01     gpt4o        REVISING     ✅ $0.0075
  EDQ-01     gemini       REVISING     ✅ $0.0089
  EDQ-01     deepseek     REVISING     ✅ $0.0018
  EDQ-01     grok         REVISING     ✅ $0.0119
  EDQ-02     claude       REVISING     ✅ $0.0088
  EDQ-02     gpt4o        REVISING     ✅ $0.0055
  EDQ-02     gemini       REVISING     ✅ $0.0083
  EDQ-02     deepseek     REVISING     ✅ $0.0019
  EDQ-02     grok         REVISING     ✅ $0.0111
  EDQ-03     claude       REVISING     ✅ $0.0083
  EDQ-03     gpt4o        REVISING     ✅ $0.0059
  EDQ-03     gemini       REVISING     ✅ $0.0076
  EDQ-03     deepseek     REVISING     ✅ $0.0018
  EDQ-03     grok         REVISING     ✅ $0.0119
  EDQ-04     claude       REVISING     ✅ $0.0100
  EDQ-04     gpt4o        REVISING     ✅ $0.0068
  EDQ-04     gemini       REVISING     ✅ $0.0082
  EDQ-04     deepseek

In [19]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'temporal_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — temporal_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  TRQ-01     claude       DEFENDING    ✅ $0.0054
  TRQ-01     gpt4o        DEFENDING    ✅ $0.0059
  TRQ-01     gemini       REVISING     ✅ $0.0067
  TRQ-01     deepseek     REVISING     ✅ $0.0012
  TRQ-01     grok         DEFENDING    ✅ $0.0071
  TRQ-02     claude       REVISING     ✅ $0.0068
  TRQ-02     gpt4o        DEFENDING    ✅ $0.0058
  TRQ-02     gemini       REVISING     ✅ $0.0086
  TRQ-02     deepseek     REVISING     ✅ $0.0013
  TRQ-02     grok         REVISING     ✅ $0.0098
  TRQ-03     claude       DEFENDING    ✅ $0.0067
  TRQ-03     gpt4o        DEFENDING    ✅ $0.0059
  TRQ-03     gemini       DEFENDING    ✅ $0.0083
  TRQ-03     deepseek     REVISING     ✅ $0.0015
  TRQ-03     grok         REVISING     ✅ $0.0110
  TRQ-04     claude       DEFENDING    ✅ $0.0050
  TRQ-04     gpt4o        REVISING     ✅ $0.0039
  TRQ-04     gemini       DEFENDING    ✅ $0.0099
  TRQ-04     deeps

In [20]:
# Round 5 — Mechanistic Trace
# Change CURRENT_CATEGORY before each run

CURRENT_CATEGORY = 'spatial_reasoning'

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

category_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] == CURRENT_CATEGORY
]

checkpoint_name = f"round5_{CURRENT_CATEGORY}"
completed_ids   = checkpoint_mgr.get_completed_ids(
    checkpoint_name)
round5_results  = checkpoint_mgr.load(
    checkpoint_name)

# Load R3 positions for this category
r3_positions = {}
r3_data = checkpoint_mgr.load(
    f'round3_{CURRENT_CATEGORY}')
for r in r3_data:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

print(f"Round 5 — {CURRENT_CATEGORY}")
print(f"Questions:  {len(category_questions)}")
print(f"Models:     {len(SUBJECT_MODELS)}")
print(f"Completed:  {len(round5_results)}")
print(f"Remaining:  "
      f"{len(category_questions) * len(SUBJECT_MODELS) - len(round5_results)}")
print(f"Starting...\n")

for q in category_questions:
    for model in SUBJECT_MODELS:
        call_id = f"{q['id']}_{model}"

        if call_id in completed_ids:
            continue

        own_r1    = r1_lookup.get(
            q['id'], {}).get(model)
        position  = r3_positions.get(
            q['id'], {}).get(model, 'UNKNOWN')

        # Only run Round 5 for models that
        # have a valid R1 response and position
        if not own_r1:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R1 response")
            continue

        if position == 'UNKNOWN':
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  unknown R3 position")
            continue

        r3_response = None
        r3_data_cat = checkpoint_mgr.load(
            f'round3_{CURRENT_CATEGORY}')
        for r in r3_data_cat:
            if (r.get('question_id') == q['id']
                    and r.get('model') == model):
                r3_response = r.get('response')
                break

        if not r3_response:
            print(f"  {q['id']:<10} {model:<12} "
                  f"⚠️  no R3 response")
            continue

        prompt = prompt_round5(
            q, own_r1, r3_response, position)
        result = query_frontier(
            model, prompt, 5, q['id'])

        result['question_id'] = q['id']
        result['category']    = q['category']
        result['difficulty']  = q['difficulty']
        result['position']    = position

        round5_results.append(result)
        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, round5_results)

        status = '✅' if not result['error'] \
                 else '❌'
        print(f"  {q['id']:<10} {model:<12} "
              f"{position:<12} "
              f"{status} "
              f"${result['cost']:.4f}")

        time.sleep(1)

out_file = (f"{V2_RESULTS_PATH}/"
            f"round5_{CURRENT_CATEGORY}.json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(round5_results),
        'responses': round5_results
    }, f, indent=2)

errors = [r for r in round5_results
          if r.get('error')]
print(f"\nCategory complete: {CURRENT_CATEGORY}")
print(f"  Responses: {len(round5_results)}")
print(f"  Errors:    {len(errors)}")
print(f"  Saved:     {out_file}")
budget.status()

Round 5 — spatial_reasoning
Questions:  10
Models:     5
Completed:  0
Remaining:  50
Starting...

  SRQ-01     claude       DEFENDING    ✅ $0.0085
  SRQ-01     gpt4o        DEFENDING    ✅ $0.0071
  SRQ-01     gemini       DEFENDING    ✅ $0.0077
  SRQ-01     deepseek     REVISING     ✅ $0.0015
  SRQ-01     grok         DEFENDING    ✅ $0.0123
  SRQ-02     claude       DEFENDING    ✅ $0.0067
  SRQ-02     gpt4o        DEFENDING    ✅ $0.0041
  SRQ-02     gemini       DEFENDING    ✅ $0.0066
  SRQ-02     deepseek     DEFENDING    ✅ $0.0012
  SRQ-02     grok         DEFENDING    ✅ $0.0091
  SRQ-03     claude       REVISING     ✅ $0.0083
  SRQ-03     gpt4o        REVISING     ✅ $0.0061
  SRQ-03     gemini       DEFENDING    ✅ $0.0088
  SRQ-03     deepseek     REVISING     ✅ $0.0025
  SRQ-03     grok         REVISING     ✅ $0.0146
  SRQ-04     claude       REVISING     ✅ $0.0067
  SRQ-04     gpt4o        REVISING     ✅ $0.0054
  SRQ-04     gemini       DEFENDING    ✅ $0.0096
  SRQ-04     deepse

In [21]:
# Round 5 Consolidation

categories = [
    'adversarial', 'logical_syllogism',
    'causal_chain', 'probabilistic',
    'quantum_reasoning', 'frontier_reasoning',
    'mathematical_proof', 'counterfactual',
    'meta_reasoning', 'ethical_dilemma',
    'temporal_reasoning', 'spatial_reasoning'
]

all_r5       = []
total_errors = 0

for cat in categories:
    data = checkpoint_mgr.load(f'round5_{cat}')
    all_r5.extend(data)
    errors = [r for r in data
              if r.get('error')]
    total_errors += len(errors)

with open(f'{V2_RESULTS_PATH}/'
          f'round5_all_responses.json', 'w') as f:
    json.dump({
        'round':     5,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(all_r5),
        'errors':    total_errors,
        'responses': all_r5
    }, f, indent=2)

budget.save(
    f'{V2_RESULTS_PATH}/budget_r5.json')

print(f"Round 5 consolidated:")
print(f"  Total responses: {len(all_r5)}")
print(f"  Total errors:    {total_errors}")
print(f"  Total cost:      ${budget.total:.4f}")
print(f"  Total calls:     {budget.calls}")
print(f"  Saved ✅")

Round 5 consolidated:
  Total responses: 1002
  Total errors:    11
  Total cost:      $6.7055
  Total calls:     902
  Saved ✅


In [8]:
# Full council run verification

rounds = [1, 2, 3, 4, 5]
files = [
    'round1_all_responses.json',
    'round2_all_responses.json',
    'round3_all_responses.json',
    'round4_all_responses.json',
    'round5_all_responses.json'
]

print("Council run final verification:")
total_all = 0
for i, f in enumerate(files):
    with open(f'{V2_RESULTS_PATH}/{f}') as fp:
        data = json.load(fp)
    count = data['total']
    total_all += count
    print(f"  Round {i+1}: {count} responses")

print(f"\n  Grand total: {total_all} responses")
print(f"  Budget used: ${budget.total:.4f}")
print(f"  Budget remaining: "
      f"${200 - budget.total:.4f}")

Council run final verification:
  Round 1: 1152 responses
  Round 2: 998 responses
  Round 3: 993 responses
  Round 4: 992 responses
  Round 5: 1002 responses

  Grand total: 5137 responses
  Budget used: $0.0000
  Budget remaining: $200.0000


In [9]:
# Check dataset file structure

with open('/content/drive/MyDrive/HF_IQR/'
          'v2_planning/dataset/'
          'HF_IQR_Master_Dataset_v2.json',
          'r') as f:
    dataset = json.load(f)

print(f"Type: {type(dataset)}")
if isinstance(dataset, dict):
    print(f"Keys: {list(dataset.keys())}")
    for k, v in dataset.items():
        if isinstance(v, list):
            print(f"  {k}: list of "
                  f"{len(v)} items")
        else:
            print(f"  {k}: {type(v).__name__}")
elif isinstance(dataset, list):
    print(f"Length: {len(dataset)}")
    if dataset:
        print(f"First item keys: "
              f"{list(dataset[0].keys())}")
        print(f"Sample ground_truth: "
              f"{dataset[0].get('ground_truth', 'MISSING')}")

Type: <class 'dict'>
Keys: ['project', 'version', 'date_created', 'researcher', 'institution', 'motto', 'lineage', 'v1_reference', 'v2_prereg_hash', 'total_questions', 'total_categories', 'categories', 'question_index', 'dataset', 'dataset_hash']
  project: str
  version: str
  date_created: str
  researcher: str
  institution: str
  motto: str
  lineage: str
  v1_reference: dict
  v2_prereg_hash: str
  total_questions: int
  total_categories: int
  categories: dict
  question_index: list of 200 items
  dataset: dict
  dataset_hash: str


In [10]:
# Check ground truth coverage correctly

dataset_questions = dataset.get('dataset', {})
question_index   = dataset.get('question_index', [])

print(f"Dataset dict keys: "
      f"{len(dataset_questions)} categories")
print(f"Question index: "
      f"{len(question_index)} items")

# Check structure of first question
# in question_index
if question_index:
    sample = question_index[0]
    print(f"\nSample question keys:")
    print(f"  {list(sample.keys())}")
    print(f"\nSample ground_truth field:")
    print(f"  {sample.get('ground_truth', 'MISSING')}")

# Count ground truth coverage
has_gt    = 0
no_gt     = 0
no_gt_ids = []

for q in question_index:
    gt = q.get('ground_truth', '')
    if gt and str(gt).strip():
        has_gt += 1
    else:
        no_gt += 1
        no_gt_ids.append(q.get('id', 'unknown'))

print(f"\nGround truth coverage:")
print(f"  Total:   {len(question_index)}")
print(f"  Has GT:  {has_gt}")
print(f"  No GT:   {no_gt}")

if no_gt_ids[:10]:
    print(f"\nFirst 10 missing:")
    for qid in no_gt_ids[:10]:
        print(f"  {qid}")

Dataset dict keys: 12 categories
Question index: 200 items

Sample question keys:
  ['id', 'category', 'difficulty']

Sample ground_truth field:
  MISSING

Ground truth coverage:
  Total:   200
  Has GT:  0
  No GT:   200

First 10 missing:
  ADV-01
  ADV-02
  ADV-03
  ADV-04
  ADV-05
  ADV-06
  ADV-07
  ADV-08
  ADV-09
  ADV-10


In [11]:
# Check full question structure in dataset dict

categories = dataset.get('dataset', {})

# Get first category and first question
first_cat = list(categories.keys())[0]
first_questions = categories[first_cat]

print(f"First category: {first_cat}")
print(f"Type: {type(first_questions)}")

if isinstance(first_questions, list):
    sample = first_questions[0]
    print(f"\nSample question keys:")
    print(f"  {list(sample.keys())}")
    print(f"\nSample question:")
    for k, v in sample.items():
        if k != 'question':
            print(f"  {k}: {str(v)[:80]}")
        else:
            print(f"  question: {str(v)[:60]}...")

First category: adversarial
Type: <class 'list'>

Sample question keys:
  ['id', 'category', 'subtype', 'difficulty', 'ground_truth', 'trap_type', 'prompt', 'reasoning_request']

Sample question:
  id: ADV-01
  category: adversarial
  subtype: implicit_contradiction
  difficulty: 3
  ground_truth: Contradiction exists. Sarah cannot be youngest if Daniel is 3 years younger than
  trap_type: implicit_contradiction
  prompt: Sarah is the youngest of four siblings.
Her oldest brother Marcus is 28 years ol
  reasoning_request: Before giving your final answer:
1. Label each inference step: Step 1, Step 2, e


In [12]:
# Rebuild ALL_QUESTIONS with full content
# including ground truth

ALL_QUESTIONS = []

categories = dataset.get('dataset', {})
for cat_name, questions in categories.items():
    for q in questions:
        ALL_QUESTIONS.append(q)

# Verify ground truth coverage
has_gt    = 0
no_gt     = 0

for q in ALL_QUESTIONS:
    gt = q.get('ground_truth', '')
    if gt and str(gt).strip():
        has_gt += 1
    else:
        no_gt += 1

print(f"ALL_QUESTIONS rebuilt:")
print(f"  Total:   {len(ALL_QUESTIONS)}")
print(f"  Has GT:  {has_gt}")
print(f"  No GT:   {no_gt}")
print(f"  Coverage: "
      f"{has_gt/len(ALL_QUESTIONS)*100:.1f}%")

# Sample check
sample = ALL_QUESTIONS[0]
print(f"\nSample ADV-01 ground truth:")
print(f"  {sample.get('ground_truth', 'MISSING')}")

ALL_QUESTIONS rebuilt:
  Total:   200
  Has GT:  200
  No GT:   0
  Coverage: 100.0%

Sample ADV-01 ground truth:
  Contradiction exists. Sarah cannot be youngest if Daniel is 3 years younger than her. Correct answer: identify contradiction before solving math.


In [14]:
# Verify Hudson Forge jury models

import requests

jury_models = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

print("Testing jury model connections:")
for model in jury_models:
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning': 'true',
                'Content-Type': 'application/json'},
            json={
                'model':       model,
                'prompt':      'Return the word: ready',
                'num_predict': 5,
                'stream':      False},
            timeout=30)
        if r.status_code == 200:
            print(f"  {model:<25} ✅")
        else:
            print(f"  {model:<25} ❌ "
                  f"status {r.status_code}")
    except Exception as e:
        print(f"  {model:<25} ❌ {str(e)[:40]}")

Testing jury model connections:
  mistral-nemo:latest       ✅
  deepseek-r1:14b           ✅
  gemma3:4b                 ✅


In [38]:
# Debug — see raw output from each model

test_q = ALL_QUESTIONS[0]
test_r = r1_lookup.get(
    'ADV-01', {}).get('claude', '')

print("Raw outputs from jury models:\n")
for jury in ['mistral-nemo:latest',
             'deepseek-r1:14b',
             'gemma3:4b']:
    r = requests.post(
        f'{NGROK_URL}/api/generate',
        headers={
            'ngrok-skip-browser-warning':
                'true',
            'Content-Type':
                'application/json'},
        json={
            'model':       jury,
            'prompt':      build_esvr_prompt(
                test_q, test_r),
            'num_predict': 400,
            'stream':      False},
        timeout=120)

    if r.status_code == 200:
        text = r.json().get('response', '')
        print(f"=== {jury} ===")
        print(text[:500])
        print()
    else:
        print(f"=== {jury} === "
              f"status {r.status_code}")

Raw outputs from jury models:

=== mistral-nemo:latest ===
{
  "steps_explicit": 1.0,
  "logic_valid": 0.7,
  "conclusion_follows": 0.7,
  "no_circular": 1.0,
  "correct": 0.0,
  "esvr_score": 0.62
}

=== deepseek-r1:14b ===
```json
{
  "steps_explicit": 1.0,
  "logic_valid": 0.9,
  "conclusion_follows": 1.0,
  "no_circular": 1.0,
  "correct": 1.0,
  "esvr_score": (1.0*0.15 + 0.9*0.30 + 1.0*0.25 + 1.0*0.15 + 1.0*0.15) = 0.8
}
```

=== gemma3:4b ===
```json
{
  "steps_explicit": 0.6,
  "logic_valid": 0.7,
  "conclusion_follows": 0.8,
  "no_circular": 0.9,
  "correct": 0.0,
  "esvr_score": 0.76
}
```



In [39]:
def parse_jury_response(text, jury_model):
    # Strip markdown fences
    text = text.replace(
        '```json', '').replace(
        '```', '').strip()

    # Find outermost JSON object
    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1:
        return None

    json_str = text[start:end+1]

    # DeepSeek-R1 puts formula in
    # esvr_score field — replace with
    # computed value
    import re
    json_str = re.sub(
        r'"esvr_score"\s*:\s*[^,}\n]+',
        '"esvr_score": 0.0',
        json_str)

    scores = json.loads(json_str)

    # Compute esvr_score from components
    scores['esvr_score'] = round(
        scores.get('steps_explicit', 0)
        * 0.15
        + scores.get('logic_valid', 0)
        * 0.30
        + scores.get('conclusion_follows', 0)
        * 0.25
        + scores.get('no_circular', 0)
        * 0.15
        + scores.get('correct', 0)
        * 0.15, 4)

    return scores


def score_esvr_jury(question, response,
                    jury_model):
    prompt = build_esvr_prompt(
        question, response)
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning':
                    'true',
                'Content-Type':
                    'application/json'},
            json={
                'model':       jury_model,
                'prompt':      prompt,
                'num_predict': 400,
                'stream':      False},
            timeout=120)

        if r.status_code == 200:
            text = r.json().get(
                'response', '')
            scores = parse_jury_response(
                text, jury_model)
            if scores is None:
                return {
                    'jury_model': jury_model,
                    'esvr_score': None,
                    'error': 'Parse failed'}
            scores['jury_model'] = jury_model
            scores['error']      = None
            return scores
        else:
            return {
                'jury_model': jury_model,
                'esvr_score': None,
                'error': f"Status "
                         f"{r.status_code}"}
    except Exception as e:
        return {
            'jury_model': jury_model,
            'esvr_score': None,
            'error':      str(e)[:80]}

print("ESVR scorer updated ✅")

# Retest all three
test_q = ALL_QUESTIONS[0]
test_r = r1_lookup.get(
    'ADV-01', {}).get('claude', '')

print(f"\nRetesting ADV-01 claude...")
for jury in ['mistral-nemo:latest',
             'deepseek-r1:14b',
             'gemma3:4b']:
    result = score_esvr_jury(
        test_q, test_r, jury)
    if not result.get('error'):
        print(f"  {jury:<25} "
              f"ESVR: "
              f"{result.get('esvr_score'):.3f}")
    else:
        print(f"  {jury:<25} "
              f"❌ {result['error'][:50]}")

ESVR scorer updated ✅

Retesting ADV-01 claude...
  mistral-nemo:latest       ESVR: 0.610
  deepseek-r1:14b           ESVR: 1.000
  gemma3:4b                 ESVR: 0.635


In [40]:
# Manual scoring reference
# Pick 5 questions across different
# categories and score them yourself

manual_sample = [
    'ADV-01',   # adversarial
    'MPQ-01',   # mathematical proof
    'QRQ-01',   # quantum reasoning
    'EDQ-01',   # ethical dilemma
    'CFQ-01'    # counterfactual
]

for qid in manual_sample:
    q = next(q for q in ALL_QUESTIONS
             if q['id'] == qid)
    r1 = r1_lookup.get(qid, {})

    print(f"\n{'='*50}")
    print(f"Question: {qid}")
    print(f"Category: {q['category']}")
    print(f"\nPrompt:")
    print(f"{q['prompt'][:200]}...")
    print(f"\nGround truth:")
    print(f"{q['ground_truth'][:200]}")
    print(f"\nClaude response:")
    print(f"{r1.get('claude', '')[:300]}...")


Question: ADV-01
Category: adversarial

Prompt:
Sarah is the youngest of four siblings.
Her oldest brother Marcus is 28 years old.
Her brother Daniel is 3 years younger than Sarah.
The average age of all four siblings is 22.

How old is Sarah?...

Ground truth:
Contradiction exists. Sarah cannot be youngest if Daniel is 3 years younger than her. Correct answer: identify contradiction before solving math.

Claude response:
REASONING CHAIN:
Step 1: Identify the known information: Marcus is 28, Daniel is (Sarah - 3) years old, and the average age of all four siblings is 22.
Step 2: Set up the average equation: (Marcus + Daniel + Sarah + Fourth_sibling) / 4 = 22
Step 3: Calculate total age of all siblings: 22 × 4 = 88 ye...

Question: MPQ-01
Category: mathematical_proof

Prompt:
Prove or disprove: The sum of any two odd numbers is always even. Show your proof in numbered steps. Use direct proof or proof by contradiction — state which method you are using....

Ground truth:
Direct proof. A

In [41]:
# Validate jury scoring against
# your manual scores

manual_sample_ids = [
    'ADV-01', 'MPQ-01', 'QRQ-01',
    'EDQ-01', 'CFQ-01'
]

your_scores = {
    'ADV-01': 0.85,
    'MPQ-01': 1.00,
    'QRQ-01': 0.90,
    'EDQ-01': 0.85,
    'CFQ-01': 0.95
}

jury_models = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

print("Validation — Claude responses scored")
print("by jury models vs your manual scores\n")

for qid in manual_sample_ids:
    q = next(q for q in ALL_QUESTIONS
             if q['id'] == qid)
    r1 = r1_lookup.get(qid, {}).get('claude')

    if not r1:
        print(f"{qid}: no R1 response")
        continue

    print(f"{qid} — {q['category']}")
    print(f"  Your score: {your_scores[qid]}")

    jury_scores = []
    for jury in jury_models:
        result = score_esvr_jury(
            q, r1, jury)
        if not result.get('error'):
            score = result['esvr_score']
            jury_scores.append(score)
            print(f"  {jury:<25} {score:.3f}")
        else:
            print(f"  {jury:<25} "
                  f"❌ {result['error'][:40]}")

    if jury_scores:
        avg = sum(jury_scores) / len(jury_scores)
        diff = abs(avg - your_scores[qid])
        print(f"  Jury average:             {avg:.3f}")
        print(f"  Difference from yours:    {diff:.3f}")
    print()

Validation — Claude responses scored
by jury models vs your manual scores

ADV-01 — adversarial
  Your score: 0.85
  mistral-nemo:latest       ❌ Expecting property name enclosed in doub
  deepseek-r1:14b           0.940
  gemma3:4b                 0.635
  Jury average:             0.787
  Difference from yours:    0.062

MPQ-01 — mathematical_proof
  Your score: 1.0
  mistral-nemo:latest       1.000
  deepseek-r1:14b           1.000
  gemma3:4b                 1.000
  Jury average:             1.000
  Difference from yours:    0.000

QRQ-01 — quantum_reasoning
  Your score: 0.9
  mistral-nemo:latest       1.000
  deepseek-r1:14b           1.000
  gemma3:4b                 1.000
  Jury average:             1.000
  Difference from yours:    0.100

EDQ-01 — ethical_dilemma
  Your score: 0.85
  mistral-nemo:latest       1.000
  deepseek-r1:14b           1.000
  gemma3:4b                 0.870
  Jury average:             0.957
  Difference from yours:    0.107

CFQ-01 — counterfactual
  You

In [42]:
# Debug Mistral-Nemo raw output on ADV-01

test_q = next(q for q in ALL_QUESTIONS
              if q['id'] == 'ADV-01')
test_r = r1_lookup.get(
    'ADV-01', {}).get('claude', '')

r = requests.post(
    f'{NGROK_URL}/api/generate',
    headers={
        'ngrok-skip-browser-warning': 'true',
        'Content-Type': 'application/json'},
    json={
        'model':       'mistral-nemo:latest',
        'prompt':      build_esvr_prompt(
            test_q, test_r),
        'num_predict': 400,
        'stream':      False},
    timeout=120)

text = r.json().get('response', '')
print("Mistral-Nemo raw output:")
print(repr(text[:600]))

Mistral-Nemo raw output:
'{\n  "steps_explicit": 1.0,\n  "logic_valid": 0.8,\n  "conclusion_follows": 0.9,\n  "no_circular": 0.0,\n  "correct": 0.0,\n  "esvr_score": 0.62\n}'


In [43]:
def parse_jury_response(text, jury_model):
    # Strip markdown fences
    text = text.replace(
        '```json', '').replace(
        '```', '').strip()

    # Find outermost JSON object
    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1:
        return None

    json_str = text[start:end+1]

    # Remove esvr_score field entirely
    # before parsing — we compute it
    import re
    json_str = re.sub(
        r',?\s*"esvr_score"\s*:[^,}\n]+',
        '', json_str)

    # Clean trailing commas before }
    json_str = re.sub(
        r',\s*}', '}', json_str)

    try:
        scores = json.loads(json_str)
    except Exception as e:
        return None

    # Compute esvr_score from components
    scores['esvr_score'] = round(
        scores.get('steps_explicit', 0)
        * 0.15
        + scores.get('logic_valid', 0)
        * 0.30
        + scores.get('conclusion_follows', 0)
        * 0.25
        + scores.get('no_circular', 0)
        * 0.15
        + scores.get('correct', 0)
        * 0.15, 4)

    return scores

print("Parser updated ✅")

# Retest all three models
test_q = next(q for q in ALL_QUESTIONS
              if q['id'] == 'ADV-01')
test_r = r1_lookup.get(
    'ADV-01', {}).get('claude', '')

print("\nRetesting all three jury models:")
for jury in ['mistral-nemo:latest',
             'deepseek-r1:14b',
             'gemma3:4b']:
    result = score_esvr_jury(
        test_q, test_r, jury)
    if not result.get('error'):
        print(f"  {jury:<25} "
              f"ESVR: {result['esvr_score']:.3f}")
    else:
        print(f"  {jury:<25} "
              f"❌ {result['error'][:50]}")

Parser updated ✅

Retesting all three jury models:
  mistral-nemo:latest       ESVR: 0.743
  deepseek-r1:14b           ESVR: 0.940
  gemma3:4b                 ESVR: 0.620


In [16]:
import re
import json
import requests

def parse_jury_response(text, jury_model):
    text = text.replace(
        '```json', '').replace(
        '```', '').strip()

    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1:
        return None

    json_str = text[start:end+1]

    json_str = re.sub(
        r',?\s*"esvr_score"\s*:[^,}\n]+',
        '', json_str)
    json_str = re.sub(
        r',\s*}', '}', json_str)

    try:
        scores = json.loads(json_str)
    except Exception:
        return None

    scores['esvr_score'] = round(
        scores.get('steps_explicit', 0) * 0.15
        + scores.get('logic_valid', 0) * 0.30
        + scores.get('conclusion_follows', 0) * 0.25
        + scores.get('no_circular', 0) * 0.15
        + scores.get('correct', 0) * 0.15, 4)

    return scores


def build_esvr_prompt(question, response):
    return f"""You are evaluating the quality
of a reasoning response. Score each criterion
honestly on a 0-1 scale.

QUESTION:
{question['prompt']}

GROUND TRUTH:
{question['ground_truth']}

RESPONSE TO EVALUATE:
{response}

Score the following criteria:
1. STEPS_EXPLICIT (0-1): Are reasoning
   steps clearly numbered and separated?
2. LOGIC_VALID (0-1): Is each step
   logically valid and follows from prior?
3. CONCLUSION_FOLLOWS (0-1): Does the
   final answer follow from the steps?
4. NO_CIRCULAR (0-1): Is circular
   reasoning absent?
5. CORRECT (0-1): Does the answer
   match the ground truth?

Respond in this exact JSON format only:
{{
  "steps_explicit": 0.0,
  "logic_valid": 0.0,
  "conclusion_follows": 0.0,
  "no_circular": 0.0,
  "correct": 0.0
}}

Return JSON only. No other text."""


def score_esvr_jury(question, response,
                    jury_model):
    prompt = build_esvr_prompt(
        question, response)
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning':
                    'true',
                'Content-Type':
                    'application/json'},
            json={
                'model':       jury_model,
                'prompt':      prompt,
                'num_predict': 400,
                'stream':      False},
            timeout=120)

        if r.status_code == 200:
            text = r.json().get(
                'response', '')
            scores = parse_jury_response(
                text, jury_model)
            if scores is None:
                return {
                    'jury_model': jury_model,
                    'esvr_score': None,
                    'error': 'Parse failed'}
            scores['jury_model'] = jury_model
            scores['error']      = None
            return scores
        else:
            return {
                'jury_model': jury_model,
                'esvr_score': None,
                'error': f"Status "
                         f"{r.status_code}"}
    except Exception as e:
        return {
            'jury_model': jury_model,
            'esvr_score': None,
            'error':      str(e)[:80]}

print("Scoring functions defined ✅")

# Quick test
test_q = next(q for q in ALL_QUESTIONS
              if q['id'] == 'ADV-01')
test_r = r1_lookup.get(
    'ADV-01', {}).get('claude', '')

for jury in ['mistral-nemo:latest',
             'deepseek-r1:14b',
             'gemma3:4b']:
    result = score_esvr_jury(
        test_q, test_r, jury)
    if not result.get('error'):
        print(f"  {jury:<25} "
              f"ESVR: {result['esvr_score']:.3f}")
    else:
        print(f"  {jury:<25} "
              f"❌ {result['error'][:40]}")

Scoring functions defined ✅
  mistral-nemo:latest       ESVR: 0.840
  deepseek-r1:14b           ESVR: 0.885
  gemma3:4b                 ESVR: 0.580


In [17]:
# ESVR Validation Sample Runner
# 40 questions proportionally sampled
# across 12 categories

import random
import time

random.seed(42)

# Build proportional sample
sample_questions = []
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions,
        min(n, len(questions)))
    sample_questions.extend(sampled)

print(f"Sample size: {len(sample_questions)}")
print(f"Categories covered: "
      f"{len(cat_groups)}")

JURY_MODELS = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

esvr_results = []
total_calls  = len(sample_questions) * \
               len(SUBJECT_MODELS) * \
               len(JURY_MODELS)

print(f"Total scoring calls: {total_calls}")
print(f"Estimated time: "
      f"~{total_calls * 15 // 60} minutes")
print(f"Starting...\n")

for q in sample_questions:
    qid = q['id']
    for model in SUBJECT_MODELS:
        r1 = r1_lookup.get(qid, {}).get(model)
        if not r1:
            continue

        model_scores = []
        for jury in JURY_MODELS:
            result = score_esvr_jury(
                q, r1, jury)
            if not result.get('error'):
                model_scores.append(
                    result['esvr_score'])
                esvr_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'model':       model,
                    'jury_model':  jury,
                    'esvr_score':  result[
                        'esvr_score'],
                    'steps_explicit': result.get(
                        'steps_explicit'),
                    'logic_valid': result.get(
                        'logic_valid'),
                    'conclusion_follows': result.get(
                        'conclusion_follows'),
                    'no_circular': result.get(
                        'no_circular'),
                    'correct':     result.get(
                        'correct'),
                    'error':       None
                })
            else:
                esvr_results.append({
                    'question_id': qid,
                    'model':       model,
                    'jury_model':  jury,
                    'esvr_score':  None,
                    'error':       result['error']
                })
            time.sleep(1)

        if model_scores:
            avg = sum(model_scores) / \
                  len(model_scores)
            print(f"  {qid:<10} {model:<12} "
                  f"avg: {avg:.3f}")

# Save validation results
with open(f'{V2_RESULTS_PATH}/'
          f'esvr_validation.json', 'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'sample_size': len(sample_questions),
        'total_scores': len(esvr_results),
        'results': esvr_results
    }, f, indent=2)

print(f"\nValidation complete:")
print(f"  Scores recorded: {len(esvr_results)}")
print(f"  Saved ✅")

Sample size: 40
Categories covered: 12
Total scoring calls: 600
Estimated time: ~150 minutes
Starting...

  ADV-04     claude       avg: 1.000
  ADV-04     gpt4o        avg: 0.873
  ADV-04     gemini       avg: 0.611
  ADV-04     deepseek     avg: 0.967
  ADV-04     grok         avg: 0.967
  ADV-01     claude       avg: 0.685
  ADV-01     gpt4o        avg: 0.700
  ADV-01     gemini       avg: 1.000
  ADV-01     deepseek     avg: 0.750
  ADV-01     grok         avg: 0.775
  ADV-09     claude       avg: 0.917
  ADV-09     gpt4o        avg: 1.000
  ADV-09     gemini       avg: 0.833
  ADV-09     deepseek     avg: 1.000
  ADV-09     grok         avg: 0.917
  ADV-08     claude       avg: 0.917
  ADV-08     gpt4o        avg: 0.713
  ADV-08     gemini       avg: 0.758
  ADV-08     deepseek     avg: 0.867
  ADV-08     grok         avg: 0.850
  LSQ-08     claude       avg: 0.963
  LSQ-08     gpt4o        avg: 0.954
  LSQ-08     gemini       avg: 0.856
  LSQ-08     deepseek     avg: 0.892
  LSQ-

## ESVR Validation — Why This Approach
Was Abandoned

### What Was Attempted

ESVR (Explicit Step Validity Rate) was
pre-registered as an LLM-as-judge metric
scoring reasoning step quality across
five criteria: steps explicit, logic valid,
conclusion follows, no circular reasoning,
and correctness against ground truth.

Three local jury models were used:
mistral-nemo:latest, deepseek-r1:14b,
and gemma3:4b via Hudson Forge IRMB-C.

### Validation Results

600 scoring calls across 40 questions
and 5 subject models produced the
following inter-rater correlations:

  mistral vs deepseek: r=0.238
  mistral vs gemma3:   r=0.470
  deepseek vs gemma3:  r=0.289

Score distribution was heavily compressed
between 0.85 and 1.00 across all models
and categories indicating the jury models
could not discriminate between strong
and weak frontier model reasoning.

### Why This Failed

Local models at the 7B-14B parameter
range lack sufficient domain knowledge
to reliably evaluate frontier model
responses on complex reasoning tasks
including quantum mechanics, advanced
mathematics, and formal logic.

The jury models scored Claude's ADV-01
response highly despite Claude missing
the embedded contradiction entirely —
a fundamental reasoning failure the
local models could not detect.

### Decision

ESVR jury scoring was abandoned after
validation. This is reported as a
methodological finding not suppressed.

Primary metrics proceed as DSS V2
and CVS V2 which involve more
discrete checkable criteria better
suited to local model evaluation.

In [18]:
# Validation analysis
# Cohen's Kappa and preliminary stats

import json
import numpy as np
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'esvr_validation.json', 'r') as f:
    validation = json.load(f)

results = validation['results']

# Filter valid scores
valid = [r for r in results
         if r.get('esvr_score') is not None]

print(f"Valid scores: {len(valid)} of "
      f"{len(results)}")

# Compute average ESVR by model
model_scores = defaultdict(list)
for r in valid:
    model_scores[r['model']].append(
        r['esvr_score'])

print(f"\nAverage ESVR by model:")
for model, scores in sorted(
        model_scores.items()):
    avg = sum(scores) / len(scores)
    print(f"  {model:<12} "
          f"{avg:.3f} "
          f"(n={len(scores)})")

# Compute inter-rater agreement
# Group by question+model pair
pairs = defaultdict(dict)
for r in valid:
    key = f"{r['question_id']}_{r['model']}"
    pairs[key][r['jury_model']] = \
        r['esvr_score']

# Compute pairwise correlation
jury_models = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

print(f"\nInter-rater correlations:")
for i in range(len(jury_models)):
    for j in range(i+1, len(jury_models)):
        j1 = jury_models[i]
        j2 = jury_models[j]
        shared = [
            (v[j1], v[j2])
            for v in pairs.values()
            if j1 in v and j2 in v]
        if shared:
            s1 = [x[0] for x in shared]
            s2 = [x[1] for x in shared]
            n  = len(shared)
            mean1 = sum(s1) / n
            mean2 = sum(s2) / n
            num = sum((a - mean1) *
                      (b - mean2)
                      for a, b in shared)
            den = (sum((a - mean1)**2
                       for a in s1) *
                   sum((b - mean2)**2
                       for b in s2)) ** 0.5
            corr = num / den if den else 0
            print(f"  {j1[:15]:<16} vs "
                  f"{j2[:15]:<16} "
                  f"r={corr:.3f} "
                  f"(n={n})")

# Category breakdown
print(f"\nAverage ESVR by category:")
cat_scores = defaultdict(list)
for r in valid:
    cat_scores[r.get('category',
                     'unknown')].append(
        r['esvr_score'])
for cat, scores in sorted(
        cat_scores.items()):
    avg = sum(scores) / len(scores)
    print(f"  {cat:<25} {avg:.3f}")

Valid scores: 583 of 600

Average ESVR by model:
  claude       0.946 (n=117)
  deepseek     0.944 (n=116)
  gemini       0.940 (n=115)
  gpt4o        0.914 (n=117)
  grok         0.945 (n=118)

Inter-rater correlations:
  mistral-nemo:la  vs deepseek-r1:14b  r=0.238 (n=184)
  mistral-nemo:la  vs gemma3:4b        r=0.470 (n=185)
  deepseek-r1:14b  vs gemma3:4b        r=0.289 (n=198)

Average ESVR by category:
  adversarial               0.864
  causal_chain              0.944
  counterfactual            0.920
  ethical_dilemma           0.917
  frontier_reasoning        0.951
  logical_syllogism         0.942
  mathematical_proof        0.985
  meta_reasoning            0.924
  probabilistic             0.972
  quantum_reasoning         0.976
  spatial_reasoning         0.882
  temporal_reasoning        0.958


In [19]:
# DSS V2 Scorer
# Restricted to binary ground truth
# categories only

DSS_VERIFIABLE_CATEGORIES = [
    'mathematical_proof',
    'quantum_reasoning',
    'logical_syllogism',
    'probabilistic',
    'temporal_reasoning',
    'spatial_reasoning'
]

DSS_METHOD_CATEGORIES = [
    'ethical_dilemma',
    'counterfactual',
    'frontier_reasoning',
    'meta_reasoning',
    'adversarial',
    'causal_chain'
]

def build_correctness_prompt(
        question, response):
    return f"""You are evaluating whether
a response correctly answers a question.

QUESTION:
{question['prompt']}

GROUND TRUTH:
{question['ground_truth']}

RESPONSE:
{response}

Is the final answer correct?
Respond in JSON only:
{{
  "correct": 0 or 1,
  "reasoning": "one sentence explanation"
}}

1 = correct or substantially correct
0 = incorrect or missed key point

JSON only. No other text."""


def score_correctness(question,
                      response,
                      jury_model):
    prompt = build_correctness_prompt(
        question, response)
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning':
                    'true',
                'Content-Type':
                    'application/json'},
            json={
                'model':       jury_model,
                'prompt':      prompt,
                'num_predict': 150,
                'stream':      False},
            timeout=120)

        if r.status_code == 200:
            text = r.json().get(
                'response', '')
            text = text.replace(
                '```json', '').replace(
                '```', '').strip()
            start = text.find('{')
            end   = text.rfind('}')
            if start == -1 or end == -1:
                return None
            result = json.loads(
                text[start:end+1])
            return {
                'correct':   int(
                    result.get('correct', 0)),
                'reasoning': result.get(
                    'reasoning', ''),
                'jury_model': jury_model,
                'error':      None
            }
        else:
            return {
                'correct':    None,
                'jury_model': jury_model,
                'error': f"Status "
                         f"{r.status_code}"}
    except Exception as e:
        return {
            'correct':    None,
            'jury_model': jury_model,
            'error':      str(e)[:80]}

print("DSS correctness scorer defined ✅")

# Test on verifiable categories
test_cases = [
    ('MPQ-01', 'claude'),
    ('QRQ-01', 'claude'),
    ('LSQ-01', 'claude')
]

print("\nTesting correctness scorer:")
for qid, model in test_cases:
    q  = next(q for q in ALL_QUESTIONS
              if q['id'] == qid)
    r1 = r1_lookup.get(qid, {}).get(model)
    if not r1:
        continue

    results = []
    for jury in ['mistral-nemo:latest',
                 'deepseek-r1:14b',
                 'gemma3:4b']:
        result = score_correctness(
            q, r1, jury)
        if result and not result.get('error'):
            results.append(result['correct'])
            print(f"  {qid} {model} "
                  f"{jury[:15]:<16} "
                  f"correct={result['correct']}")

    if results:
        majority = 1 if sum(results) >= 2 \
                   else 0
        print(f"  {qid} majority vote: "
              f"{majority}\n")

DSS correctness scorer defined ✅

Testing correctness scorer:
  MPQ-01 claude mistral-nemo:la  correct=1
  MPQ-01 claude deepseek-r1:14b  correct=1
  MPQ-01 claude gemma3:4b        correct=1
  MPQ-01 majority vote: 1

  QRQ-01 claude mistral-nemo:la  correct=1
  QRQ-01 claude deepseek-r1:14b  correct=1
  QRQ-01 claude gemma3:4b        correct=1
  QRQ-01 majority vote: 1

  LSQ-01 claude mistral-nemo:la  correct=1
  LSQ-01 claude deepseek-r1:14b  correct=1
  LSQ-01 claude gemma3:4b        correct=1
  LSQ-01 majority vote: 1



In [20]:
# Test correctness scorer on a known
# failure case — ADV-01 claude missed
# the contradiction

test_q = next(q for q in ALL_QUESTIONS
              if q['id'] == 'ADV-01')
test_r = r1_lookup.get(
    'ADV-01', {}).get('claude', '')

print("Testing failure detection — ADV-01:")
print(f"Ground truth: {test_q['ground_truth'][:80]}")
print()

results = []
for jury in ['mistral-nemo:latest',
             'deepseek-r1:14b',
             'gemma3:4b']:
    result = score_correctness(
        test_q, test_r, jury)
    if result and not result.get('error'):
        results.append(result['correct'])
        print(f"  {jury[:20]:<22} "
              f"correct={result['correct']} "
              f"— {result.get('reasoning', '')[:60]}")

if results:
    majority = 1 if sum(results) >= 2 else 0
    print(f"\n  Majority vote: {majority}")
    print(f"  Expected:      0")
    print(f"  Match: "
          f"{'✅' if majority == 0 else '❌'}")

Testing failure detection — ADV-01:
Ground truth: Contradiction exists. Sarah cannot be youngest if Daniel is 3 years younger than

  mistral-nemo:latest    correct=0 — The question states Sarah is the youngest but also Daniel is
  deepseek-r1:14b        correct=0 — The problem contains a logical contradiction where Daniel ca
  gemma3:4b              correct=0 — The problem contains a contradiction: Sarah is the youngest,

  Majority vote: 0
  Expected:      0
  Match: ✅


In [21]:
# Full DSS correctness scoring
# Verifiable categories only

import time

DSS_VERIFIABLE_CATEGORIES = [
    'mathematical_proof',
    'quantum_reasoning',
    'logical_syllogism',
    'probabilistic',
    'temporal_reasoning',
    'spatial_reasoning'
]

SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

JURY_MODELS = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

# Get verifiable questions
verifiable_questions = [
    q for q in ALL_QUESTIONS
    if q['category'] in
    DSS_VERIFIABLE_CATEGORIES]

print(f"Verifiable questions: "
      f"{len(verifiable_questions)}")
print(f"Models: {len(SUBJECT_MODELS)}")
print(f"Jury: {len(JURY_MODELS)}")
total = (len(verifiable_questions) *
         len(SUBJECT_MODELS) *
         len(JURY_MODELS))
print(f"Total calls: {total}")
print(f"Estimated time: "
      f"~{total * 8 // 60} minutes\n")

correctness_results = []

for q in verifiable_questions:
    qid = q['id']
    for model in SUBJECT_MODELS:
        r1 = r1_lookup.get(
            qid, {}).get(model)
        if not r1:
            continue

        jury_votes = []
        for jury in JURY_MODELS:
            result = score_correctness(
                q, r1, jury)
            if result and \
               not result.get('error'):
                jury_votes.append(
                    result['correct'])
                correctness_results.append({
                    'question_id': qid,
                    'category':   q['category'],
                    'difficulty': q['difficulty'],
                    'model':      model,
                    'jury_model': jury,
                    'correct':    result['correct'],
                    'reasoning':  result.get(
                        'reasoning', ''),
                    'error':      None
                })
            else:
                correctness_results.append({
                    'question_id': qid,
                    'category':   q['category'],
                    'model':      model,
                    'jury_model': jury,
                    'correct':    None,
                    'error':      result.get(
                        'error') if result
                        else 'No response'
                })
            time.sleep(1)

        if jury_votes:
            majority = 1 if \
                sum(jury_votes) >= 2 \
                else 0
            print(f"  {qid:<10} {model:<12} "
                  f"correct={majority} "
                  f"({sum(jury_votes)}/"
                  f"{len(jury_votes)})")

# Save results
with open(f'{V2_RESULTS_PATH}/'
          f'dss_correctness.json',
          'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'categories': DSS_VERIFIABLE_CATEGORIES,
        'total':      len(correctness_results),
        'results':    correctness_results
    }, f, indent=2)

print(f"\nDSS correctness scoring complete:")
print(f"  Results: {len(correctness_results)}")
print(f"  Saved ✅")

Verifiable questions: 95
Models: 5
Jury: 3
Total calls: 1425
Estimated time: ~190 minutes

  LSQ-01     claude       correct=1 (3/3)
  LSQ-01     gpt4o        correct=1 (3/3)
  LSQ-01     gemini       correct=1 (3/3)
  LSQ-01     deepseek     correct=1 (3/3)
  LSQ-01     grok         correct=1 (3/3)
  LSQ-02     claude       correct=1 (3/3)
  LSQ-02     gpt4o        correct=1 (3/3)
  LSQ-02     gemini       correct=1 (3/3)
  LSQ-02     deepseek     correct=1 (3/3)
  LSQ-02     grok         correct=1 (3/3)
  LSQ-03     claude       correct=1 (3/3)
  LSQ-03     gpt4o        correct=1 (3/3)
  LSQ-03     gemini       correct=1 (3/3)
  LSQ-03     deepseek     correct=1 (3/3)
  LSQ-03     grok         correct=1 (3/3)
  LSQ-04     claude       correct=1 (3/3)
  LSQ-04     gpt4o        correct=1 (3/3)
  LSQ-04     gemini       correct=1 (3/3)
  LSQ-04     deepseek     correct=1 (3/3)
  LSQ-04     grok         correct=1 (3/3)
  LSQ-05     claude       correct=1 (3/3)
  LSQ-05     gpt4o        c

In [22]:
# DSS Correctness Analysis

import json
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'dss_correctness.json', 'r') as f:
    dss_data = json.load(f)

results = dss_data['results']
valid = [r for r in results
         if r.get('correct') is not None]

# Compute majority vote per question/model
pairs = defaultdict(list)
for r in valid:
    key = f"{r['question_id']}_{r['model']}"
    pairs[key].append(r['correct'])

majority_votes = {}
for key, votes in pairs.items():
    majority_votes[key] = 1 if \
        sum(votes) >= 2 else 0

# Correctness rate by model
model_correct = defaultdict(list)
for key, vote in majority_votes.items():
    model = key.split('_')[1]
    model_correct[model].append(vote)

print("Correctness rate by model:")
for model, votes in sorted(
        model_correct.items()):
    rate = sum(votes) / len(votes)
    print(f"  {model:<12} "
          f"{rate:.3f} "
          f"({sum(votes)}/{len(votes)})")

# Correctness rate by category
cat_correct = defaultdict(list)
for r in valid:
    key = f"{r['question_id']}_{r['model']}"
    if key in majority_votes:
        cat_correct[r['category']].append(
            majority_votes[key])

print(f"\nCorrectness rate by category:")
for cat, votes in sorted(
        cat_correct.items()):
    rate = sum(votes) / len(votes)
    n    = len(votes)
    print(f"  {cat:<25} "
          f"{rate:.3f} (n={n})")

print(f"\nOverall correctness rate:")
all_votes = list(majority_votes.values())
overall = sum(all_votes) / len(all_votes)
print(f"  {overall:.3f} "
      f"({sum(all_votes)}/{len(all_votes)})")

Correctness rate by model:
  claude       0.968 (92/95)
  deepseek     0.958 (91/95)
  gemini       0.937 (89/95)
  gpt4o        0.884 (84/95)
  grok         0.935 (87/93)

Correctness rate by category:
  logical_syllogism         0.990 (n=300)
  mathematical_proof        1.000 (n=298)
  probabilistic             1.000 (n=297)
  quantum_reasoning         0.960 (n=225)
  spatial_reasoning         0.791 (n=148)
  temporal_reasoning        0.699 (n=146)

Overall correctness rate:
  0.937 (443/473)


In [23]:
# Detailed breakdown for low-correctness
# categories

for target_cat in ['temporal_reasoning',
                   'spatial_reasoning']:
    print(f"\n{target_cat}:")
    cat_model = defaultdict(list)
    for r in valid:
        if r['category'] == target_cat:
            key = (f"{r['question_id']}_"
                   f"{r['model']}")
            if key in majority_votes:
                cat_model[r['model']].append(
                    majority_votes[key])

    for model, votes in sorted(
            cat_model.items()):
        rate = sum(votes) / len(votes)
        print(f"  {model:<12} "
              f"{rate:.3f} "
              f"({sum(votes)}/{len(votes)})")


temporal_reasoning:
  claude       0.800 (24/30)
  deepseek     0.900 (27/30)
  gemini       0.724 (21/29)
  gpt4o        0.500 (15/30)
  grok         0.556 (15/27)

spatial_reasoning:
  claude       0.900 (27/30)
  deepseek     0.700 (21/30)
  gemini       0.700 (21/30)
  gpt4o        0.857 (24/28)
  grok         0.800 (24/30)


In [24]:
# Compute DSS V2 scores
# PS + RQR + CR components

from collections import defaultdict
import json

# Load Round 3 positions
with open(f'{V2_RESULTS_PATH}/'
          f'round3_all_responses.json',
          'r') as f:
    r3_data = json.load(f)

r3_positions = {}
for r in r3_data['responses']:
    qid   = r.get('question_id')
    model = r.get('model')
    pos   = r.get('position', 'UNKNOWN')
    if qid and model:
        if qid not in r3_positions:
            r3_positions[qid] = {}
        r3_positions[qid][model] = pos

# Build correctness lookup
# majority vote per question/model
correct_lookup = {}
for key, vote in majority_votes.items():
    correct_lookup[key] = vote

# Compute DSS for verifiable categories
dss_scores = []

for q in ALL_QUESTIONS:
    qid = q['id']
    if q['category'] not in \
       DSS_VERIFIABLE_CATEGORIES:
        continue

    for model in SUBJECT_MODELS:
        position = r3_positions.get(
            qid, {}).get(model, 'UNKNOWN')
        if position == 'UNKNOWN':
            continue

        correct_key = f"{qid}_{model}"
        cr = correct_lookup.get(
            correct_key, None)
        if cr is None:
            continue

        # PS — Position Stability
        # 1.0 if DEFENDING 0.5 if REVISING
        ps = 1.0 if position == 'DEFENDING' \
             else 0.5

        # RQR — use correctness as proxy
        # since we do not have step scores
        # Honest limitation — note in paper
        rqr = float(cr)

        # CR — correctness rating
        cr_score = float(cr)

        # DSS formula from pre-registration
        dss = (ps * 0.30) + \
              (rqr * 0.40) + \
              (cr_score * 0.30)

        dss_scores.append({
            'question_id': qid,
            'category':    q['category'],
            'difficulty':  q['difficulty'],
            'model':       model,
            'position':    position,
            'ps':          ps,
            'rqr':         rqr,
            'cr':          cr_score,
            'dss':         round(dss, 4)
        })

# Save DSS scores
with open(f'{V2_RESULTS_PATH}/'
          f'dss_scores.json', 'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(dss_scores),
        'scores':    dss_scores
    }, f, indent=2)

# Summary by model
model_dss = defaultdict(list)
for s in dss_scores:
    model_dss[s['model']].append(s['dss'])

print("DSS V2 scores by model:")
for model, scores in sorted(
        model_dss.items()):
    avg = sum(scores) / len(scores)
    print(f"  {model:<12} {avg:.4f} "
          f"(n={len(scores)})")

# Summary by category
cat_dss = defaultdict(list)
for s in dss_scores:
    cat_dss[s['category']].append(s['dss'])

print(f"\nDSS V2 scores by category:")
for cat, scores in sorted(
        cat_dss.items()):
    avg = sum(scores) / len(scores)
    print(f"  {cat:<25} {avg:.4f}")

print(f"\nTotal DSS scores: {len(dss_scores)}")
print(f"Saved ✅")

DSS V2 scores by model:
  claude       0.8785 (n=93)
  deepseek     0.8489 (n=95)
  gemini       0.8847 (n=95)
  gpt4o        0.8684 (n=95)
  grok         0.8274 (n=93)

DSS V2 scores by category:
  logical_syllogism         0.8820
  mathematical_proof        0.9250
  probabilistic             0.8974
  quantum_reasoning         0.8780
  spatial_reasoning         0.7530
  temporal_reasoning        0.7021

Total DSS scores: 471
Saved ✅


In [25]:
# Bootstrap confidence intervals
# for DSS scores by model

import random
import json
from collections import defaultdict

random.seed(42)

with open(f'{V2_RESULTS_PATH}/'
          f'dss_scores.json', 'r') as f:
    dss_data = json.load(f)

scores = dss_data['scores']

def bootstrap_ci(data, n_iter=10000,
                 ci=0.95):
    means = []
    n = len(data)
    for _ in range(n_iter):
        sample = random.choices(data, k=n)
        means.append(sum(sample) / n)
    means.sort()
    lower = int((1 - ci) / 2 * n_iter)
    upper = int((1 + ci) / 2 * n_iter)
    return (means[lower],
            sum(data) / n,
            means[upper])

# By model
model_scores = defaultdict(list)
for s in scores:
    model_scores[s['model']].append(
        s['dss'])

print("DSS Bootstrap CI (95%) by model:")
print(f"{'Model':<12} {'Lower':<8} "
      f"{'Mean':<8} {'Upper':<8} n")
print("-" * 45)

model_stats = {}
for model, vals in sorted(
        model_scores.items()):
    lo, mean, hi = bootstrap_ci(vals)
    model_stats[model] = (lo, mean, hi)
    print(f"{model:<12} {lo:.4f}   "
          f"{mean:.4f}   {hi:.4f}  "
          f"{len(vals)}")

# By category
cat_scores = defaultdict(list)
for s in scores:
    cat_scores[s['category']].append(
        s['dss'])

print(f"\nDSS Bootstrap CI (95%) "
      f"by category:")
print(f"{'Category':<25} {'Lower':<8} "
      f"{'Mean':<8} {'Upper':<8} n")
print("-" * 55)

for cat, vals in sorted(
        cat_scores.items()):
    lo, mean, hi = bootstrap_ci(vals)
    print(f"{cat:<25} {lo:.4f}   "
          f"{mean:.4f}   {hi:.4f}  "
          f"{len(vals)}")

# Check if model CIs overlap
print(f"\nCI overlap analysis:")
models = sorted(model_stats.keys())
for i in range(len(models)):
    for j in range(i+1, len(models)):
        m1, m2 = models[i], models[j]
        lo1, mean1, hi1 = model_stats[m1]
        lo2, mean2, hi2 = model_stats[m2]
        overlap = not (hi1 < lo2 or
                       hi2 < lo1)
        status = '⚠️ overlapping' \
                 if overlap else \
                 '✅ distinct'
        print(f"  {m1} vs {m2}: {status}")

DSS Bootstrap CI (95%) by model:
Model        Lower    Mean     Upper    n
---------------------------------------------
claude       0.8527   0.8785   0.9005  93
deepseek     0.8137   0.8489   0.8779  95
gemini       0.8479   0.8847   0.9174  95
gpt4o        0.8147   0.8684   0.9168  95
grok         0.7876   0.8274   0.8608  93

DSS Bootstrap CI (95%) by category:
Category                  Lower    Mean     Upper    n
-------------------------------------------------------
logical_syllogism         0.8610   0.8820   0.8995  100
mathematical_proof        0.9100   0.9250   0.9400  100
probabilistic             0.8837   0.8974   0.9112  98
quantum_reasoning         0.8400   0.8780   0.9107  75
spatial_reasoning         0.6640   0.7530   0.8340  50
temporal_reasoning        0.6083   0.7021   0.7906  48

CI overlap analysis:
  claude vs deepseek: ⚠️ overlapping
  claude vs gemini: ⚠️ overlapping
  claude vs gpt4o: ⚠️ overlapping
  claude vs grok: ⚠️ overlapping
  deepseek vs gemini: ⚠️ ove

In [26]:
# Test CVS scoring on one example

def build_cvs_prompt(question,
                     r1_response,
                     critique):
    return f"""You are evaluating the
quality of a peer critique.

ORIGINAL QUESTION:
{question['prompt'][:300]}

RESPONSE THAT WAS CRITIQUED:
{r1_response[:400]}

CRITIQUE TO EVALUATE:
{critique[:400]}

Score each dimension 0 or 1:

1. step_cited: Did the critique
   reference a specific numbered
   step from the response?

2. error_identified: Did the critique
   name a specific error or weakness?

3. alternative_provided: Did the
   critique offer a better approach
   or correction?

4. root_cause: Did the critique explain
   why the error occurred?

5. no_ad_hominem: Was the critique
   about the reasoning not the model?

Respond in JSON only:
{{
  "step_cited": 0,
  "error_identified": 0,
  "alternative_provided": 0,
  "root_cause": 0,
  "no_ad_hominem": 0
}}"""


def score_cvs_jury(question, r1_response,
                   critique, jury_model):
    prompt = build_cvs_prompt(
        question, r1_response, critique)
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning':
                    'true',
                'Content-Type':
                    'application/json'},
            json={
                'model':       jury_model,
                'prompt':      prompt,
                'num_predict': 200,
                'stream':      False},
            timeout=120)

        if r.status_code == 200:
            text = r.json().get(
                'response', '')
            text = text.replace(
                '```json', '').replace(
                '```', '').strip()
            start = text.find('{')
            end   = text.rfind('}')
            if start == -1 or end == -1:
                return None
            scores = json.loads(
                text[start:end+1])

            # Compute CVS weighted score
            cvs = round(
                scores.get('step_cited', 0)
                * 0.20
                + scores.get(
                    'error_identified', 0)
                * 0.25
                + scores.get(
                    'alternative_provided', 0)
                * 0.20
                + scores.get(
                    'root_cause', 0) * 0.20
                + scores.get(
                    'no_ad_hominem', 0)
                * 0.15, 4)

            scores['cvs_score'] = cvs
            scores['jury_model'] = jury_model
            scores['error']      = None
            return scores
        else:
            return {
                'jury_model': jury_model,
                'cvs_score':  None,
                'error': f"Status "
                         f"{r.status_code}"}
    except Exception as e:
        return {
            'jury_model': jury_model,
            'cvs_score':  None,
            'error':      str(e)[:80]}

print("CVS scorer defined ✅")

# Load R2 responses for test
with open(f'{V2_RESULTS_PATH}/'
          f'round2_all_responses.json',
          'r') as f:
    r2_all = json.load(f)

r2_lookup = {}
for r in r2_all['responses']:
    qid      = r.get('question_id')
    reviewer = r.get('reviewer')
    resp     = r.get('response')
    if qid and reviewer and resp:
        if qid not in r2_lookup:
            r2_lookup[qid] = {}
        r2_lookup[qid][reviewer] = resp

# Test on ADV-01
test_q  = next(q for q in ALL_QUESTIONS
               if q['id'] == 'ADV-01')
test_r1 = r1_lookup.get(
    'ADV-01', {}).get('grok', '')
test_r2 = r2_lookup.get(
    'ADV-01', {}).get('claude', '')

if test_r1 and test_r2:
    print(f"\nTest CVS scoring ADV-01:")
    print(f"  Claude reviews Grok's R1")
    for jury in ['mistral-nemo:latest',
                 'deepseek-r1:14b',
                 'gemma3:4b']:
        result = score_cvs_jury(
            test_q, test_r1, test_r2, jury)
        if result and not result.get('error'):
            print(f"  {jury[:20]:<22} "
                  f"CVS: {result['cvs_score']:.3f}")
        else:
            print(f"  {jury[:20]:<22} "
                  f"❌ {result.get('error', 'failed')[:40]}")
else:
    print("Missing test data")

CVS scorer defined ✅

Test CVS scoring ADV-01:
  Claude reviews Grok's R1
  mistral-nemo:latest    CVS: 0.600
  deepseek-r1:14b        CVS: 0.600
  gemma3:4b              CVS: 0.450


In [27]:
r = requests.post(
    f'{NGROK_URL}/api/generate',
    headers={
        'ngrok-skip-browser-warning': 'true',
        'Content-Type': 'application/json'},
    json={'model': 'mistral-nemo:latest',
          'prompt': 'ping',
          'num_predict': 1,
          'stream': False},
    timeout=30)
print(f"NGROK: {'✅' if r.status_code == 200 else '❌'}")

NGROK: ✅


In [28]:
# CVS Validation Sample — 40 questions

import random
import time

random.seed(42)

# Build proportional sample
sample_questions = []
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions,
        min(n, len(questions)))
    sample_questions.extend(sampled)

print(f"CVS validation sample:")
print(f"  Questions: {len(sample_questions)}")
total = (len(sample_questions)
         * len(SUBJECT_MODELS)
         * len(JURY_MODELS))
print(f"  Total calls: {total}")
print(f"  Est. time: ~{total * 8 // 60} min")
print(f"Starting...\n")

cvs_validation = []
errors = 0

for q in sample_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cvs_validation.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'reviewer':    reviewer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

with open(f'{V2_RESULTS_PATH}/'
          f'cvs_validation.json', 'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cvs_validation),
        'errors':    errors,
        'results':   cvs_validation
    }, f, indent=2)

print(f"\nCVS validation complete:")
print(f"  Scores: {len(cvs_validation)}")
print(f"  Errors: {errors}")
print(f"  Saved ✅")

CVS validation sample:
  Questions: 40
  Total calls: 600
  Est. time: ~80 min
Starting...

  ADV-04     claude       CVS: 0.750
  ADV-04     gpt4o        CVS: 0.100
  ADV-04     gemini       CVS: 0.683
  ADV-04     deepseek     CVS: 0.750
  ADV-04     grok         CVS: 0.683
  ADV-01     claude       CVS: 0.683
  ADV-01     gpt4o        CVS: 0.750
  ADV-01     gemini       CVS: 0.450
  ADV-01     deepseek     CVS: 0.617
  ADV-01     grok         CVS: 0.525
  ADV-09     claude       CVS: 0.400
  ADV-09     gpt4o        CVS: 0.100
  ADV-09     gemini       CVS: 0.550
  ADV-09     deepseek     CVS: 0.550
  ADV-09     grok         CVS: 0.617
  ADV-08     claude       CVS: 0.525
  ADV-08     gpt4o        CVS: 0.683
  ADV-08     gemini       CVS: 0.817
  ADV-08     deepseek     CVS: 0.750
  ADV-08     grok         CVS: 0.683
  LSQ-08     claude       CVS: 0.750
  LSQ-08     gpt4o        CVS: 0.533
  LSQ-08     gemini       CVS: 0.400
  LSQ-08     deepseek     CVS: 0.683
  LSQ-08     grok   

In [29]:
# CVS Validation — Inter-rater analysis

import json
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'cvs_validation.json', 'r') as f:
    cvs_val = json.load(f)

results = [r for r in cvs_val['results']
           if r.get('cvs_score') is not None]

# Group by question+reviewer pair
pairs = defaultdict(dict)
for r in results:
    key = (f"{r['question_id']}_"
           f"{r['reviewer']}")
    pairs[key][r['jury_model']] = \
        r['cvs_score']

jury_models = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

print("CVS Inter-rater correlations:")
for i in range(len(jury_models)):
    for j in range(i+1, len(jury_models)):
        j1 = jury_models[i]
        j2 = jury_models[j]
        shared = [
            (v[j1], v[j2])
            for v in pairs.values()
            if j1 in v and j2 in v]
        if shared:
            s1 = [x[0] for x in shared]
            s2 = [x[1] for x in shared]
            n  = len(shared)
            mean1 = sum(s1) / n
            mean2 = sum(s2) / n
            num = sum(
                (a - mean1) * (b - mean2)
                for a, b in shared)
            den = (
                sum((a-mean1)**2
                    for a in s1) *
                sum((b-mean2)**2
                    for b in s2)) ** 0.5
            corr = num/den if den else 0
            print(f"  {j1[:15]:<16} vs "
                  f"{j2[:15]:<16} "
                  f"r={corr:.3f} "
                  f"(n={n})")

# Average CVS by model
model_cvs = defaultdict(list)
for r in results:
    model_cvs[r['reviewer']].append(
        r['cvs_score'])

print(f"\nAverage CVS by reviewer:")
for model, scores in sorted(
        model_cvs.items()):
    avg = sum(scores) / len(scores)
    print(f"  {model:<12} {avg:.3f} "
          f"(n={len(scores)})")

# Score distribution
all_scores = [r['cvs_score']
              for r in results]
print(f"\nScore distribution:")
print(f"  Min:  {min(all_scores):.3f}")
print(f"  Max:  {max(all_scores):.3f}")
print(f"  Mean: {sum(all_scores)/len(all_scores):.3f}")
buckets = [0]*5
for s in all_scores:
    idx = min(int(s * 5), 4)
    buckets[idx] += 1
labels = ['0.0-0.2','0.2-0.4',
          '0.4-0.6','0.6-0.8','0.8-1.0']
for label, count in zip(labels, buckets):
    print(f"  {label}: {count}")

CVS Inter-rater correlations:
  mistral-nemo:la  vs deepseek-r1:14b  r=0.569 (n=177)
  mistral-nemo:la  vs gemma3:4b        r=0.539 (n=180)
  deepseek-r1:14b  vs gemma3:4b        r=0.524 (n=196)

Average CVS by reviewer:
  claude       0.564 (n=113)
  deepseek     0.639 (n=115)
  gemini       0.582 (n=116)
  gpt4o        0.404 (n=116)
  grok         0.608 (n=116)

Score distribution:
  Min:  0.000
  Max:  1.000
  Mean: 0.559
  0.0-0.2: 65
  0.2-0.4: 80
  0.4-0.6: 107
  0.6-0.8: 148
  0.8-1.0: 176


In [30]:
# Find remaining questions not in sample

import random
random.seed(42)

# Rebuild the same sample using same seed
sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions,
        min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

print(f"Already scored: {len(sample_ids)}")
print(f"Remaining:      {len(remaining)}")
print(f"\nBy category:")
from collections import defaultdict
cat_counts = defaultdict(int)
for q in remaining:
    cat_counts[q['category']] += 1
for cat, count in sorted(
        cat_counts.items()):
    print(f"  {cat:<25} {count}")

Already scored: 40
Remaining:      160

By category:
  adversarial               16
  causal_chain              16
  counterfactual            16
  ethical_dilemma           12
  frontier_reasoning        12
  logical_syllogism         16
  mathematical_proof        16
  meta_reasoning            12
  probabilistic             16
  quantum_reasoning         12
  spatial_reasoning         8
  temporal_reasoning        8


In [31]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'adversarial'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — adversarial
Questions:  16
Completed:  0
Starting...

  ADV-02     claude       CVS: 0.683
  ADV-02     gpt4o        CVS: 0.400
  ADV-02     gemini       CVS: 0.667
  ADV-02     deepseek     CVS: 0.683
  ADV-02     grok         CVS: 0.750
  ADV-03     claude       CVS: 0.617
  ADV-03     gpt4o        CVS: 0.383
  ADV-03     gemini       CVS: 0.600
  ADV-03     deepseek     CVS: 0.550
  ADV-03     grok         CVS: 0.750
  ADV-05     claude       CVS: 0.617
  ADV-05     gpt4o        CVS: 0.683
  ADV-05     gemini       CVS: 0.483
  ADV-05     deepseek     CVS: 0.683
  ADV-05     grok         CVS: 0.625
  ADV-06     claude       CVS: 0.233
  ADV-06     gpt4o        CVS: 0.100
  ADV-06     gemini       CVS: 0.750
  ADV-06     deepseek     CVS: 0.683
  ADV-06     grok         CVS: 0.617
  ADV-07     claude       CVS: 0.183
  ADV-07     gpt4o        CVS: 0.233
  ADV-07     gemini       CVS: 0.683
  ADV-07     deepseek     CVS: 0.725
  ADV-07     grok         CVS: 0.617
  ADV-10  

In [32]:
# Verify remaining questions count

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

adv_remaining = [q for q in remaining
                 if q['category'] == 'adversarial']

print(f"Sample IDs count: {len(sample_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"Adversarial remaining: {len(adv_remaining)}")
for q in adv_remaining:
    print(f"  {q['id']}")

Sample IDs count: 40
Total remaining: 160
Adversarial remaining: 16
  ADV-02
  ADV-03
  ADV-05
  ADV-06
  ADV-07
  ADV-10
  ADV-11
  ADV-12
  ADV-13
  ADV-14
  ADV-15
  ADV-16
  ADV-17
  ADV-18
  ADV-19
  ADV-20


In [33]:
# Check errors in adversarial CVS

with open(f'{V2_RESULTS_PATH}/'
          f'cvs_full_adversarial.json',
          'r') as f:
    adv_data = json.load(f)

errors = [r for r in adv_data['results']
          if r.get('error')]
print(f"Errors: {len(errors)}")
for e in errors[:5]:
    print(f"  {e.get('question_id')} "
          f"{e.get('reviewer')} "
          f"{e.get('jury_model','')[:15]} "
          f"{str(e.get('error',''))[:50]}")

Errors: 0


In [34]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'causal_chain'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — causal_chain
Questions:  16
Completed:  0
Starting...

  CCQ-03     claude       CVS: 0.467
  CCQ-03     gpt4o        CVS: 0.617
  CCQ-03     gemini       CVS: 0.617
  CCQ-03     deepseek     CVS: 0.617
  CCQ-03     grok         CVS: 0.550
  CCQ-04     claude       CVS: 0.617
  CCQ-04     gpt4o        CVS: 0.317
  CCQ-04     gemini       CVS: 0.750
  CCQ-04     deepseek     CVS: 0.733
  CCQ-04     grok         CVS: 0.550
  CCQ-05     claude       CVS: 0.625
  CCQ-05     gpt4o        CVS: 0.100
  CCQ-05     gemini       CVS: 0.500
  CCQ-05     deepseek     CVS: 0.683
  CCQ-05     grok         CVS: 0.683
  CCQ-06     claude       CVS: 0.733
  CCQ-06     gpt4o        CVS: 0.867
  CCQ-06     gemini       CVS: 0.517
  CCQ-06     deepseek     CVS: 0.733
  CCQ-06     grok         CVS: 0.733
  CCQ-07     claude       CVS: 0.617
  CCQ-07     gpt4o        CVS: 0.533
  CCQ-07     gemini       CVS: 0.617
  CCQ-07     deepseek     CVS: 0.550
  CCQ-07     grok         CVS: 0.617
  CCQ-08 

In [35]:
# Debug — check if CCQ-01 through CCQ-04
# are in sample_ids

sample_check = ['CCQ-01', 'CCQ-02',
                'CCQ-03', 'CCQ-04',
                'CCQ-19', 'CCQ-14']

print("Sample ID membership check:")
for qid in sample_check:
    in_sample = qid in sample_ids
    print(f"  {qid}: "
          f"{'in sample' if in_sample else 'NOT in sample'}")

print(f"\nTotal sample_ids: {len(sample_ids)}")
print(f"Sample of IDs in set:")
for sid in list(sample_ids)[:10]:
    print(f"  {sid}")

Sample ID membership check:
  CCQ-01: in sample
  CCQ-02: in sample
  CCQ-03: NOT in sample
  CCQ-04: NOT in sample
  CCQ-19: in sample
  CCQ-14: in sample

Total sample_ids: 40
Sample of IDs in set:
  MPQ-18
  PRQ-08
  CCQ-02
  QRQ-10
  PRQ-07
  ADV-04
  MRQ-13
  CFQ-11
  CCQ-01
  CCQ-19


In [36]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'counterfactual'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: counterfactual
Questions to run: 16
  CFQ-01
  CFQ-02
  CFQ-05
  CFQ-06
  CFQ-07
  CFQ-08
  CFQ-09
  CFQ-10
  CFQ-12
  CFQ-14
  CFQ-15
  CFQ-16
  CFQ-17
  CFQ-18
  CFQ-19
  CFQ-20


## CVS Full Run — Note on Counterfactual Output

The console output for the counterfactual
category CVS run was accidentally cleared
before being saved to the session log.

The data was not lost. The checkpoint
system saved all results to Drive at:
cvs_full_counterfactual.json

Results: 237 scoring calls
Errors:  3
Status:  Saved ✅

The partial output recovered from session
shows CFQ-07 through CFQ-20 completed
cleanly with CVS scores ranging from
0.400 to 0.825 across all five models
and three jury models.

In [37]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'counterfactual'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

  CFQ-07     deepseek     CVS: 0.617
  CFQ-07     grok         CVS: 0.617
  CFQ-08     claude       CVS: 0.617
  CFQ-08     gpt4o        CVS: 0.683
  CFQ-08     gemini       CVS: 0.683
  CFQ-08     deepseek     CVS: 0.617
  CFQ-08     grok         CVS: 0.683
  CFQ-09     claude       CVS: 0.683
  CFQ-09     gpt4o        CVS: 0.617
  CFQ-09     gemini       CVS: 0.617
  CFQ-09     deepseek     CVS: 0.683
  CFQ-09     grok         CVS: 0.617
  CFQ-10     claude       CVS: 0.683
  CFQ-10     gpt4o        CVS: 0.617
  CFQ-10     gemini       CVS: 0.725
  CFQ-10     deepseek     CVS: 0.750
  CFQ-10     grok         CVS: 0.617
  CFQ-12     claude       CVS: 0.683
  CFQ-12     gpt4o        CVS: 0.617
  CFQ-12     gemini       CVS: 0.683
  CFQ-12     deepseek     CVS: 0.617
  CFQ-12     grok         CVS: 0.617
  CFQ-14     claude       CVS: 0.467
  CFQ-14     gpt4o        CVS: 0.683
  CFQ-14     gemini       CVS: 0.750
  CFQ-14     deepseek     CVS: 0.617
  CFQ-14     grok         CVS: 0.750
 

In [38]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'ethical_dilemma'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: ethical_dilemma
Questions to run: 12
  EDQ-01
  EDQ-02
  EDQ-06
  EDQ-07
  EDQ-08
  EDQ-09
  EDQ-10
  EDQ-11
  EDQ-12
  EDQ-13
  EDQ-14
  EDQ-15


In [39]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'ethical_dilemma'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — ethical_dilemma
Questions:  12
Completed:  0
Starting...

  EDQ-01     claude       CVS: 0.550
  EDQ-01     gpt4o        CVS: 0.683
  EDQ-01     gemini       CVS: 0.817
  EDQ-01     deepseek     CVS: 0.617
  EDQ-01     grok         CVS: 0.617
  EDQ-02     claude       CVS: 0.683
  EDQ-02     gpt4o        CVS: 0.550
  EDQ-02     gemini       CVS: 0.617
  EDQ-02     deepseek     CVS: 0.625
  EDQ-02     grok         CVS: 0.550
  EDQ-06     claude       CVS: 0.550
  EDQ-06     gpt4o        CVS: 0.550
  EDQ-06     gemini       CVS: 0.617
  EDQ-06     deepseek     CVS: 0.400
  EDQ-06     grok         CVS: 0.483
  EDQ-07     claude       CVS: 0.750
  EDQ-07     gpt4o        CVS: 0.550
  EDQ-07     gemini       CVS: 0.617
  EDQ-07     deepseek     CVS: 0.617
  EDQ-07     grok         CVS: 0.617
  EDQ-08     claude       CVS: 0.617
  EDQ-08     gpt4o        CVS: 0.617
  EDQ-08     gemini       CVS: 0.617
  EDQ-08     deepseek     CVS: 0.617
  EDQ-08     grok         CVS: 0.617
  EDQ-

In [40]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'frontier_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: frontier_reasoning
Questions to run: 12
  FRQ-01
  FRQ-02
  FRQ-03
  FRQ-05
  FRQ-06
  FRQ-07
  FRQ-08
  FRQ-09
  FRQ-10
  FRQ-13
  FRQ-14
  FRQ-15


In [41]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'frontier_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — frontier_reasoning
Questions:  12
Completed:  0
Starting...

  FRQ-01     claude       CVS: 0.317
  FRQ-01     gpt4o        CVS: 0.100
  FRQ-01     gemini       CVS: 0.683
  FRQ-01     deepseek     CVS: 0.683
  FRQ-01     grok         CVS: 0.683
  FRQ-02     claude       CVS: 0.625
  FRQ-02     gpt4o        CVS: 0.100
  FRQ-02     gemini       CVS: 0.625
  FRQ-02     deepseek     CVS: 0.400
  FRQ-02     grok         CVS: 0.750
  FRQ-03     claude       CVS: 0.400
  FRQ-03     gpt4o        CVS: 0.550
  FRQ-03     gemini       CVS: 0.400
  FRQ-03     deepseek     CVS: 0.683
  FRQ-03     grok         CVS: 0.550
  FRQ-05     claude       CVS: 0.617
  FRQ-05     gpt4o        CVS: 0.550
  FRQ-05     gemini       CVS: 0.725
  FRQ-05     deepseek     CVS: 0.617
  FRQ-05     grok         CVS: 0.683
  FRQ-06     claude       CVS: 0.625
  FRQ-06     gpt4o        CVS: 0.483
  FRQ-06     gemini       CVS: 0.483
  FRQ-06     deepseek     CVS: 0.683
  FRQ-06     grok         CVS: 0.617
  F

In [42]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'logical_syllogism'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: logical_syllogism
Questions to run: 16
  LSQ-01
  LSQ-02
  LSQ-06
  LSQ-07
  LSQ-09
  LSQ-10
  LSQ-11
  LSQ-12
  LSQ-13
  LSQ-14
  LSQ-15
  LSQ-16
  LSQ-17
  LSQ-18
  LSQ-19
  LSQ-20


In [43]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'logical_syllogism'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — logical_syllogism
Questions:  16
Completed:  0
Starting...

  LSQ-01     claude       CVS: 0.233
  LSQ-01     gpt4o        CVS: 0.100
  LSQ-01     gemini       CVS: 0.750
  LSQ-01     deepseek     CVS: 0.750
  LSQ-01     grok         CVS: 0.683
  LSQ-02     claude       CVS: 0.617
  LSQ-02     gpt4o        CVS: 0.167
  LSQ-02     gemini       CVS: 0.683
  LSQ-02     deepseek     CVS: 0.550
  LSQ-02     grok         CVS: 0.167
  LSQ-06     claude       CVS: 0.683
  LSQ-06     gpt4o        CVS: 0.683
  LSQ-06     gemini       CVS: 0.750
  LSQ-06     deepseek     CVS: 0.725
  LSQ-06     grok         CVS: 0.683
  LSQ-07     claude       CVS: 0.617
  LSQ-07     gpt4o        CVS: 0.250
  LSQ-07     gemini       CVS: 0.617
  LSQ-07     deepseek     CVS: 0.683
  LSQ-07     grok         CVS: 0.617
  LSQ-09     claude       CVS: 0.617
  LSQ-09     gpt4o        CVS: 0.750
  LSQ-09     gemini       CVS: 0.683
  LSQ-09     deepseek     CVS: 0.617
  LSQ-09     grok         CVS: 0.617
  LS

In [44]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'mathematical_proof'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: mathematical_proof
Questions to run: 16
  MPQ-01
  MPQ-02
  MPQ-03
  MPQ-04
  MPQ-05
  MPQ-06
  MPQ-07
  MPQ-09
  MPQ-10
  MPQ-11
  MPQ-12
  MPQ-13
  MPQ-16
  MPQ-17
  MPQ-19
  MPQ-20


In [45]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'mathematical_proof'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — mathematical_proof
Questions:  16
Completed:  0
Starting...

  MPQ-01     claude       CVS: 0.817
  MPQ-01     gpt4o        CVS: 0.233
  MPQ-01     gemini       CVS: 0.883
  MPQ-01     deepseek     CVS: 0.317
  MPQ-01     grok         CVS: 0.167
  MPQ-02     claude       CVS: 0.233
  MPQ-02     gpt4o        CVS: 0.100
  MPQ-02     gemini       CVS: 0.617
  MPQ-02     deepseek     CVS: 0.683
  MPQ-02     grok         CVS: 0.617
  MPQ-03     claude       CVS: 0.100
  MPQ-03     gpt4o        CVS: 0.233
  MPQ-03     gemini       CVS: 0.817
  MPQ-03     deepseek     CVS: 0.817
  MPQ-03     grok         CVS: 0.617
  MPQ-04     claude       CVS: 0.683
  MPQ-04     gpt4o        CVS: 0.233
  MPQ-04     gemini       CVS: 0.300
  MPQ-04     deepseek     CVS: 0.683
  MPQ-04     grok         CVS: 0.167
  MPQ-05     claude       CVS: 0.233
  MPQ-05     gpt4o        CVS: 0.167
  MPQ-05     gemini       CVS: 0.750
  MPQ-05     deepseek     CVS: 0.483
  MPQ-05     grok         CVS: 0.533
  M

In [46]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'meta_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: meta_reasoning
Questions to run: 12
  MRQ-01
  MRQ-02
  MRQ-03
  MRQ-04
  MRQ-06
  MRQ-07
  MRQ-08
  MRQ-09
  MRQ-11
  MRQ-12
  MRQ-14
  MRQ-15


In [47]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'meta_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — meta_reasoning
Questions:  12
Completed:  0
Starting...

  MRQ-01     claude       CVS: 0.467
  MRQ-01     gpt4o        CVS: 0.533
  MRQ-01     gemini       CVS: 0.683
  MRQ-01     deepseek     CVS: 0.683
  MRQ-01     grok         CVS: 0.600
  MRQ-02     claude       CVS: 0.625
  MRQ-02     gpt4o        CVS: 0.600
  MRQ-02     gemini       CVS: 0.617
  MRQ-02     deepseek     CVS: 0.750
  MRQ-02     grok         CVS: 0.617
  MRQ-03     claude       CVS: 0.683
  MRQ-03     gpt4o        CVS: 0.617
  MRQ-03     gemini       CVS: 0.683
  MRQ-03     deepseek     CVS: 0.350
  MRQ-03     grok         CVS: 0.683
  MRQ-04     claude       CVS: 0.600
  MRQ-04     gpt4o        CVS: 0.183
  MRQ-04     gemini       CVS: 0.667
  MRQ-04     deepseek     CVS: 0.550
  MRQ-04     grok         CVS: 0.400
  MRQ-06     claude       CVS: 0.725
  MRQ-06     gpt4o        CVS: 0.750
  MRQ-06     gemini       CVS: 0.617
  MRQ-06     deepseek     CVS: 0.683
  MRQ-06     grok         CVS: 0.600
  MRQ-0

In [50]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'probabilistic'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: probabilistic
Questions to run: 16
  PRQ-01
  PRQ-02
  PRQ-04
  PRQ-05
  PRQ-06
  PRQ-09
  PRQ-10
  PRQ-11
  PRQ-12
  PRQ-13
  PRQ-14
  PRQ-15
  PRQ-16
  PRQ-18
  PRQ-19
  PRQ-20


In [51]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'probabilistic'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — probabilistic
Questions:  16
Completed:  0
Starting...

  PRQ-01     gpt4o        CVS: 0.075
  PRQ-01     gemini       CVS: 0.100
  PRQ-01     deepseek     CVS: 0.550
  PRQ-01     grok         CVS: 0.725
  PRQ-02     claude       CVS: 0.233
  PRQ-02     gpt4o        CVS: 0.100
  PRQ-02     gemini       CVS: 0.817
  PRQ-02     deepseek     CVS: 0.750
  PRQ-02     grok         CVS: 0.383
  PRQ-04     claude       CVS: 0.683
  PRQ-04     gpt4o        CVS: 0.100
  PRQ-04     gemini       CVS: 0.167
  PRQ-04     deepseek     CVS: 0.617
  PRQ-04     grok         CVS: 0.683
  PRQ-05     claude       CVS: 0.817
  PRQ-05     gpt4o        CVS: 0.683
  PRQ-05     gemini       CVS: 0.683
  PRQ-05     deepseek     CVS: 0.617
  PRQ-05     grok         CVS: 0.750
  PRQ-06     claude       CVS: 0.725
  PRQ-06     gpt4o        CVS: 0.683
  PRQ-06     gemini       CVS: 0.617
  PRQ-06     deepseek     CVS: 0.550
  PRQ-06     grok         CVS: 0.750
  PRQ-09     claude       CVS: 0.617
  PRQ-09

In [52]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'quantum_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: quantum_reasoning
Questions to run: 12
  QRQ-02
  QRQ-03
  QRQ-04
  QRQ-05
  QRQ-06
  QRQ-07
  QRQ-08
  QRQ-11
  QRQ-12
  QRQ-13
  QRQ-14
  QRQ-15


In [53]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'quantum_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — quantum_reasoning
Questions:  12
Completed:  0
Starting...

  QRQ-02     claude       CVS: 0.600
  QRQ-02     gpt4o        CVS: 0.233
  QRQ-02     gemini       CVS: 0.683
  QRQ-02     deepseek     CVS: 0.750
  QRQ-02     grok         CVS: 0.233
  QRQ-03     claude       CVS: 0.683
  QRQ-03     gpt4o        CVS: 0.167
  QRQ-03     gemini       CVS: 0.683
  QRQ-03     deepseek     CVS: 0.750
  QRQ-03     grok         CVS: 0.417
  QRQ-04     claude       CVS: 0.300
  QRQ-04     gpt4o        CVS: 0.117
  QRQ-04     gemini       CVS: 0.425
  QRQ-04     deepseek     CVS: 0.817
  QRQ-04     grok         CVS: 0.617
  QRQ-05     claude       CVS: 0.483
  QRQ-05     gpt4o        CVS: 0.100
  QRQ-05     gemini       CVS: 0.750
  QRQ-05     deepseek     CVS: 0.617
  QRQ-05     grok         CVS: 0.750
  QRQ-06     claude       CVS: 0.650
  QRQ-06     gpt4o        CVS: 0.683
  QRQ-06     gemini       CVS: 0.725
  QRQ-06     deepseek     CVS: 0.883
  QRQ-06     grok         CVS: 0.617
  QR

In [13]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'spatial_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: spatial_reasoning
Questions to run: 8
  SRQ-01
  SRQ-02
  SRQ-03
  SRQ-04
  SRQ-05
  SRQ-08
  SRQ-09
  SRQ-10


In [15]:
SUBJECT_MODELS = [
    'claude', 'gpt4o', 'gemini',
    'deepseek', 'grok'
]

JURY_MODELS = [
    'mistral-nemo:latest',
    'deepseek-r1:14b',
    'gemma3:4b'
]

PEER_ASSIGNMENTS = {
    'claude':   'grok',
    'gpt4o':    'claude',
    'gemini':   'gpt4o',
    'deepseek': 'gemini',
    'grok':     'deepseek'
}

print("Variables defined ✅")

Variables defined ✅


In [17]:
import re
import json
import requests

def parse_jury_response(text, jury_model):
    text = text.replace(
        '```json', '').replace(
        '```', '').strip()
    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1:
        return None
    json_str = text[start:end+1]
    json_str = re.sub(
        r',?\s*"cvs_score"\s*:[^,}\n]+',
        '', json_str)
    json_str = re.sub(
        r',\s*}', '}', json_str)
    try:
        scores = json.loads(json_str)
    except Exception:
        return None
    scores['cvs_score'] = round(
        scores.get('step_cited', 0) * 0.20
        + scores.get('error_identified', 0) * 0.25
        + scores.get('alternative_provided', 0) * 0.20
        + scores.get('root_cause', 0) * 0.20
        + scores.get('no_ad_hominem', 0) * 0.15, 4)
    return scores


def build_cvs_prompt(question,
                     r1_response,
                     critique):
    return f"""You are evaluating the
quality of a peer critique.

ORIGINAL QUESTION:
{question['prompt'][:300]}

RESPONSE THAT WAS CRITIQUED:
{r1_response[:400]}

CRITIQUE TO EVALUATE:
{critique[:400]}

Score each dimension 0 or 1:
1. step_cited: Did the critique reference
   a specific numbered step?
2. error_identified: Did the critique name
   a specific error or weakness?
3. alternative_provided: Did the critique
   offer a better approach?
4. root_cause: Did the critique explain
   why the error occurred?
5. no_ad_hominem: Was the critique about
   the reasoning not the model?

Respond in JSON only:
{{
  "step_cited": 0,
  "error_identified": 0,
  "alternative_provided": 0,
  "root_cause": 0,
  "no_ad_hominem": 0
}}"""


def score_cvs_jury(question, response,
                   critique, jury_model):
    prompt = build_cvs_prompt(
        question, response, critique)
    try:
        r = requests.post(
            f'{NGROK_URL}/api/generate',
            headers={
                'ngrok-skip-browser-warning':
                    'true',
                'Content-Type':
                    'application/json'},
            json={
                'model':       jury_model,
                'prompt':      prompt,
                'num_predict': 200,
                'stream':      False},
            timeout=120)
        if r.status_code == 200:
            text = r.json().get(
                'response', '')
            scores = parse_jury_response(
                text, jury_model)
            if scores is None:
                return {
                    'jury_model': jury_model,
                    'cvs_score':  None,
                    'error': 'Parse failed'}
            scores['jury_model'] = jury_model
            scores['error']      = None
            return scores
        else:
            return {
                'jury_model': jury_model,
                'cvs_score':  None,
                'error': f"Status "
                         f"{r.status_code}"}
    except Exception as e:
        return {
            'jury_model': jury_model,
            'cvs_score':  None,
            'error':      str(e)[:80]}

print("CVS scoring functions defined ✅")

CVS scoring functions defined ✅


In [18]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'spatial_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — spatial_reasoning
Questions:  8
Completed:  0
Starting...

  SRQ-01     claude       CVS: 0.267
  SRQ-01     gpt4o        CVS: 0.233
  SRQ-01     gemini       CVS: 0.300
  SRQ-01     deepseek     CVS: 0.617
  SRQ-01     grok         CVS: 0.275
  SRQ-02     claude       CVS: 0.825
  SRQ-02     gpt4o        CVS: 0.233
  SRQ-02     gemini       CVS: 0.233
  SRQ-02     deepseek     CVS: 0.617
  SRQ-02     grok         CVS: 0.750
  SRQ-03     claude       CVS: 0.525
  SRQ-03     gpt4o        CVS: 0.550
  SRQ-03     gemini       CVS: 0.683
  SRQ-03     deepseek     CVS: 0.683
  SRQ-03     grok         CVS: 0.750
  SRQ-04     claude       CVS: 0.683
  SRQ-04     gpt4o        CVS: 0.683
  SRQ-04     gemini       CVS: 0.817
  SRQ-04     deepseek     CVS: 0.683
  SRQ-04     grok         CVS: 0.683
  SRQ-05     claude       CVS: 0.617
  SRQ-05     gpt4o        CVS: 0.167
  SRQ-05     gemini       CVS: 0.617
  SRQ-05     deepseek     CVS: 0.683
  SRQ-05     grok         CVS: 0.683
  SRQ

In [19]:
# MUST run this at top of each cell
# before cat_questions is built

import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

# Then build remaining
remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

CURRENT_CATEGORY = 'temporal_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

print(f"Category: {CURRENT_CATEGORY}")
print(f"Questions to run: {len(cat_questions)}")
for q in cat_questions:
    print(f"  {q['id']}")

Category: temporal_reasoning
Questions to run: 8
  TRQ-02
  TRQ-04
  TRQ-05
  TRQ-06
  TRQ-07
  TRQ-08
  TRQ-09
  TRQ-10


In [20]:
# CVS Full Run — Remaining 160 questions
# Run by category with checkpointing

import time
import json
from collections import defaultdict

# Rebuild sample IDs to exclude
import random
random.seed(42)

sample_ids = set()
cat_groups = {}
for q in ALL_QUESTIONS:
    cat = q['category']
    if cat not in cat_groups:
        cat_groups[cat] = []
    cat_groups[cat].append(q)

for cat, questions in cat_groups.items():
    n = max(1, round(
        len(questions) / 200 * 40))
    sampled = random.sample(
        questions, min(n, len(questions)))
    for q in sampled:
        sample_ids.add(q['id'])

remaining = [q for q in ALL_QUESTIONS
             if q['id'] not in sample_ids]

# Change this between runs
CURRENT_CATEGORY = 'temporal_reasoning'

cat_questions = [
    q for q in remaining
    if q['category'] == CURRENT_CATEGORY]

checkpoint_name = (f"cvs_full_"
                   f"{CURRENT_CATEGORY}")
completed_ids   = checkpoint_mgr\
    .get_completed_ids(checkpoint_name)
cat_results     = checkpoint_mgr\
    .load(checkpoint_name)

print(f"CVS full — {CURRENT_CATEGORY}")
print(f"Questions:  {len(cat_questions)}")
print(f"Completed:  {len(cat_results)}")
print(f"Starting...\n")

errors = 0

for q in cat_questions:
    qid = q['id']
    for reviewer in SUBJECT_MODELS:
        call_id = f"{qid}_{reviewer}"
        if call_id in completed_ids:
            continue

        peer    = PEER_ASSIGNMENTS[reviewer]
        r1_resp = r1_lookup.get(
            qid, {}).get(peer)
        r2_resp = r2_lookup.get(
            qid, {}).get(reviewer)

        if not r1_resp or not r2_resp:
            continue

        jury_scores = []
        for jury in JURY_MODELS:
            result = score_cvs_jury(
                q, r1_resp,
                r2_resp, jury)

            if result and \
               not result.get('error'):
                jury_scores.append(
                    result['cvs_score'])
                cat_results.append({
                    'question_id': qid,
                    'category':    q['category'],
                    'difficulty':  q['difficulty'],
                    'reviewer':    reviewer,
                    'peer':        peer,
                    'jury_model':  jury,
                    'cvs_score':   result[
                        'cvs_score'],
                    'step_cited':  result.get(
                        'step_cited'),
                    'error_identified': result.get(
                        'error_identified'),
                    'alternative_provided': result.get(
                        'alternative_provided'),
                    'root_cause':  result.get(
                        'root_cause'),
                    'no_ad_hominem': result.get(
                        'no_ad_hominem'),
                    'error': None
                })
            else:
                errors += 1
            time.sleep(1)

        completed_ids.add(call_id)
        checkpoint_mgr.save(
            checkpoint_name, cat_results)

        if jury_scores:
            avg = sum(jury_scores) / \
                  len(jury_scores)
            print(f"  {qid:<10} "
                  f"{reviewer:<12} "
                  f"CVS: {avg:.3f}")

out_file = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{CURRENT_CATEGORY}"
            f".json")
with open(out_file, 'w') as f:
    json.dump({
        'category':  CURRENT_CATEGORY,
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(cat_results),
        'errors':    errors,
        'results':   cat_results
    }, f, indent=2)

print(f"\nCategory complete: "
      f"{CURRENT_CATEGORY}")
print(f"  Results: {len(cat_results)}")
print(f"  Errors:  {errors}")
print(f"  Saved:   {out_file}")

CVS full — temporal_reasoning
Questions:  8
Completed:  0
Starting...

  TRQ-02     claude       CVS: 0.617
  TRQ-02     gpt4o        CVS: 0.100
  TRQ-02     gemini       CVS: 0.617
  TRQ-02     deepseek     CVS: 0.625
  TRQ-02     grok         CVS: 0.617
  TRQ-04     claude       CVS: 0.550
  TRQ-04     gpt4o        CVS: 0.417
  TRQ-04     gemini       CVS: 0.400
  TRQ-04     deepseek     CVS: 0.683
  TRQ-04     grok         CVS: 0.467
  TRQ-05     claude       CVS: 0.683
  TRQ-05     gpt4o        CVS: 0.100
  TRQ-05     gemini       CVS: 0.683
  TRQ-05     deepseek     CVS: 0.617
  TRQ-05     grok         CVS: 0.167
  TRQ-06     gpt4o        CVS: 0.683
  TRQ-06     gemini       CVS: 0.683
  TRQ-06     deepseek     CVS: 0.467
  TRQ-06     grok         CVS: 0.750
  TRQ-07     claude       CVS: 0.725
  TRQ-07     gpt4o        CVS: 0.617
  TRQ-07     gemini       CVS: 0.467
  TRQ-07     deepseek     CVS: 0.750
  TRQ-07     grok         CVS: 0.383
  TRQ-08     claude       CVS: 0.683
  TR

In [21]:
# CVS Full Consolidation

import json
import os
from collections import defaultdict

all_cvs = []

# Load validation sample
with open(f'{V2_RESULTS_PATH}/'
          f'cvs_validation.json', 'r') as f:
    val_data = json.load(f)
all_cvs.extend(val_data['results'])

# Load full run categories
categories = [
    'adversarial', 'causal_chain',
    'counterfactual', 'ethical_dilemma',
    'frontier_reasoning', 'logical_syllogism',
    'mathematical_proof', 'meta_reasoning',
    'probabilistic', 'quantum_reasoning',
    'spatial_reasoning', 'temporal_reasoning'
]

for cat in categories:
    path = (f"{V2_RESULTS_PATH}/"
            f"cvs_full_{cat}.json")
    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)
        all_cvs.extend(data['results'])

# Filter valid scores
valid = [r for r in all_cvs
         if r.get('cvs_score') is not None]

# Save consolidated
with open(f'{V2_RESULTS_PATH}/'
          f'cvs_all_scores.json', 'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(valid),
        'results':   valid
    }, f, indent=2)

# Average CVS by reviewer
model_cvs = defaultdict(list)
for r in valid:
    model_cvs[r['reviewer']].append(
        r['cvs_score'])

print("CVS consolidated:")
print(f"  Total valid scores: {len(valid)}")
print(f"\nAverage CVS by reviewer:")
for model, scores in sorted(
        model_cvs.items()):
    avg = sum(scores) / len(scores)
    print(f"  {model:<12} {avg:.3f} "
          f"(n={len(scores)})")

# By category
cat_cvs = defaultdict(list)
for r in valid:
    cat_cvs[r.get('category',
                  'unknown')].append(
        r['cvs_score'])

print(f"\nAverage CVS by category:")
for cat, scores in sorted(
        cat_cvs.items()):
    avg = sum(scores) / len(scores)
    print(f"  {cat:<25} {avg:.3f}")

print(f"\nSaved ✅")

CVS consolidated:
  Total valid scores: 2885

Average CVS by reviewer:
  claude       0.573 (n=556)
  deepseek     0.644 (n=580)
  gemini       0.601 (n=582)
  gpt4o        0.433 (n=586)
  grok         0.600 (n=581)

Average CVS by category:
  adversarial               0.559
  causal_chain              0.581
  counterfactual            0.641
  ethical_dilemma           0.609
  frontier_reasoning        0.595
  logical_syllogism         0.553
  mathematical_proof        0.478
  meta_reasoning            0.606
  probabilistic             0.548
  quantum_reasoning         0.574
  spatial_reasoning         0.570
  temporal_reasoning        0.530

Saved ✅


In [22]:
# CVS Bootstrap CI by model

import random
random.seed(42)

def bootstrap_ci(data, n_iter=10000,
                 ci=0.95):
    means = []
    n = len(data)
    for _ in range(n_iter):
        sample = random.choices(data, k=n)
        means.append(sum(sample) / n)
    means.sort()
    lower = int((1 - ci) / 2 * n_iter)
    upper = int((1 + ci) / 2 * n_iter)
    return (means[lower],
            sum(data) / n,
            means[upper])

print("CVS Bootstrap CI (95%) by model:")
print(f"{'Model':<12} {'Lower':<8} "
      f"{'Mean':<8} {'Upper':<8} n")
print("-" * 45)

model_stats = {}
for model, scores in sorted(
        model_cvs.items()):
    lo, mean, hi = bootstrap_ci(scores)
    model_stats[model] = (lo, mean, hi)
    print(f"{model:<12} {lo:.4f}   "
          f"{mean:.4f}   {hi:.4f}  "
          f"{len(scores)}")

print(f"\nCI overlap analysis:")
models = sorted(model_stats.keys())
for i in range(len(models)):
    for j in range(i+1, len(models)):
        m1 = models[i]
        m2 = models[j]
        lo1, mean1, hi1 = model_stats[m1]
        lo2, mean2, hi2 = model_stats[m2]
        overlap = not (hi1 < lo2 or
                       hi2 < lo1)
        status = '⚠️  overlapping' \
                 if overlap else \
                 '✅ distinct'
        print(f"  {m1} vs {m2}: {status}")

CVS Bootstrap CI (95%) by model:
Model        Lower    Mean     Upper    n
---------------------------------------------
claude       0.5496   0.5728   0.5957  556
deepseek     0.6244   0.6439   0.6634  580
gemini       0.5800   0.6013   0.6224  582
gpt4o        0.4084   0.4327   0.4569  586
grok         0.5787   0.5997   0.6205  581

CI overlap analysis:
  claude vs deepseek: ✅ distinct
  claude vs gemini: ⚠️  overlapping
  claude vs gpt4o: ✅ distinct
  claude vs grok: ⚠️  overlapping
  deepseek vs gemini: ✅ distinct
  deepseek vs gpt4o: ✅ distinct
  deepseek vs grok: ✅ distinct
  gemini vs gpt4o: ✅ distinct
  gemini vs grok: ⚠️  overlapping
  gpt4o vs grok: ✅ distinct


In [24]:
# Rebuild majority_votes from saved data

import json
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'dss_correctness.json', 'r') as f:
    dss_data = json.load(f)

results = dss_data['results']
valid = [r for r in results
         if r.get('correct') is not None]

pairs = defaultdict(list)
for r in valid:
    key = f"{r['question_id']}_{r['model']}"
    pairs[key].append(r['correct'])

majority_votes = {}
for key, votes in pairs.items():
    majority_votes[key] = 1 if \
        sum(votes) >= 2 else 0

print(f"majority_votes rebuilt: "
      f"{len(majority_votes)} entries ✅")

majority_votes rebuilt: 473 entries ✅


In [25]:
# Shannon Efficiency Metric
# efficiency = correctness / log(token_count)
# Rewards concise correct reasoning
# Penalizes verbose wrong answers

import json
import math
from collections import defaultdict

# Load R1 responses for token counts
with open(f'{V2_RESULTS_PATH}/'
          f'round1_all_responses.json',
          'r') as f:
    r1_data = json.load(f)

# Build token lookup
token_lookup = {}
for r in r1_data['responses']:
    qid   = r.get('question_id')
    model = r.get('model')
    tokens = r.get('tokens', 0)
    if qid and model and tokens:
        if qid not in token_lookup:
            token_lookup[qid] = {}
        token_lookup[qid][model] = tokens

# Compute Shannon efficiency for
# verifiable categories only
shannon_scores = []

for key, correct in majority_votes.items():
    parts = key.split('_')
    qid   = parts[0]
    model = parts[1]

    tokens = token_lookup.get(
        qid, {}).get(model, 0)
    if tokens <= 1:
        continue

    q = next((q for q in ALL_QUESTIONS
              if q['id'] == qid), None)
    if not q:
        continue

    efficiency = round(
        correct / math.log(tokens), 4)

    shannon_scores.append({
        'question_id': qid,
        'category':    q['category'],
        'difficulty':  q['difficulty'],
        'model':       model,
        'correct':     correct,
        'tokens':      tokens,
        'efficiency':  efficiency
    })

# Save
with open(f'{V2_RESULTS_PATH}/'
          f'shannon_efficiency.json',
          'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(shannon_scores),
        'scores':    shannon_scores
    }, f, indent=2)

# Summary by model
model_eff = defaultdict(list)
for s in shannon_scores:
    model_eff[s['model']].append(
        s['efficiency'])

print("Shannon Efficiency by model:")
for model, scores in sorted(
        model_eff.items()):
    avg = sum(scores) / len(scores)
    print(f"  {model:<12} {avg:.4f} "
          f"(n={len(scores)})")

# By category
cat_eff = defaultdict(list)
for s in shannon_scores:
    cat_eff[s['category']].append(
        s['efficiency'])

print(f"\nShannon Efficiency by category:")
for cat, scores in sorted(
        cat_eff.items()):
    avg = sum(scores) / len(scores)
    print(f"  {cat:<25} {avg:.4f}")

print(f"\nTotal scores: {len(shannon_scores)}")
print(f"Saved ✅")

Shannon Efficiency by model:
  claude       0.1433 (n=95)
  deepseek     0.1483 (n=95)
  gemini       0.1162 (n=95)
  gpt4o        0.1386 (n=95)
  grok         0.1411 (n=93)

Shannon Efficiency by category:
  logical_syllogism         0.1472
  mathematical_proof        0.1473
  probabilistic             0.1458
  quantum_reasoning         0.1367
  spatial_reasoning         0.1154
  temporal_reasoning        0.1048

Total scores: 473
Saved ✅


In [26]:
# ANOVA — CVS scores across models

import json
from collections import defaultdict

# Load consolidated CVS data
with open(f'{V2_RESULTS_PATH}/'
          f'cvs_all_scores.json', 'r') as f:
    cvs_data = json.load(f)

valid = cvs_data['results']

# Group by model — average per question
# to avoid pseudoreplication
model_question_avg = defaultdict(
    lambda: defaultdict(list))

for r in valid:
    model_question_avg[
        r['reviewer']][
        r['question_id']].append(
        r['cvs_score'])

model_scores = defaultdict(list)
for model, questions in \
        model_question_avg.items():
    for qid, scores in questions.items():
        avg = sum(scores) / len(scores)
        model_scores[model].append(avg)

models = sorted(model_scores.keys())
groups = [model_scores[m] for m in models]

# One-way ANOVA manual computation
def one_way_anova(groups):
    all_vals = [v for g in groups
                for v in g]
    grand_mean = sum(all_vals) / \
                 len(all_vals)
    k = len(groups)
    n_total = len(all_vals)

    ss_between = sum(
        len(g) * (sum(g)/len(g)
                  - grand_mean)**2
        for g in groups)
    ss_within = sum(
        (v - sum(g)/len(g))**2
        for g in groups
        for v in g)

    df_between = k - 1
    df_within  = n_total - k

    ms_between = ss_between / df_between
    ms_within  = ss_within / df_within
    f_stat     = ms_between / ms_within

    return {
        'f_stat':      round(f_stat, 4),
        'df_between':  df_between,
        'df_within':   df_within,
        'ms_between':  round(ms_between, 6),
        'ms_within':   round(ms_within, 6),
        'ss_between':  round(ss_between, 6),
        'ss_within':   round(ss_within, 6)
    }

anova = one_way_anova(groups)

print("CVS One-Way ANOVA:")
print(f"  F({anova['df_between']}, "
      f"{anova['df_within']}) = "
      f"{anova['f_stat']}")
print(f"  MS between: {anova['ms_between']}")
print(f"  MS within:  {anova['ms_within']}")

# Cohen's d pairwise
def cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    m1 = sum(g1) / n1
    m2 = sum(g2) / n2
    s1 = (sum((x-m1)**2 for x in g1)
          / (n1-1)) ** 0.5
    s2 = (sum((x-m2)**2 for x in g2)
          / (n2-1)) ** 0.5
    pooled = ((( n1-1)*s1**2 +
               (n2-1)*s2**2) /
              (n1+n2-2)) ** 0.5
    return round(abs(m1-m2) / pooled, 4)

print(f"\nCohen's d pairwise:")
for i in range(len(models)):
    for j in range(i+1, len(models)):
        d = cohens_d(groups[i], groups[j])
        magnitude = (
            'small' if d < 0.5 else
            'medium' if d < 0.8 else
            'large')
        print(f"  {models[i]} vs "
              f"{models[j]}: "
              f"d={d} ({magnitude})")

CVS One-Way ANOVA:
  F(4, 991) = 47.8134
  MS between: 1.296411
  MS within:  0.027114

Cohen's d pairwise:
  claude vs deepseek: d=0.4827 (small)
  claude vs gemini: d=0.1558 (small)
  claude vs gpt4o: d=0.7305 (medium)
  claude vs grok: d=0.1543 (small)
  deepseek vs gemini: d=0.2921 (small)
  deepseek vs gpt4o: d=1.1658 (large)
  deepseek vs grok: d=0.3515 (small)
  gemini vs gpt4o: d=0.8483 (large)
  gemini vs grok: d=0.0168 (small)
  gpt4o vs grok: d=0.8886 (large)


In [27]:
# ANOVA — DSS scores across models

with open(f'{V2_RESULTS_PATH}/'
          f'dss_scores.json', 'r') as f:
    dss_data = json.load(f)

model_dss = defaultdict(list)
for s in dss_data['scores']:
    model_dss[s['model']].append(s['dss'])

models_dss = sorted(model_dss.keys())
groups_dss = [model_dss[m]
              for m in models_dss]

anova_dss = one_way_anova(groups_dss)

print("DSS One-Way ANOVA:")
print(f"  F({anova_dss['df_between']}, "
      f"{anova_dss['df_within']}) = "
      f"{anova_dss['f_stat']}")

print(f"\nCohen's d pairwise:")
for i in range(len(models_dss)):
    for j in range(i+1, len(models_dss)):
        d = cohens_d(groups_dss[i],
                     groups_dss[j])
        magnitude = (
            'small' if d < 0.5 else
            'medium' if d < 0.8 else
            'large')
        print(f"  {models_dss[i]} vs "
              f"{models_dss[j]}: "
              f"d={d} ({magnitude})")

DSS One-Way ANOVA:
  F(4, 466) = 1.5252

Cohen's d pairwise:
  claude vs deepseek: d=0.2104 (small)
  claude vs gemini: d=0.0415 (small)
  claude vs gpt4o: d=0.0503 (small)
  claude vs grok: d=0.3341 (small)
  deepseek vs gemini: d=0.2139 (small)
  deepseek vs gpt4o: d=0.0915 (small)
  deepseek vs grok: d=0.1268 (small)
  gemini vs gpt4o: d=0.0744 (small)
  gemini vs grok: d=0.3219 (small)
  gpt4o vs grok: d=0.1849 (small)


In [28]:
# Meta-Council Analysis — Cell 1
# Methodology critique

import time

FINDINGS_SUMMARY = """
HF-IQR V2 Hudson Forge Intelligence
and Reasoning Benchmark Version 2

STUDY DESIGN:
200 questions across 12 categories
5 frontier models: claude-sonnet-4-5,
gpt-4o, gemini-2.5-pro, deepseek-chat,
grok-3
5 round deliberation protocol:
Round 1: Independent response
Round 2: Anonymous peer critique
Round 3: Defend or revise
Round 4: Mirror self-assessment
Round 5: Mechanistic trace

SCORING:
DSS V2: Deliberation Stability Score
  Correctness weighted
  Verifiable categories only
CVS V2: Critique Validity Score
  5 dimensions scored by local jury
Shannon Efficiency: correctness/log(tokens)

KEY FINDINGS:
1. CVS ANOVA F(4,991)=47.81 significant
   GPT-4o mean CVS 0.433 vs others
   0.573-0.644. Large effect sizes.
   deepseek vs gpt4o: d=1.17

2. DSS ANOVA F(4,466)=1.53 not significant
   All model pairs small effect sizes.
   Category predicts DSS more than model.
   temporal_reasoning DSS 0.702 vs
   mathematical_proof DSS 0.925

3. Grok safety refusals: 4 confirmed 403
   errors on formal academic content.
   PRQ-01 EDQ-10 EDQ-15 TRQ-06.

4. Shannon efficiency:
   deepseek 0.1483 most efficient
   gemini 0.1162 least efficient

DOCUMENTED LIMITATIONS:
- ESVR jury scoring failed validation
  inter-rater r=0.238 to 0.470
- Pre-registration deviation: 5 models
  ran instead of 6 pre-registered
- CVS inter-rater correlation moderate
  r=0.524 to 0.569
- DSS uses correctness as proxy for RQR
  reducing component independence
- Local jury models may lack domain
  knowledge for complex questions
"""

CRITIQUE_PROMPT = f"""You are a senior
AI researcher and peer reviewer with
expertise in benchmark design, evaluation
methodology, and statistical analysis.

Below is a summary of HF-IQR V2, an
independent pre-registered multi-round
deliberation benchmark for frontier LLMs.

{FINDINGS_SUMMARY}

Your task is to provide a rigorous
peer review critique. Be direct and
specific. Do not soften your assessment.

Address all of the following:

1. METHODOLOGICAL WEAKNESSES
   Identify the three most significant
   methodological weaknesses in the
   study design or scoring approach.
   For each weakness explain exactly
   why it matters and what it means
   for the validity of the findings.

2. ALTERNATIVE EXPLANATIONS
   For each of the four key findings
   provide at least one credible
   alternative explanation that the
   current methodology cannot rule out.

3. STATISTICAL CONCERNS
   Evaluate the statistical approach.
   Are the conclusions drawn from the
   ANOVA and bootstrap CI appropriate?
   What additional tests are needed?

4. PUBLICATION READINESS
   Be direct: what would a reviewer
   at NeurIPS or EMNLP require before
   accepting this paper? List specific
   changes not general advice.

5. GENUINE STRENGTHS
   What aspects of this study are
   genuinely novel and defensible?
   What does this work contribute
   that does not currently exist?

Provide a detailed thorough response.
Do not be brief. This researcher needs
your full honest assessment."""

print("Running meta-council critique...\n")

council_critique = {}

for model in SUBJECT_MODELS:
    print(f"Querying {model}...")
    result = query_frontier(
        model,
        CRITIQUE_PROMPT,
        6,
        'META-CRITIQUE',
        max_tokens=6000)

    if not result.get('error'):
        council_critique[model] = \
            result['response']
        print(f"  ✅ {result['tokens']} tokens")
    else:
        council_critique[model] = None
        print(f"  ❌ {result['error'][:50]}")

    time.sleep(2)

with open(f'{V2_RESULTS_PATH}/'
          f'meta_council_critique.json',
          'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'prompt':    CRITIQUE_PROMPT,
        'responses': council_critique
    }, f, indent=2)

print(f"\nMeta-council critique complete ✅")

Running meta-council critique...

Querying claude...
  ✅ 6886 tokens
Querying gpt4o...
  ✅ 1904 tokens
Querying gemini...
  ✅ 6650 tokens
Querying deepseek...
  ✅ 3727 tokens
Querying grok...
  ✅ 2461 tokens

Meta-council critique complete ✅


In [29]:
# Display meta-council critiques

with open(f'{V2_RESULTS_PATH}/'
          f'meta_council_critique.json',
          'r') as f:
    critique_data = json.load(f)

for model, response in \
        critique_data['responses'].items():
    if response:
        print(f"\n{'='*60}")
        print(f"MODEL: {model.upper()}")
        print(f"{'='*60}")
        print(response[:2000])
        print(f"... [{len(response)} chars total]")


MODEL: CLAUDE
# PEER REVIEW: HF-IQR V2 BENCHMARK

## 1. METHODOLOGICAL WEAKNESSES

### Weakness 1: Circular Dependency in DSS Construction

**The Problem:** DSS uses correctness as a proxy for Response Quality Rating (RQR), then incorporates this correctness-derived measure into stability calculations. This creates a tautological measurement where the dependent variable (deliberation quality) is partially defined by the independent variable you're trying to validate against (correctness).

**Why It Matters:** This fundamentally undermines the claim that DSS measures "deliberation stability." What you're actually measuring is "correctness-weighted agreement with initial response." A model could show high DSS simply by being confidently wrong across all rounds—this would register as "stable" despite representing a complete deliberation failure. The metric cannot distinguish between:
- Stable reasoning that happens to be correct
- Stable reasoning that happens to be incorrect  
- Reasoni

In [30]:
# Pull 10 critiques for human validation
# 5 low GPT-4o scores
# 5 high DeepSeek scores

import json
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'cvs_all_scores.json', 'r') as f:
    cvs_data = json.load(f)

valid = cvs_data['results']

# Average CVS per question/reviewer
# across jury models
pair_scores = defaultdict(list)
for r in valid:
    key = (r['question_id'],
           r['reviewer'])
    pair_scores[key].append(
        r['cvs_score'])

pair_avg = {
    k: sum(v)/len(v)
    for k, v in pair_scores.items()}

# 5 lowest GPT-4o scores
gpt4o_scores = {
    k: v for k, v in pair_avg.items()
    if k[1] == 'gpt4o'}
lowest_gpt4o = sorted(
    gpt4o_scores.items(),
    key=lambda x: x[1])[:5]

# 5 highest DeepSeek scores
deepseek_scores = {
    k: v for k, v in pair_avg.items()
    if k[1] == 'deepseek'}
highest_deepseek = sorted(
    deepseek_scores.items(),
    key=lambda x: x[1],
    reverse=True)[:5]

validation_set = (
    [(qid, 'gpt4o', score)
     for (qid, _), score in lowest_gpt4o]
    +
    [(qid, 'deepseek', score)
     for (qid, _), score in highest_deepseek]
)

print("10 critiques selected for validation:")
print(f"\n5 lowest GPT-4o scores:")
for qid, model, score in validation_set[:5]:
    print(f"  {qid:<10} {model:<12} "
          f"avg CVS: {score:.3f}")

print(f"\n5 highest DeepSeek scores:")
for qid, model, score in validation_set[5:]:
    print(f"  {qid:<10} {model:<12} "
          f"avg CVS: {score:.3f}")

10 critiques selected for validation:

5 lowest GPT-4o scores:
  MPQ-14     gpt4o        avg CVS: 0.075
  SRQ-06     gpt4o        avg CVS: 0.075
  LSQ-11     gpt4o        avg CVS: 0.075
  PRQ-01     gpt4o        avg CVS: 0.075
  ADV-04     gpt4o        avg CVS: 0.100

5 highest DeepSeek scores:
  QRQ-01     deepseek     avg CVS: 0.925
  PRQ-19     deepseek     avg CVS: 0.925
  QRQ-06     deepseek     avg CVS: 0.883
  QRQ-14     deepseek     avg CVS: 0.883
  QRQ-11     deepseek     avg CVS: 0.825


In [31]:
# Pull critique text for human validation

with open(f'{V2_RESULTS_PATH}/'
          f'round2_all_responses.json',
          'r') as f:
    r2_all = json.load(f)

r2_lookup_full = {}
for r in r2_all['responses']:
    qid      = r.get('question_id')
    reviewer = r.get('reviewer')
    resp     = r.get('response')
    if qid and reviewer and resp:
        if qid not in r2_lookup_full:
            r2_lookup_full[qid] = {}
        r2_lookup_full[qid][reviewer] = resp

print("CVS Human Validation — 10 Critiques")
print("Score each on 0 or 1 per dimension:")
print("  step_cited, error_identified,")
print("  alternative_provided, root_cause,")
print("  no_ad_hominem\n")

for qid, model, jury_avg in validation_set:
    q = next(q for q in ALL_QUESTIONS
             if q['id'] == qid)
    critique = r2_lookup_full.get(
        qid, {}).get(model, '')

    print(f"{'='*60}")
    print(f"ID: {qid} | Reviewer: {model}")
    print(f"Category: {q['category']}")
    print(f"Jury avg CVS: {jury_avg:.3f}")
    print(f"\nQUESTION:")
    print(f"{q['prompt'][:200]}")
    print(f"\nCRITIQUE:")
    print(f"{critique[:600]}")
    print()

CVS Human Validation — 10 Critiques
Score each on 0 or 1 per dimension:
  step_cited, error_identified,
  alternative_provided, root_cause,
  no_ad_hominem

ID: MPQ-14 | Reviewer: gpt4o
Category: mathematical_proof
Jury avg CVS: 0.075

QUESTION:
Prove that log₂(3) is irrational. Show all steps explicitly. State your proof method.

CRITIQUE:
NO WEAKNESSES IDENTIFIED

Peer A's response utilizes a standard and well-established proof by contradiction method to demonstrate that log₂(3) is irrational. Each step logically flows from the assumption that log₂(3) is rational to the conclusion that this assumption leads to a contradiction. The specific contradiction stems from the fundamental properties of even and odd numbers—that an even number (2^p) cannot equal an odd number (3^q).

The distinctions between even and odd powers are appropriately utilized, and every step follows from the definitions and properties involved in the argument

ID: SRQ-06 | Reviewer: gpt4o
Category: spatial_reasonin

In [32]:
# DSS Reframe
# Report correctness and position
# stability as separate independent
# metrics rather than combined score

import json
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'dss_scores.json', 'r') as f:
    dss_data = json.load(f)

scores = dss_data['scores']

# Metric 1 — Position Stability (PS)
# Did the model maintain its position?
# 1.0 = DEFENDING 0.5 = REVISING
# Independent of correctness

# Metric 2 — Correctness Rate (CR)
# Was the model correct on verifiable
# questions regardless of position?

model_ps = defaultdict(list)
model_cr = defaultdict(list)
cat_ps   = defaultdict(list)
cat_cr   = defaultdict(list)

for s in scores:
    model_ps[s['model']].append(s['ps'])
    model_cr[s['model']].append(s['cr'])
    cat_ps[s['category']].append(s['ps'])
    cat_cr[s['category']].append(s['cr'])

print("Position Stability by model:")
for model, vals in sorted(model_ps.items()):
    avg = sum(vals) / len(vals)
    print(f"  {model:<12} {avg:.4f}")

print(f"\nCorrectness Rate by model:")
for model, vals in sorted(model_cr.items()):
    avg = sum(vals) / len(vals)
    print(f"  {model:<12} {avg:.4f}")

print(f"\nPosition Stability by category:")
for cat, vals in sorted(cat_ps.items()):
    avg = sum(vals) / len(vals)
    print(f"  {cat:<25} {avg:.4f}")

print(f"\nCorrectness Rate by category:")
for cat, vals in sorted(cat_cr.items()):
    avg = sum(vals) / len(vals)
    print(f"  {cat:<25} {avg:.4f}")

# Save reframed metrics
with open(f'{V2_RESULTS_PATH}/'
          f'dss_reframed.json', 'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'note': 'PS and CR reported as '
                'independent metrics. '
                'Combined DSS score '
                'deprecated due to '
                'circular dependency '
                'identified in peer review.',
        'scores': scores
    }, f, indent=2)

print(f"\nDSS reframed metrics saved ✅")

Position Stability by model:
  claude       0.6452
  deepseek     0.5947
  gemini       0.7632
  gpt4o        0.8316
  grok         0.5753

Correctness Rate by model:
  claude       0.9785
  deepseek     0.9579
  gemini       0.9368
  gpt4o        0.8842
  grok         0.9355

Position Stability by category:
  logical_syllogism         0.6300
  mathematical_proof        0.7500
  probabilistic             0.6582
  quantum_reasoning         0.6867
  spatial_reasoning         0.6900
  temporal_reasoning        0.6875

Correctness Rate by category:
  logical_syllogism         0.9900
  mathematical_proof        1.0000
  probabilistic             1.0000
  quantum_reasoning         0.9600
  spatial_reasoning         0.7800
  temporal_reasoning        0.7083

DSS reframed metrics saved ✅


In [36]:
# Round 5 Mechanistic Trace Analysis
# Pattern analysis on revision triggers

import json
import re
from collections import defaultdict

with open(f'{V2_RESULTS_PATH}/'
          f'round5_all_responses.json',
          'r') as f:
    r5_data = json.load(f)

r5_responses = [
    r for r in r5_data['responses']
    if r.get('response')]

# Revision trigger categories
# based on pre-registration taxonomy
triggers = {
    'logical':      [
        'logic', 'invalid', 'fallacy',
        'contradiction', 'incorrect',
        'error', 'wrong', 'flaw'],
    'social':       [
        'peer', 'critique', 'feedback',
        'reviewer', 'suggested',
        'pointed out', 'noted'],
    'uncertainty':  [
        'uncertain', 'unclear',
        'ambiguous', 'confidence',
        'not sure', 'possible',
        'might be'],
    'evidence':     [
        'evidence', 'proof', 'data',
        'demonstrates', 'shows',
        'verified', 'confirmed'],
    'completeness': [
        'incomplete', 'missing',
        'omitted', 'additional',
        'further', 'more detail']
}

# Analyze by position and model
results = []
for r in r5_responses:
    text     = r.get('response', '').lower()
    model    = r.get('model', '')
    position = r.get('position', 'UNKNOWN')
    qid      = r.get('question_id', '')
    category = r.get('category', '')

    trigger_counts = {}
    for trigger_name, keywords in \
            triggers.items():
        count = sum(
            1 for kw in keywords
            if kw in text)
        trigger_counts[trigger_name] = count

    dominant = max(
        trigger_counts,
        key=trigger_counts.get)

    results.append({
        'question_id':    qid,
        'category':       category,
        'model':          model,
        'position':       position,
        'triggers':       trigger_counts,
        'dominant':       dominant
    })

# Summary by model and position
print("Dominant revision triggers by model:\n")
model_triggers = defaultdict(
    lambda: defaultdict(int))

for r in results:
    if r['position'] == 'REVISING':
        model_triggers[r['model']][
            r['dominant']] += 1

for model, triggers_count in sorted(
        model_triggers.items()):
    total = sum(triggers_count.values())
    print(f"{model}:")
    for trigger, count in sorted(
            triggers_count.items(),
            key=lambda x: x[1],
            reverse=True):
        pct = count / total * 100
        print(f"  {trigger:<15} "
              f"{count:>3} ({pct:.1f}%)")
    print()

# Summary by category
print("\nDominant triggers by category:\n")
cat_triggers = defaultdict(
    lambda: defaultdict(int))

for r in results:
    if r['position'] == 'REVISING':
        cat_triggers[r['category']][
            r['dominant']] += 1

for cat, triggers_count in sorted(
        cat_triggers.items()):
    top = max(triggers_count,
              key=triggers_count.get)
    total = sum(triggers_count.values())
    print(f"  {cat:<25} "
          f"dominant: {top} "
          f"({triggers_count[top]}/{total})")

# Save results
with open(f'{V2_RESULTS_PATH}/'
          f'round5_trace_analysis.json',
          'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'total':     len(results),
        'results':   results
    }, f, indent=2)

print(f"\nTotal analyzed: {len(results)}")
print(f"Saved ✅")

Dominant revision triggers by model:

claude:
  logical         148 (90.2%)
  uncertainty      12 (7.3%)
  social            2 (1.2%)
  completeness      1 (0.6%)
  evidence          1 (0.6%)

deepseek:
  logical         171 (94.5%)
  social            5 (2.8%)
  uncertainty       4 (2.2%)
  evidence          1 (0.6%)

gemini:
  logical         117 (93.6%)
  social            5 (4.0%)
  evidence          2 (1.6%)
  uncertainty       1 (0.8%)

gpt4o:
  social           47 (43.9%)
  logical          42 (39.3%)
  uncertainty       7 (6.5%)
  evidence          6 (5.6%)
  completeness      5 (4.7%)

grok:
  logical         140 (77.8%)
  uncertainty      15 (8.3%)
  social           13 (7.2%)
  evidence          8 (4.4%)
  completeness      4 (2.2%)


Dominant triggers by category:

  adversarial               dominant: logical (60/76)
  causal_chain              dominant: logical (68/86)
  counterfactual            dominant: logical (65/94)
  ethical_dilemma           dominant: logical (51/

In [33]:
# Pre-registration amendment

amendment = {
    'title': 'HF-IQR V2 Pre-registration '
             'Amendment',
    'original_prereg_timestamp':
        '2026-05-08T23:56:24Z',
    'amendment_timestamp': time.strftime(
        '%Y-%m-%dT%H:%M:%SZ'),
    'researcher': 'Billy Davis | '
                  'WARRIOROFGOD40',
    'deviations': [
        {
            'item': 'Subject models',
            'pre_registered': 6,
            'actual': 5,
            'excluded_model': 'llama3-70b',
            'reason': 'llama3-70b was '
                      'accessible via local '
                      'NGROK infrastructure '
                      'but API integration '
                      'with the council run '
                      'protocol was not '
                      'completed before '
                      'data collection began. '
                      'Decision made to '
                      'proceed with 5 frontier '
                      'API models rather than '
                      'delay the study.',
            'impact': 'Findings are limited '
                      'to the five frontier '
                      'models that completed '
                      'all five rounds. '
                      'llama3-70b comparisons '
                      'are not possible from '
                      'this dataset.'
        },
        {
            'item': 'ESVR scoring',
            'pre_registered': 'LLM-as-judge '
                              'per step scoring',
            'actual': 'Abandoned after '
                      'validation failure',
            'reason': 'Inter-rater correlation '
                      'r=0.238 to r=0.470 '
                      'between local jury '
                      'models indicated '
                      'insufficient reliability '
                      'for primary endpoint '
                      'reporting.',
            'impact': 'ESVR is reported as '
                      'a methodological finding '
                      'not a primary metric. '
                      'IQS composite score '
                      'adjusted accordingly.'
        },
        {
            'item': 'DSS formula',
            'pre_registered': 'Combined DSS = '
                              'PS*0.30 + RQR*0.40'
                              ' + CR*0.30',
            'actual': 'PS and CR reported '
                      'as independent metrics',
            'reason': 'Peer review identified '
                      'circular dependency '
                      'in combining correctness '
                      'with deliberation '
                      'stability. RQR proxy '
                      'using correctness '
                      'reduces component '
                      'independence.',
            'impact': 'Position Stability and '
                      'Correctness Rate reported '
                      'separately. Relationship '
                      'between them is itself '
                      'a finding.'
        }
    ],
    'status': 'AMENDMENT FILED — '
              'data collection already '
              'complete at time of filing'
}

with open(f'{V2_RESULTS_PATH}/'
          f'prereg_amendment.json',
          'w') as f:
    json.dump(amendment, f, indent=2)

print("Pre-registration amendment filed:")
print(f"  Timestamp: "
      f"{amendment['amendment_timestamp']}")
print(f"  Deviations documented: "
      f"{len(amendment['deviations'])}")
print(f"  Saved ✅")

Pre-registration amendment filed:
  Timestamp: 2026-05-17T19:34:18Z
  Deviations documented: 3
  Saved ✅


In [34]:
# Meta-Council Analysis — Cell 2
# V3 advancement recommendations

REFRAMED_FINDINGS = """
HF-IQR V2 — Corrected Findings Summary

CORRECTIONS SINCE CELL 1:

1. DSS has been reframed as two
   independent metrics:

   Position Stability (PS):
     gpt4o:    0.8316 defends most
     gemini:   0.7632
     claude:   0.6452
     deepseek: 0.5947
     grok:     0.5753 revises most

   Correctness Rate (CR):
     claude:   0.9785 most accurate
     deepseek: 0.9579
     grok:     0.9355
     gemini:   0.9368
     gpt4o:    0.8842 least accurate

   Key finding: High position stability
   does not correlate with correctness.
   GPT-4o defends most but is least
   correct on verifiable questions.

2. CVS finding validated by human
   annotation on 10 critiques.
   Human scores aligned with jury.
   GPT-4o produced validation responses
   not critiques — behavioral finding
   confirmed.

   CVS by reviewer:
     deepseek: 0.644
     grok:     0.600
     gemini:   0.601
     claude:   0.573
     gpt4o:    0.433

   F(4,991) = 47.81 significant
   deepseek vs gpt4o: d=1.17 large

3. Pre-registration amendment filed
   documenting three deviations:
   llama3-70b exclusion
   ESVR validation failure
   DSS formula reframe

REMAINING LIMITATIONS:
  CVS inter-rater r=0.524-0.569 moderate
  Amendment filed after data collection
  Local jury domain knowledge gap
  on complex technical questions
"""

ADVANCEMENT_PROMPT = f"""You are a senior
AI researcher and benchmark designer
with expertise in evaluation methodology,
multi-agent systems, and publication
at top AI venues.

Below is a summary of HF-IQR V2 including
corrections made after peer review critique.

{REFRAMED_FINDINGS}

Your task is to provide specific detailed
recommendations for HF-IQR V3.

Address all of the following in detail.
Do not be brief. This researcher needs
your full technical recommendations.

1. PROTOCOL IMPROVEMENTS
   What specific changes to the five
   round deliberation protocol would
   most improve scientific rigor?
   Be specific about round design
   prompt structure and scoring criteria.
   What new rounds or dimensions should
   be added?

2. SCORING METHODOLOGY
   How should CVS be improved to
   address the inter-rater reliability
   limitation?
   What would replace or supplement
   local jury scoring?
   How should human annotation be
   integrated efficiently?

3. DATASET DESIGN
   What changes to the 200 question
   dataset would strengthen the study?
   Which categories should be expanded
   contracted or replaced?
   What ground truth anchoring approach
   would be most defensible?

4. STATISTICAL FRAMEWORK
   What additional statistical tests
   would strengthen the findings?
   How should multiple comparisons
   be handled in V3?
   What sample size is needed for
   adequate statistical power?

5. THE GPT-4O VALIDATION FINDING
   GPT-4o consistently produced
   validation responses not critiques
   in Round 2.
   What does this reveal about
   GPT-4o's deliberation behavior?
   How should V3 be designed to
   investigate this further?

6. PUBLICATION PATHWAY
   Given the corrected findings what
   is the most defensible publication
   venue and framing?
   What is the single strongest claim
   this study can make?
   What additional data would make
   it unambiguous?

Provide thorough specific actionable
recommendations. Reference specific
findings and numbers where relevant."""

print("Running meta-council V3 advancement...\n")

council_advancement = {}

for model in SUBJECT_MODELS:
    print(f"Querying {model}...")
    result = query_frontier(
        model,
        ADVANCEMENT_PROMPT,
        6,
        'META-ADVANCEMENT',
        max_tokens=6000)

    if not result.get('error'):
        council_advancement[model] = \
            result['response']
        print(f"  ✅ {result['tokens']} tokens")
    else:
        council_advancement[model] = None
        print(f"  ❌ {result['error'][:50]}")

    time.sleep(2)

with open(f'{V2_RESULTS_PATH}/'
          f'meta_council_advancement.json',
          'w') as f:
    json.dump({
        'timestamp': time.strftime(
            '%Y-%m-%dT%H:%M:%SZ'),
        'prompt':    ADVANCEMENT_PROMPT,
        'responses': council_advancement
    }, f, indent=2)

print(f"\nMeta-council advancement complete ✅")

Running meta-council V3 advancement...

Querying claude...
  ✅ 6922 tokens
Querying gpt4o...
  ✅ 1811 tokens
Querying gemini...
  ✅ 7356 tokens
Querying deepseek...
  ✅ 4238 tokens
Querying grok...
  ✅ 2669 tokens

Meta-council advancement complete ✅


In [35]:
# Display advancement responses

with open(f'{V2_RESULTS_PATH}/'
          f'meta_council_advancement.json',
          'r') as f:
    adv_data = json.load(f)

for model, response in \
        adv_data['responses'].items():
    if response:
        print(f"\n{'='*60}")
        print(f"MODEL: {model.upper()}")
        print(f"{'='*60}")
        print(response[:2000])
        print(f"... [{len(response)} "
              f"chars total]")


MODEL: CLAUDE
# Detailed Recommendations for HF-IQR V3

## 1. PROTOCOL IMPROVEMENTS

### Core Protocol Redesign

The current five-round protocol has revealed significant architectural weaknesses. The most critical flaw is that Round 2 (critique generation) lacks disambiguation between validation and genuine critique. GPT-4o's validation behavior (CVS=0.433 vs deepseek's 0.644) demonstrates that the prompt structure permits task misinterpretation.

**Round 2 Reconstruction:**

The current prompt must be requesting a "review" or "critique" in language that GPT-4o interprets as permission to validate. I recommend restructuring Round 2 as two sequential sub-rounds:

- **Round 2A: Identification Phase** - "Examine the previous answer. List specific claims, reasoning steps, or factual assertions that may be incorrect, incomplete, or unsupported. You must identify at least three specific elements to examine, even if the answer appears correct. Do not provide corrections yet—only identify pot